# NCM Omni - Colab, self-contained, direct feed

One arm, one loss, one path from the Hub to weights. **The code ships inside this notebook**: 13
files, 4,496 lines, materialised to `.ncm_stage` and `PKG` by the cells below, with a
sha256 check before anything imports. Nothing is `pip install`ed from git and nothing is imported
from a clone, so a session cannot silently run yesterday's model.

Two sources, one switch (`SRC` in the knobs cell):

| `SRC` | what happens | when |
|---|---|---|
| `"snapshot"` | the cells below write 13 files and the run uses exactly that code | default; reproducible; offline |
| `"git"` | `PKG` is deleted and re-cloned at `PIN`, then sha-checked against the snapshot | you pushed a fix and want it without restarting the kernel |

Both paths end in the same place: `sys.modules` purged for `omni*`, `core`, `run_omni`; bytecode
caches invalidated; `PKG` on `sys.path`; the sha table compared and printed. The purge half matters
as much as the pull: a re-clone in a live kernel keeps executing the old module objects until the
kernel restarts, and the only symptom is "my fix did nothing".

Shipped files:

| file | lines | sha256[:12] |
|---|---|---|
| `core.py` | 436 | `9eedde9c648e` |
| `run_omni.py` | 522 | `fd1f3af6a282` |
| `omni/__init__.py` | 22 | `bf26fc4b7cbc` |
| `omni/schedule.py` | 124 | `c95c751537db` |
| `omni/codecs.py` | 300 | `fc169198c5e5` |
| `omni/producer.py` | 688 | `12c17140d3a7` |
| `omni/livestream.py` | 242 | `4af049323dcf` |
| `omni/data.py` | 187 | `3255e16da85e` |
| `omni/model.py` | 640 | `c53ea47fa4ab` |
| `omni/moe.py` | 112 | `09985713dd00` |
| `omni/preflight.py` | 195 | `702b7b4291b2` |
| `omni/train.py` | 579 | `f67c46f917ab` |
| `omni/eval.py` | 449 | `7128800618ca` |

Not shipped: `tests/`, `_probe_sources.py`, `run/` (artefacts), `.env` (secrets). Regenerate this
notebook with `python tools/build_notebook.py`; `--check` fails if a payload cell drifted.

## 0. Knobs - one cell drives the whole run

In [ ]:
# ---- where the code comes from ----
# "snapshot": the 13 cells further down are the code. No network, no clone.
# "git":      PKG is deleted and re-cloned at PIN. The purge-and-re-pull path.
SRC = "snapshot"

PIN = "main"        # git mode: branch, tag or commit sha. Only a sha is reproducible.
GIT_URL = "https://github.com/adi274903/ncm-omni-v2.git"
GH_PAT = ""         # private repo only; filled from a Colab secret in the next cell

PKG = "/content/ncm_omni"     # where the package lands - and the cwd for every artefact below
STAGE = PKG.rsplit("/", 1)[0] + "/.ncm_stage"   # payload staging dir, deleted after the copy

# ---- model: d=1536, L=16, 3 experts x 360.0M = 1.178B total, 462M active ----
PRESET = "moex"     # "sm" | "md" | "lg" | "moex"
MOE = 3             # one expert per modality, routing forced by packed position
MOE_FFN = 4880      # expert SwiGLU width -> 360.0M per expert, h/d = 3.18
                    # 0 = fall back to mlp_ratio 4.0 (2048 at d=768), which is not 360M

# ---- what the run is allowed to spend ----
TOKENS = 0.002      # BILLIONS of live code predictions. The budget; STEPS is derived from it.
BATCH = 32          # 0 = derive from VRAM
TFLOPS = 0          # 0 = auto: 25 on this box's 3070 (20-30 measured), 125 on A100, 15 on T4
LEN = 256           # base_len for text and image. Longer samples = fewer rows for the same token
                    # count, which matters because the Hub feed, not the GPU, sets the pace.
EVAL_EVERY = 50
LOG_EVERY = 10
HOLD = 64           # rows pulled off the front of the feed and held out (direct-feed path only)
PRE_STEPS = 60      # preflight steps. The gate is cheap at 1.18B and catches a dead head early.
PRE_BATCH = 4
USE_SHARDS = True   # produce first, train from npz: a ~2 samples/s feed cannot keep 1.18B busy
INSTALL = True      # pip-install the runtime deps
SMOKE = 0           # >0 = timing-only run of that many steps
ARM = "strict"

# sha256[:12] of every file this notebook carries. Generated by tools/build_notebook.py - if a
# payload cell above was edited by hand, this table is stale and the verify cell will say so.
EXPECTED = {
    "core.py": "9eedde9c648e",
    "run_omni.py": "fd1f3af6a282",
    "omni/__init__.py": "bf26fc4b7cbc",
    "omni/schedule.py": "c95c751537db",
    "omni/codecs.py": "fc169198c5e5",
    "omni/producer.py": "12c17140d3a7",
    "omni/livestream.py": "4af049323dcf",
    "omni/data.py": "3255e16da85e",
    "omni/model.py": "c53ea47fa4ab",
    "omni/moe.py": "09985713dd00",
    "omni/preflight.py": "702b7b4291b2",
    "omni/train.py": "f67c46f917ab",
    "omni/eval.py": "7128800618ca"
}

print(f"{SRC} mode | {len(EXPECTED)} files | arm {ARM} | batch {BATCH} | budget {TOKENS}B live codes | PKG {PKG}")

## 0.1 Runtime deps

`torch` is on the Colab image already and is deliberately not pinned here: a plain
`pip install -r requirements.txt` can replace a CUDA 13 wheel with a CPU one, and the failure shows
up as a slow run, not an error.

In [ ]:
import os
import subprocess
import sys

RUNTIME = "datasets huggingface_hub soundfile av pillow tqdm tensorboard".split()
if INSTALL:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *RUNTIME], check=True)

# Must be set before the first CUDA allocation to matter. At 1.198B with fp32 params + AdamW there
# is no slack: without expandable segments a freed 19 GiB model cannot be re-used for the next one,
# and the OOM blames a 30 MiB tensor with 38 GiB "in use".
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

import torch
print("torch", torch.__version__, "| cuda", torch.cuda.is_available())
print("alloc conf:", os.environ.get("PYTORCH_CUDA_ALLOC_CONF"))
print("deps:", " ".join(RUNTIME), "| installed" if INSTALL else "| skipped")

## 1. Credentials

Colab secrets first (they survive a restart), then the process environment. `HF_TOKEN` only raises
rate limits and unlocks gated sources; the git path needs a PAT only for a private clone.

In [ ]:
import os


def _secret(name):
    """-> value from the env, else from Colab's secret store, else ""."""
    v = os.environ.get(name, "")
    if v:
        return v
    try:
        from google.colab import userdata
        return userdata.get(name) or ""
    except Exception:
        return ""


HF_TOKEN = _secret("HF_TOKEN")
GH_PAT = GH_PAT or _secret("GITHUB_PAT") or _secret("GIT_PAT") or _secret("GITHUB_TOKEN")
if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN        # datasets / huggingface_hub read this directly
print({"HF_TOKEN": bool(HF_TOKEN), "GITHUB PAT": bool(GH_PAT), "SRC": SRC})

## 2. Purge, then get the code

The order is the point: forget the modules, delete the directory, *then* put code back. Deleting
files while `omni.model` sits in `sys.modules` is the re-pull that appears to do nothing.

`git` mode fetches `PIN` into an empty repo rather than cloning a branch, so a commit sha works
exactly like a branch name.

In [ ]:
import importlib
import os
import shutil
import subprocess
import sys


def purge_modules():
    """Drop every module this notebook is about to redefine, and the bytecode caches.

    Files on disk are not enough: Python caches the module object, so a re-pull in a live kernel
    keeps executing the old code and the only symptom is "my fix did nothing".
    """
    mine = [m for m in list(sys.modules)
            if m == "omni" or m.startswith("omni.") or m in ("core", "run_omni")]
    for m in mine:
        del sys.modules[m]
    importlib.invalidate_caches()
    return sorted(mine)


PARENT = os.path.dirname(PKG.rstrip("/")) or "/"
os.makedirs(PARENT, exist_ok=True)
os.chdir(PARENT)                     # never stand inside the directory about to be deleted

dropped = purge_modules()
shutil.rmtree(STAGE, ignore_errors=True)
shutil.rmtree(PKG, ignore_errors=True)
os.makedirs(os.path.join(STAGE, "omni"))
print("purged modules:", dropped or "none")
print(f"staging {STAGE} <- the {len(EXPECTED)} payload cells below")

if SRC == "git":
    url = GIT_URL.replace("https://", f"https://x-access-token:{GH_PAT}@") if GH_PAT else GIT_URL
    os.makedirs(PKG)
    steps = [["git", "init", "-q"],
             ["git", "remote", "add", "origin", url],
             ["git", "fetch", "-q", "--depth", "1", "origin", PIN],
             ["git", "checkout", "-q", "-f", "FETCH_HEAD"]]
    for argv in steps:
        print("+", argv[0], argv[1], "..." if argv[-1] == url else argv[-1])
        subprocess.run(argv, cwd=PKG, check=True, capture_output=True, text=True)
    head = subprocess.run(["git", "-C", PKG, "rev-parse", "--short", "HEAD"],
                          capture_output=True, text=True).stdout.strip()
    print("git: checked out", PIN, "->", head)

## 2.1 The 13 files

Each cell is a verbatim `%%writefile` of one file into `.ncm_stage/`. Nothing here is edited inside the
notebook - edit the `.py` in the repo and run `python tools/build_notebook.py`, or the sha table and
the body will disagree and the verify cell will refuse to continue.

They write to a staging dir, never straight into `PKG`: in `git` mode the clone must win, and a
payload cell that overwrote it afterwards would make `SRC="git"` a lie.

In [ ]:
%%writefile .ncm_stage/core.py
"""NCM (arXiv:2610.14162) level-parallel generator - kernel-optimised core.

Paper -> code mapping (full table in notebook cell 0):
  Fig.4(b) "all K levels are predicted in one forward pass"  -> NCMLevelParallel.forward
  Fig.4(b) "block-diagonal attention restricts each input
            block to itself"                                -> LevelSchedule.dense_mask / flex_block_mask
  Sec.3.2  conditioning on the reconstruction z_hat(k-1)
            "resized to the l_k positions of level k"       -> NCMLevelParallel(conds=...)
  Sec.3.3  residual VQ + resampler R_k: l_k -> L            -> MultiscaleRVQVAE
  Table 6  GQA, QK-norm, RoPE, SwiGLU                       -> LevelAttention / Block

[DEV] per-level positions restart at 0 (VAR-style). The paper does not state the position
      scheme; this is a named choice, not a silent one.
[DEV] cross_level=True opens j<k blocks (block-causal instead of block-diagonal). Ablation
      knob only - the paper's default is block-diagonal. Off by default.
"""
import os
from dataclasses import dataclass
from typing import List, Optional, Sequence, Tuple

import torch
import torch.nn as nn
import torch.nn.functional as F

try:
    from torch.nn.attention.flex_attention import flex_attention, create_block_mask
    HAS_FLEX = True
except Exception:                                     # pragma: no cover
    flex_attention = create_block_mask = None
    HAS_FLEX = False

# flex_attention is only fused (and only then faster than SDPA) under torch.compile;
# called eagerly it materialises the full score matrix and warns. Compile it once, lazily.
COMPILE_FLEX = os.environ.get("NCM_COMPILE_FLEX", "1") == "1"
_COMPILED_FLEX = None


def compiled_flex():
    """torch.compile(flex_attention, dynamic=False); permanently disabled on first failure."""
    global _COMPILED_FLEX, COMPILE_FLEX
    if _COMPILED_FLEX is None:
        try:
            _COMPILED_FLEX = torch.compile(flex_attention, dynamic=False)
        except Exception:
            COMPILE_FLEX, _COMPILED_FLEX = False, False
    return _COMPILED_FLEX

try:
    from liger_kernel.transformers import LigerFusedLinearCrossEntropyLoss
    HAS_LIGER = True
except Exception:
    HAS_LIGER = False


# --------------------------------------------------------------------------- schedule
@dataclass(frozen=True)
class LevelSchedule:
    """Level lengths l_1..l_K, optional conditional prefix (source/puzzle/null)."""
    level_lengths: Tuple[int, ...]
    prefix_len: int = 0

    def __post_init__(self):
        assert len(self.level_lengths) > 0 and all(l > 0 for l in self.level_lengths)

    @property
    def n_levels(self) -> int:
        return len(self.level_lengths)

    @property
    def total_len(self) -> int:
        return self.prefix_len + sum(self.level_lengths)

    def offsets(self) -> List[int]:
        out, c = [], self.prefix_len
        for l in self.level_lengths:
            out.append(c)
            c += l
        return out

    def level_ids(self, device="cpu") -> torch.Tensor:
        """-1 for prefix positions, k (0-based) for level-k positions."""
        ids = torch.full((self.total_len,), -1, dtype=torch.long)
        for k, (o, l) in enumerate(zip(self.offsets(), self.level_lengths)):
            ids[o:o + l] = k
        return ids.to(device)

    def pos_ids(self, device="cpu") -> torch.Tensor:
        p = torch.arange(self.prefix_len, dtype=torch.long, device=device)
        for l in self.level_lengths:
            p = torch.cat([p, torch.arange(l, dtype=torch.long, device=device)])
        return p

    def single_level(self, k: int) -> "LevelSchedule":
        return LevelSchedule((self.level_lengths[k],), self.prefix_len)

    # ------------------------------------------------------------------ masks
    def dense_mask(self, dtype=torch.float32, device="cpu", cross_level: bool = False) -> torch.Tensor:
        """Additive [1,1,T,T]: 0 = visible, -inf = blocked.

        Visible(q in level k, kv in level j) = (j is prefix) or (j == k).
        """
        ids = self.level_ids()
        q, kv = ids[:, None], ids[None, :]
        allowed = (kv == -1) | (kv == q)
        if cross_level:
            allowed = allowed | ((q >= 0) & (kv >= 0) & (kv < q))
        m = torch.zeros(self.total_len, self.total_len, dtype=dtype)
        m.masked_fill_(~allowed, float("-inf"))
        return m.view(1, 1, self.total_len, self.total_len).to(device)

    def flex_block_mask(self, B, H, device="cuda", cross_level: bool = False):
        assert HAS_FLEX, "flex_attention unavailable in this torch build"
        ids = self.level_ids(device)

        def mask_mod(b, h, q_idx, kv_idx):
            ql, kl = ids[q_idx], ids[kv_idx]
            allowed = (kl == -1) | (kl == ql)
            if cross_level:
                allowed = allowed | ((ql >= 0) & (kl >= 0) & (kl < ql))
            return allowed

        return create_block_mask(mask_mod, B=B, H=H, Q_LEN=self.total_len,
                                 KV_LEN=self.total_len, device=device)


# --------------------------------------------------------------------------- layers
def precompute_rope_cos_sin(positions: torch.Tensor, head_dim: int, base: float = 10000.0):
    half = head_dim // 2
    inv = 1.0 / (base ** (torch.arange(0, half, dtype=torch.float32, device=positions.device) / half))
    f = positions.float()[:, None] * inv[None, :]
    return torch.cat([f.cos(), f.cos()], -1), torch.cat([f.sin(), f.sin()], -1)


def apply_rope(x, cos, sin):
    half = x.shape[-1] // 2
    x1, x2 = x[..., :half], x[..., half:]
    return x * cos + torch.cat([-x2, x1], -1) * sin


class RMSNorm(nn.Module):
    def __init__(self, d, eps=1e-6):
        super().__init__()
        self.w, self.eps = nn.Parameter(torch.ones(d)), eps

    def forward(self, x):
        return self.w * x * torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + self.eps)


class LevelAttention(nn.Module):
    """GQA attention: flex_attention+block_mask (fast) or SDPA+dense mask (fallback).

    Fast path keeps K/V at n_kv_head via enable_gqa - the reference implementation
    repeat_interleaves to n_head, materialising a full-size K/V copy every forward.
    """

    def __init__(self, n_embd, n_head, n_kv_head=None, dropout=0.0, qk_norm=False):
        super().__init__()
        assert n_embd % n_head == 0
        self.n_head, self.n_kv_head = n_head, n_kv_head or n_head
        assert n_head % self.n_kv_head == 0
        self.hd, self.dropout = n_embd // n_head, dropout
        self.c_attn = nn.Linear(n_embd, n_embd + 2 * self.n_kv_head * self.hd)
        self.c_proj = nn.Linear(n_embd, n_embd)
        self.q_norm = RMSNorm(self.hd) if qk_norm else None
        self.k_norm = RMSNorm(self.hd) if qk_norm else None

    def forward(self, x, block_mask=None, dense_mask=None, rope=None):
        B, T, C = x.shape
        q, k, v = self.c_attn(x).split([C, self.n_kv_head * self.hd, self.n_kv_head * self.hd], 2)
        q = q.view(B, T, self.n_head, self.hd).transpose(1, 2)
        k = k.view(B, T, self.n_kv_head, self.hd).transpose(1, 2)
        v = v.view(B, T, self.n_kv_head, self.hd).transpose(1, 2)
        if self.q_norm is not None:
            q, k = self.q_norm(q), self.k_norm(k)
        if rope is not None:
            cos, sin = rope
            cos, sin = cos[:T].to(q.dtype), sin[:T].to(q.dtype)
            q, k = apply_rope(q, cos, sin), apply_rope(k, cos, sin)

        if block_mask is not None:
            fn = flex_attention
            if COMPILE_FLEX and q.is_cuda:
                cf = compiled_flex()
                if cf is not False:                  # False => compile unavailable
                    try:
                        y = cf(q, k, v, block_mask=block_mask,
                               enable_gqa=self.n_kv_head != self.n_head)
                        return self._head_out(y)
                    except Exception:
                        # e.g. head_dim < 16 (Inductor NYI), or a mask_mod it cannot trace
                        globals()["COMPILE_FLEX"] = False
            y = fn(q, k, v, block_mask=block_mask,
                   enable_gqa=self.n_kv_head != self.n_head)
        else:
            if self.n_kv_head != self.n_head:
                rep = self.n_head // self.n_kv_head
                k, v = k.repeat_interleave(rep, 1), v.repeat_interleave(rep, 1)
            dm = None if dense_mask is None else dense_mask[..., :T, :T]
            y = F.scaled_dot_product_attention(
                q, k, v, attn_mask=dm, dropout_p=self.dropout if self.training else 0.0,
                is_causal=False)
        return self._head_out(y)

    def _head_out(self, y):
        B, H, T, hd = y.shape
        y = y.transpose(1, 2).contiguous().view(B, T, H * hd)
        return self.c_proj(y)


class Block(nn.Module):
    def __init__(self, n_embd, n_head, n_kv_head=None, dropout=0.0, mlp_ratio=4.0, qk_norm=False):
        super().__init__()
        self.ln1, self.ln2 = RMSNorm(n_embd), RMSNorm(n_embd)
        self.attn = LevelAttention(n_embd, n_head, n_kv_head, dropout, qk_norm)
        h = int(2 * mlp_ratio * n_embd / 3 // 8 * 8)          # swiglu hidden width
        self.w_gate, self.w_up, self.w_down = (nn.Linear(n_embd, h), nn.Linear(n_embd, h),
                                               nn.Linear(h, n_embd))

    def forward(self, x, block_mask=None, dense_mask=None, rope=None):
        x = x + self.attn(self.ln1(x), block_mask=block_mask, dense_mask=dense_mask, rope=rope)
        n = self.ln2(x)
        return x + self.w_down(F.silu(self.w_gate(n)) * self.w_up(n))


# --------------------------------------------------------------------------- generator
class NCMLevelConfig:
    def __init__(self, schedule: LevelSchedule, codebook_size=64, n_embd=32, n_layer=2,
                 n_head=4, n_kv_head=2, d_latent=16, d_prompt=8, dropout=0.0,
                 qk_norm=True, max_level_len=None):
        self.schedule, self.codebook_size, self.n_embd = schedule, codebook_size, n_embd
        self.n_layer, self.n_head, self.n_kv_head = n_layer, n_head, n_kv_head
        self.d_latent, self.d_prompt, self.dropout = d_latent, d_prompt, dropout
        self.qk_norm = qk_norm
        self.max_level_len = max_level_len or max(max(schedule.level_lengths), schedule.prefix_len)


class NCMLevelParallel(nn.Module):
    """One transformer, all K levels predicted in a single forward (Fig.4b)."""

    def __init__(self, cfg: NCMLevelConfig):
        super().__init__()
        self.cfg, self.schedule = cfg, cfg.schedule
        K = self.schedule.n_levels
        self.level_emb = nn.Embedding(K, cfg.n_embd)
        self.tok_emb = nn.ModuleList([nn.Embedding(cfg.codebook_size, cfg.n_embd) for _ in range(K)])
        self.pos_emb = nn.Embedding(cfg.max_level_len + 1, cfg.n_embd)
        self.cond_proj = nn.Linear(cfg.d_latent, cfg.n_embd)     # z_hat(k-1) -> hidden
        self.prompt_proj = nn.Linear(cfg.d_prompt, cfg.n_embd)   # conditional prefix
        self.blocks = nn.ModuleList([Block(cfg.n_embd, cfg.n_head, cfg.n_kv_head, cfg.dropout,
                                           qk_norm=cfg.qk_norm) for _ in range(cfg.n_layer)])
        self.ln_f = RMSNorm(cfg.n_embd)
        self.heads = nn.ModuleList([nn.Linear(cfg.n_embd, cfg.codebook_size, bias=False)
                                    for _ in range(K)])

    # ------------------------------------------------------------------ internals
    def _rope(self, device):
        return precompute_rope_cos_sin(self.schedule.pos_ids(device),
                                       self.cfg.n_embd // self.cfg.n_head)

    def _tokens(self, k, codes_k, cond_k):
        l = self.schedule.level_lengths[k]
        assert codes_k.shape[-1] == l and cond_k.shape[1] == l, (k, codes_k.shape, cond_k.shape)
        return (self.tok_emb[k](codes_k) + self.cond_proj(cond_k) + self.level_emb.weight[k]
                + self.pos_emb(torch.arange(l, device=codes_k.device)))

    def _enc(self, x, kw, rope):
        for b in self.blocks:
            x = b(x, rope=rope, **kw)
        return self.ln_f(x)

    def _kw(self, B, device, sched, cross_level, use_flex):
        if use_flex is None:
            use_flex = HAS_FLEX and torch.device(device).type == "cuda"
        if use_flex:
            return {"block_mask": sched.flex_block_mask(B, self.cfg.n_head, device, cross_level)}
        return {"dense_mask": sched.dense_mask(torch.float32, device, cross_level)}

    # ------------------------------------------------------------------ fused path
    def forward(self, codes, conds, prefix=None, cross_level=False, use_flex=None):
        """All K levels in ONE forward pass -> list of per-level logits."""
        S, B = self.schedule, codes[0].size(0)
        parts = [] if not S.prefix_len else [self.prompt_proj(prefix)]
        for k in range(S.n_levels):
            parts.append(self._tokens(k, codes[k], conds[k]))
        x = torch.cat(parts, 1)
        h = self._enc(x, self._kw(B, x.device, S, cross_level, use_flex), self._rope(x.device))
        return [self.heads[k](h[:, o:o + l]) for k, (o, l) in enumerate(zip(S.offsets(), S.level_lengths))]

    # ------------------------------------------------------------------ oracle path
    def forward_sequential(self, codes, conds, prefix=None, cross_level=False, use_flex=None):
        """K separate forwards - the obvious implementation. Parity oracle, not a fallback."""
        S, B = self.schedule, codes[0].size(0)
        outs = []
        for k in range(S.n_levels):
            sub = S.single_level(k)
            parts = [] if not S.prefix_len else [self.prompt_proj(prefix)]
            parts.append(self._tokens(k, codes[k], conds[k]))
            x = torch.cat(parts, 1)
            rope = precompute_rope_cos_sin(sub.pos_ids(x.device), self.cfg.n_embd // self.cfg.n_head)
            h = self._enc(x, self._kw(B, x.device, sub, cross_level, use_flex), rope)
            outs.append(self.heads[k](h[:, S.prefix_len:S.prefix_len + S.level_lengths[k]]))
        return outs


# --------------------------------------------------------------------------- loss
def level_ce(logits, targets, chunk=None, label_smoothing=0.0):
    """Sum-of-levels CE. `chunk` splits the position axis; must match the single shot."""
    if HAS_LIGER and chunk is None:
        loss = logits[0].new_zeros(())
        for lg, tg in zip(logits, targets):
            loss = loss + LigerFusedLinearCrossEntropyLoss()(lg.reshape(-1, lg.size(-1)),
                                                            tg.reshape(-1, 1))
        return loss
    total, n = logits[0].new_zeros(()), 0
    for lg, tg in zip(logits, targets):
        flat_lg, flat_tg = lg.reshape(-1, lg.size(-1)), tg.reshape(-1)
        if chunk is None:
            total = total + F.cross_entropy(flat_lg, flat_tg, reduction="sum",
                                            label_smoothing=label_smoothing)
        else:
            part = flat_lg.new_zeros(())
            for i in range(0, flat_lg.size(0), chunk):
                part = part + F.cross_entropy(flat_lg[i:i + chunk], flat_tg[i:i + chunk],
                                              reduction="sum", label_smoothing=label_smoothing)
            total = total + part
        n += flat_tg.numel()
    return total / n


# --------------------------------------------------------------------------- tokenizer
class Resampler(nn.Module):
    """R_k: l_k -> L ('resized to the l_k positions of level k', Sec.3.3)."""

    def __init__(self, L):
        super().__init__()
        self.L = L

    def forward(self, e):                                 # [B,l_k,d] -> [B,L,d]
        return F.interpolate(e.transpose(1, 2), size=self.L, mode="nearest").transpose(1, 2)


class EMAQuantizer(nn.Module):
    def __init__(self, V, d, decay=0.99, eps=1e-5):
        super().__init__()
        self.register_buffer("codebook", torch.randn(V, d) / d ** 0.5)
        self.register_buffer("ema_count", torch.ones(V))
        self.register_buffer("ema_sum", self.codebook.clone())
        self.decay, self.eps, self.V = decay, eps, V

    def forward(self, z, update=True):                    # z: [B,l,d]
        d2 = (z.pow(2).sum(-1, keepdim=True) - 2 * z @ self.codebook.T
              + self.codebook.pow(2).sum(-1))
        idx = d2.argmin(-1)
        if update and self.training:
            with torch.no_grad():                         # EMA update: no graph, no grad
                onehot = F.one_hot(idx.reshape(-1), self.V).to(z.dtype)
                flat = z.reshape(-1, z.size(-1)).detach()
                self.ema_count.mul_(self.decay).add_(onehot.sum(0), alpha=1 - self.decay)
                self.ema_sum.mul_(self.decay).add_(onehot.T @ flat, alpha=1 - self.decay)
                self.codebook.copy_(self.ema_sum / (self.ema_count + self.eps).unsqueeze(1))
        zq = self.codebook[idx]
        return idx, zq, F.mse_loss(zq, z.detach())


class MultiscaleRVQVAE(nn.Module):
    """Toy multiscale residual VQ-VAE: text -> z [B,L,d] -> K levels of codes."""

    def __init__(self, vocab, schedule: LevelSchedule, d=16, codebook_size=64, decay=0.99):
        super().__init__()
        self.schedule, self.d, self.V = schedule, d, codebook_size
        self.L = schedule.level_lengths[-1]               # finest level == full length
        self.emb = nn.Embedding(vocab, d)
        self.enc = nn.Sequential(nn.Linear(d, d), nn.GELU(), nn.Linear(d, d))
        self.quantizers = nn.ModuleList([EMAQuantizer(codebook_size, d, decay)
                                         for _ in schedule.level_lengths])
        self.resamplers = nn.ModuleList([Resampler(self.L) for _ in schedule.level_lengths])
        self.dec = nn.Sequential(nn.Linear(d, d), nn.GELU(), nn.Linear(d, vocab))

    def encode(self, x):
        return self.enc(self.emb(x))

    def quantize(self, z, level_cutoff=None):
        """codes[k] is [B,l_k] (-1 past the cutoff); zhat[k] is the cumulative L x d recon."""
        codes, zhat = [], []
        residual, zc = z.clone(), torch.zeros_like(z)
        for k, l in enumerate(self.schedule.level_lengths):
            if level_cutoff is not None and k >= level_cutoff:
                codes.append(torch.full((z.size(0), l), -1, dtype=torch.long, device=z.device))
                zhat.append(zc)
                continue
            zk = F.interpolate(residual.transpose(1, 2), size=l, mode="nearest").transpose(1, 2)
            idx, zq, _ = self.quantizers[k](zk)
            zc = zc + self.resamplers[k](zq)
            residual = z - zc
            codes.append(idx)
            zhat.append(zc)
        return codes, zhat

    def forward(self, x, level_cutoff=None):
        z = self.encode(x)
        codes, zhat = self.quantize(z, level_cutoff)
        return {"codes": codes, "zhat": zhat, "logits": [self.dec(zh) for zh in zhat], "z": z}

    def decode_codes(self, codes):
        zc = None
        for k, c in enumerate(codes):
            contrib = self.resamplers[k](self.quantizers[k].codebook[c])
            zc = contrib if zc is None else zc + contrib
        return self.dec(zc)


# --------------------------------------------------------------------------- sampling
@torch.no_grad()
def generate_codes(generator, tokenizer, prefix=None, B=1, device="cpu", cross_level=False):
    """Coarse-to-fine sampling: exactly K forwards, all l_k codes of a level in parallel."""
    g = generator.eval()
    S = g.schedule
    if prefix is not None and prefix.size(0) != B:      # a single prompt for a batch of B
        assert prefix.size(0) == 1, (prefix.shape, B)
        prefix = prefix.expand(B, -1, -1)
    codes, prev = [], None
    for k, l in enumerate(S.level_lengths):
        cond = (torch.zeros(B, l, tokenizer.d, device=device) if prev is None
                else F.interpolate(prev.transpose(1, 2), size=l, mode="nearest").transpose(1, 2))
        dummy = torch.zeros(B, l, dtype=torch.long, device=device)
        parts = [] if not S.prefix_len else [g.prompt_proj(prefix)]
        parts.append(g._tokens(k, dummy, cond))
        x = torch.cat(parts, 1)
        sub = S.single_level(k)
        kw = g._kw(B, x.device, sub, cross_level, None)
        rope = precompute_rope_cos_sin(sub.pos_ids(x.device), g.cfg.n_embd // g.cfg.n_head)
        h = g._enc(x, kw, rope)[:, S.prefix_len:]
        codes.append(g.heads[k](h).argmax(-1))
        zq = tokenizer.quantizers[k].codebook[codes[-1]]
        prev = tokenizer.resamplers[k](zq) if prev is None else prev + tokenizer.resamplers[k](zq)
    return codes


In [ ]:
%%writefile .ncm_stage/run_omni.py
"""One entry point for the whole omni pipeline: config -> data -> train -> val -> eval.

    python run_omni.py check                     # invariants, no data needed
    python run_omni.py produce  [--budget N]     # writes cfg/*.json then runs the producer
    python run_omni.py train --smoke 10          # timing run, writes nothing
    python run_omni.py train                     # full run + checkpoints
    python run_omni.py eval                      # NFE-vs-quality table
    python run_omni.py all --budget 20000        # produce in the background, wait, train, eval

Both arms use the same schedule, vocab and model width; the only difference is the attention
mask (`block` vs `ar`), which is what makes the comparison a matched-params comparison.
"""
import argparse
import glob
import json
import os
import subprocess
import sys
import time

HERE = os.path.dirname(os.path.abspath(__file__))
RUN = os.path.join(HERE, "run")


# Model/optimiser presets. Parameter counts are measured, not nominal: at base_len=1024 the
# positional table is 1025 x d, so the same width counts for more than it does at 64.
#   sm  10.44M  d384  L6  h6  kv2     -> the debug harness that produced the 10M result
#   md  83.6M   d768  L13 h12 kv3     -> the kill test rung (measured 83.59M at base_len 1024)
#   lg  454.1M  d1536 L18 h16 kv4     -> the paper number, rented H100 only
#   moex 1.178B total / 459M active   d1536 L16 h16 kv4, 3 experts x 359.8M -> needs
#                                      --moe 3 --moe-ffn 4880, or it is just another lg
PRESETS = {
    "sm": dict(n_embd=384,  n_layer=6,  n_head=6,  n_kv_head=2, batch=8,  lr=3e-4,   warmup=100),
    "md": dict(n_embd=768,  n_layer=13, n_head=12, n_kv_head=3, batch=8,  lr=2e-4,   warmup=200),
    "lg": dict(n_embd=1536, n_layer=18, n_head=16, n_kv_head=4, batch=8, lr=1.5e-4, warmup=300),
    "moex": dict(n_embd=1536, n_layer=16, n_head=16, n_kv_head=4, batch=8, lr=1.5e-4, warmup=300),
}


def apply_overrides(cfg, a):
    """Preset dims, schedule length, eval-set size. Every command goes through here, because
    each one rewrites run/cfg/*.json - a mismatch between produce and train would silently
    train on a schedule the shards were not encoded with."""
    preset = getattr(a, "preset", None)
    if preset and preset in PRESETS:
        p = PRESETS[preset]
        cfg["model"].update({k: p[k] for k in ("n_embd", "n_layer", "n_head", "n_kv_head")})
        cfg["train"].update(batch=p["batch"], lr=p["lr"], warmup=p["warmup"])
    if getattr(a, "batch", 0):
        cfg["train"]["batch"] = int(a.batch)
    if getattr(a, "no_compile", False):
        cfg["train"]["compile"] = False
    if getattr(a, "save_every", None):
        cfg["train"]["save_every"] = int(a.save_every)
    if getattr(a, "log_every", None):
        cfg["train"]["log_every"] = int(a.log_every)
    n = getattr(a, "base_len", 0)
    if n:
        r = n / 64.0                       # 64 is the reference text/image length
        cfg["sched"]["base_len"] = dict(text=int(n), image=int(n),
                                        audio=max(4, int(round(96 * r / 4)) * 4),
                                        video=32)
        # video stays at 32: its base_len is frames x grid^2, so 512 would read 128 frames of
        # every clip. It is data-path only and currently dead in eval, so it is not worth the
        # decode. Raise it in build_codec if that ever changes.
    moe = getattr(a, "moe", 0)
    if moe:
        cfg["model"]["n_experts"] = int(moe)
        cfg["model"]["expert_map"] = {"text": 0, "image": 1, "audio": 2}
        mff = getattr(a, "moe_ffn", 0)
        if mff:
            cfg["model"]["moe_ffn"] = int(mff)
    hv = getattr(a, "hetero_vocab", 0)
    if hv:
        K = cfg["sched"]["K"]
        v = dict(cfg["vocab"])
        v["text"] = [int(cfg["vocab"]["text"])] * (K - 1) + [int(hv)]
        cfg["vocab"] = v
    sb = getattr(a, "shuffle_buf", 0)
    if sb:
        cfg["produce"]["shuffle_buf"] = int(sb)
    ev = getattr(a, "eval_every", None)
    if ev:
        cfg["train"]["eval_every"] = int(ev)
    pflr = getattr(a, "preflight_lr", None)
    if pflr is not None:
        cfg.setdefault("preflight", {})["lr"] = float(pflr)
    pfs = getattr(a, "preflight_steps", None)
    if pfs is not None:
        cfg.setdefault("preflight", {})["steps"] = int(pfs)
    if getattr(a, "no_preflight", False):
        cfg.setdefault("preflight", {})["enabled"] = False
    m = getattr(a, "holdout_mult", 0)
    if m:
        cfg["produce"]["holdout"] = {k: int(v * m) for k, v in cfg["produce"]["holdout"].items()}
    return cfg


def write_cfgs_all(cfg, budget=None, steps=None):
    return write_cfgs(cfg, budget=budget, steps=steps)


def default_cfg(budget=20000, out=os.path.join(RUN, "stream")):
    return dict(
        # three modalities: video was dropped because a frame is an image, so frames are encoded
        # as image samples instead of training a modality that measured below its copy prior.
        sched=dict(modalities=["text", "image", "audio"],
                   base_len=dict(text=64, image=64, audio=96), K=6, R=2, null_code=0),
        vocab=dict(text=257, image=257, audio=257),   # codebook_size + the NULL slot
        model=dict(n_embd=384, n_layer=6, n_head=6, n_kv_head=2, dropout=0.0,
                   qk_norm=True, mask_mode="block"),
        train=dict(steps=2000, batch=8, lr=3e-4, warmup=100, wd=0.01, clip=1.0,
                   log_every=10, eval_every=1000, save_every=500, compile=True),
        produce=dict(out_dir=out, sample_budget=budget, shard_samples=64,
                     max_ahead_shards=24, shuffle_buf=512, batch_docs=8, seed=3407,
                     holdout=dict(text=32, image=24, audio=16),
                     # 41.7 / 41.7 / 16.7 of SAMPLES, which is equal LIVE CODES per modality at the
                     # 50x / 50x / 30x token targets: text and image contribute 2016 packed positions
                     # each, audio 3024, so equal shares would not be equal codes. At the old 60/20/20
                     # the image expert was the binding constraint and the whole run had to be 1.83x
                     # larger to give it its tokens.
                     # Physics, chemistry and a second math-instruction source were carved OUT of
                     # their own modality rather than added on top: text still totals 0.4166 and
                     # image 0.4167, so the 41.7/41.7/16.7 sample budget - and with it the equal
                     # live-codes-per-modality argument above - is unchanged. Every pre-existing
                     # text source is scaled by 0.875, every pre-existing image source by 0.850.
                     mix=dict(text_fineweb=0.0728, text_cosmopedia=0.0425, text_wikipedia=0.0243,
                              text_math_web=0.0425, text_math_fine=0.0365, text_math_numina=0.0304,
                              text_arxiv=0.0365, text_code=0.0547, text_code_inst=0.0243,
                              text_physics=0.0208, text_chemistry=0.0208, text_math_inst=0.0104,
                              image_docvqa=0.0886, image_imagenet=0.0886, image_coco=0.0708,
                              image_wikiart=0.0531, image_flowers=0.0354, image_food=0.0177,
                              image_latex=0.0208, image_scienceqa=0.0417,
                              audio_libri=0.0625, audio_libritts=0.0417, audio_fleurs=0.0417,
                              audio_peoples=0.0208),
                     codebook_size=256, prefer_real=True,
                     K=None, R=None, base_len=None),
    )


def train_cfg(cfg, mode="strict", out_dir=None, steps=None):
    """The flat config one trainer reads: what write_cfgs puts on disk, built in memory.

    write_cfgs and the notebook both go through this, so a direct-feed run differs from a shard
    run in the data source and nothing else.
    """
    prod = cfg["produce"]
    t = dict(cfg["train"])
    if steps:
        t["steps"] = int(steps)
    t.update(sched=cfg["sched"], vocab=cfg["vocab"],
             model=dict(cfg["model"], mask_mode=mode),
             out_dir=out_dir or os.path.join(RUN, "train_" + mode),
             shard_dir=prod["out_dir"],
             producer_log=os.path.join(RUN, "producer.log"),
             eval_files={g: os.path.join(prod["out_dir"], f"eval_{g}.npz")
                         for g in prod["holdout"]},
             consume=False,
             stream_timeout=1800,
             preflight=cfg.get("preflight") or {})
    return t


def write_cfgs(cfg, budget=None, steps=None):
    from omni.schedule import OmniSchedule
    os.makedirs(os.path.join(RUN, "cfg"), exist_ok=True)
    sch = cfg["sched"]
    sched = OmniSchedule(base_len=sch["base_len"], K=sch["K"], R=sch["R"],
                         modalities=tuple(sch.get("modalities") or MODALITIES))
    prod = dict(cfg["produce"])
    if budget:
        prod["sample_budget"] = budget
    prod.update(base_len=sch["base_len"], K=sch["K"], R=sch["R"],
                codebook_size=cfg["produce"]["codebook_size"])
    prod.setdefault("only_sources", None)
    paths = dict(
        produce=os.path.join(RUN, "cfg", "produce.json"),
        train_block=os.path.join(RUN, "cfg", "train_block.json"),
        train_ar=os.path.join(RUN, "cfg", "train_ar.json"),
        train_strict=os.path.join(RUN, "cfg", "train_strict.json"),
        eval=os.path.join(RUN, "cfg", "eval.json"),
    )
    json.dump(prod, open(paths["produce"], "w"), indent=2)

    for arm, mode in (("train_block", "block"), ("train_ar", "ar"),
                      ("train_strict", "strict")):
        t = train_cfg(cfg, mode, os.path.join(RUN, arm), steps)
        json.dump(t, open(paths[arm], "w"), indent=2)

    json.dump(dict(block_ckpt=os.path.join(RUN, "train_block", "final.pt"),
                   ar_ckpt=os.path.join(RUN, "train_ar", "final.pt"),
                   strict_ckpt=os.path.join(RUN, "train_strict", "final.pt"),
                   eval_files=sorted(glob.glob(os.path.join(prod["out_dir"], "eval_*.npz"))),
                   # 0 = every held-out row. 48 of the ~320 rows a x4 carve produces threw
                   # away 85% of the eval signal, and the AR-parity score reads one slot per
                   # modality per row, so its error bar is set by this number.
                   n_samples=0, out=os.path.join(RUN, "eval_table.json")),
              open(paths["eval"], "w"), indent=2)
    print("wrote", ", ".join(paths.values()))
    return paths


def cmd_check(args):
    import torch
    from omni import model as M
    from omni.data import collate, synthetic_batch
    from omni.schedule import OmniSchedule
    from omni.train import build, n_params

    print("torch", torch.__version__, "| cuda", torch.cuda.is_available())
    M.selfcheck()

    cfg = default_cfg()
    sch = cfg["sched"]
    m, sched = build(dict(sched=sch, vocab=cfg["vocab"], model=cfg["model"]))
    print(f"model: {n_params(m)/1e6:.2f}M params | packed {sched.total_len} positions/sample")
    print(sched.describe())

    b = synthetic_batch(sched, cfg["vocab"], B=3, present_all=False)
    out = m(b)
    print("logits:", {f"{k[0]}_L{k[1]}": tuple(v.shape) for k, v in list(out.items())[:4]}, "...")

    prod = subprocess.run([sys.executable, "-u", "-m", "omni.producer", "--selfcheck",
                           _write_tmp(prod_cfg_only(cfg))],
                          cwd=HERE, capture_output=True, text=True)
    print(prod.stdout[-800:] or prod.stderr[-800:])
    assert prod.returncode == 0, "producer selfcheck failed"
    print("PIPELINE CHECK OK")


def prod_cfg_only(cfg):
    from omni.schedule import OmniSchedule
    sch = cfg["sched"]
    sched = OmniSchedule(base_len=sch["base_len"], K=sch["K"], R=sch["R"],
                         modalities=tuple(sch.get("modalities") or MODALITIES))
    p = {k: v for k, v in cfg["produce"].items() if k not in ("K", "R", "base_len")}
    p.update(base_len=sch["base_len"], K=sch["K"], R=sch["R"],
             codebook_size=cfg["produce"]["codebook_size"], only_sources=None)
    return p


def _write_tmp(obj):
    path = os.path.join(RUN, "cfg", "produce_selfcheck.json")
    os.makedirs(os.path.dirname(path), exist_ok=True)
    json.dump(obj, open(path, "w"), indent=2)
    return path


def _nshards(d):
    try:
        return sum(1 for f in os.listdir(d) if f.startswith("shard_") and f.endswith(".npz"))
    except FileNotFoundError:
        return 0


def cmd_produce(args):
    cfg = apply_overrides(default_cfg(), args)
    # ShardStream reads with consume=False, so shard files are never deleted: the rolling-buffer
    # cap (max_ahead_shards) can only ever block forever once it is hit. Batch production wants
    # the whole corpus on disk, so allow the cap to be raised explicitly.
    if getattr(args, "max_ahead", 0):
        cfg["produce"]["max_ahead_shards"] = int(args.max_ahead)
    paths = write_cfgs(cfg, budget=args.budget)
    os.makedirs(RUN, exist_ok=True)
    nworkers = max(1, args.workers)
    if nworkers == 1:
        log = open(os.path.join(RUN, "producer.log"), "w")
        p = subprocess.Popen([sys.executable, "-u", "-m", "omni.producer", paths["produce"]],
                             cwd=HERE, stdout=log, stderr=subprocess.STDOUT)
        print("producer pid", p.pid, "-> ", os.path.join(RUN, "producer.log"))
        if args.foreground:
            p.wait()
            return p.returncode
        print("background; training can start once EVAL_READY exists")
        return 0

    # ---- parallel: N processes, each reading a disjoint slice (index % N) of every stream.
    # The held-out blocks come from worker 0 only, and this process owns EVAL_READY/DONE, so a
    # half-finished fan-out can never look ready.
    base = json.load(open(paths["produce"]))
    stream_dir = base["out_dir"]                 # cfg from default_cfg() has no out_dir
    extra = sum(base["holdout"].values())        # worker 0's budget is spent on the eval blocks
    for f in ("EVAL_READY", "DONE", "STOP"):
        fp = os.path.join(stream_dir, f)
        if os.path.exists(fp):
            os.remove(fp)
    total = int(base["sample_budget"])
    per = total // nworkers
    procs, logs = [], []
    for i in range(nworkers):
        c = json.loads(json.dumps(base))
        c["sample_budget"] = per + (total - per * nworkers if i == 0 else 0) + (extra if i == 0 else 0)
        c["worker"] = [i, nworkers]
        cp = os.path.join(RUN, "cfg", f"produce_w{i}.json")
        json.dump(c, open(cp, "w"), indent=2)
        f = open(os.path.join(RUN, f"producer_w{i}.log"), "w")
        logs.append(f)
        if i:
            time.sleep(2.5)                  # spread the initial burst of stream opens
        procs.append(subprocess.Popen(
            [sys.executable, "-u", "-m", "omni.producer", cp, "--worker", f"{i}/{nworkers}"],
            cwd=HERE, stdout=f, stderr=subprocess.STDOUT))
    print(f"{nworkers} producers | budget {total} -> {per}/worker | logs run/producer_w*.log")
    if not args.foreground:
        print("background; training can start once EVAL_READY exists")
        return 0

    # emit() fills a shuffle buffer before feeding the writer, so the first shard cannot exist
    # until shuffle_buf + shard_samples samples have been pulled PER WORKER. Reporting only the
    # shard count makes that interval indistinguishable from a hang.
    first_at = int(base.get("shuffle_buf", 512)) + int(base.get("shard_samples", 64))
    print(f"  first shard needs ~{first_at} samples/worker ({first_at * nworkers:,} total, "
          f"shuffle_buf + shard_samples): 0 shards before that is normal", flush=True)
    t0, codes = time.time(), []
    try:
        while any(p.poll() is None for p in procs):
            time.sleep(10)
            done = _nshards(stream_dir)
            got = 0
            for i in range(nworkers):                 # the worker's own counter, not the file count
                try:
                    got += int(json.load(open(os.path.join(stream_dir, f"progress_w{i}.json")))[
                        "samples"])
                except Exception:                     # noqa: BLE001 - not written yet
                    pass
            el = max(1e-9, time.time() - t0)
            print(f"  {done} shards | {got:,}/{total:,} samples | "
                  f"{got / el:.1f}/s | {time.time() - t0:.0f}s", flush=True)
    except KeyboardInterrupt:
        for p in procs:
            p.terminate()
        raise
    codes = [p.wait() for p in procs]
    for f in logs:
        f.close()
    if any(codes):
        print("producer failure codes:", codes)
        return 1

    stats = []
    for i in range(nworkers):
        sp = os.path.join(stream_dir, f"stats_w{i}.json")
        if os.path.exists(sp):
            stats.append(json.load(open(sp)))
    samples = sum(s.get("samples", 0) for s in stats)
    shards = _nshards(stream_dir)
    mins = (time.time() - t0) / 60
    open(os.path.join(stream_dir, "EVAL_READY"), "w").write("1")
    out = dict(samples=samples, shards=shards, workers=nworkers,
               elapsed_min=round(mins, 2), samples_per_s=round(samples / max(mins * 60, 1e-9), 2),
               per_worker=[s.get("samples", 0) for s in stats],
               per_source={k: sum(s.get("per_source", {}).get(k, 0) for s in stats)
                           for k in (stats[0].get("per_source", {}) if stats else {})})
    json.dump(out, open(os.path.join(stream_dir, "stats.json"), "w"), indent=2)
    open(os.path.join(stream_dir, "DONE"), "w").write("1")
    print("DONE", {k: out[k] for k in ("samples", "shards", "workers", "elapsed_min",
                                       "samples_per_s")})
    return 0


def cmd_train(args):
    cfg = apply_overrides(default_cfg(), args)
    paths = write_cfgs(cfg, budget=args.budget, steps=args.steps)
    which = paths.get(f"train_{args.arm}") or paths["train_block"]
    if getattr(args, "arms", None):
        arms = [a.strip() for a in args.arms.split(",") if a.strip()]
        bad = [a for a in arms if a not in ("strict", "ar", "block")]
        if bad:
            raise SystemExit(f"unknown arms {bad}; pick from strict, ar, block")
        tcfg = json.load(open(which))
        live = dict(cfg["produce"])
        live["sched"], live["vocab"] = cfg["sched"], cfg["vocab"]
        live["holdout"] = cfg["produce"]["holdout"]
        # One doc per draw. `batch_docs=8` makes `Source.batch` block until it has 8 decodable
        # rows, and a slow modality (video downloads, audio decode) then stalls the whole stream
        # for minutes while every other source sits idle. Weights already control the mix.
        # With the threaded pump a slow source can no longer block the others, so pull in chunks:
        # one network round trip per 8 rows instead of per row.
        live["batch_docs"] = 8
        tcfg["arms"] = arms
        if getattr(args, "live", False):
            tcfg["live_cfg"] = live
        tcfg["out_dir"] = RUN              # duplex saves into run/train_{arm}/
        os.makedirs(os.path.join(RUN, "cfg"), exist_ok=True)
        which = os.path.join(RUN, "cfg", "train_duplex.json")
        json.dump(tcfg, open(which, "w"), indent=1)
        print(f"duplex: arms {arms} | live stream, no shards | out {RUN}")
    if getattr(args, "inproc", False):
        # In a notebook the subprocess buys nothing and costs a second interpreter: `pip` lands in
        # the kernel's python, the subprocess may resolve a different one, and the hard exit that
        # exists only to escape the subprocess would kill the kernel. Call the loop directly.
        from omni.train import run_duplex
        tcfg["hard_exit"] = False
        print("in-process: hard exit disabled, the cell keeps running after the last step")
        return run_duplex(tcfg, smoke=args.smoke or None)
    cmd = [sys.executable, "-u", "-m", "omni.train", which]
    if args.smoke:
        cmd += ["--smoke", str(args.smoke)]
    return subprocess.call(cmd, cwd=HERE)


def cmd_eval(args):
    cfg = apply_overrides(default_cfg(), args)
    paths = write_cfgs(cfg)
    return subprocess.call([sys.executable, "-u", "-m", "omni.eval", paths["eval"]], cwd=HERE)


def cmd_all(args):
    """Producer in the background, then wait for the held-out blocks, then both arms, then eval."""
    cfg = apply_overrides(default_cfg(), args)
    paths = write_cfgs(cfg, budget=args.budget, steps=args.steps)
    os.makedirs(RUN, exist_ok=True)
    stream_dir = cfg["produce"]["out_dir"]
    log = open(os.path.join(RUN, "producer.log"), "w")
    p = subprocess.Popen([sys.executable, "-u", "-m", "omni.producer", paths["produce"]],
                         cwd=HERE, stdout=log, stderr=subprocess.STDOUT)
    print("producer pid", p.pid)
    t0 = time.time()
    while not os.path.exists(os.path.join(stream_dir, "EVAL_READY")):
        if p.poll() is not None:
            print("producer exited early; tail:")
            print("".join(open(os.path.join(RUN, "producer.log")).readlines()[-25:]))
            return 1
        if args.wait_timeout and time.time() - t0 > args.wait_timeout:
            print(f"no held-out blocks after {args.wait_timeout}s; tail:")
            print("".join(open(os.path.join(RUN, "producer.log")).readlines()[-25:]))
            return 1
        time.sleep(5)
    print(f"EVAL_READY after {time.time() - t0:.0f}s")
    rc = 0
    for arm in ("block", "ar"):
        cmd = [sys.executable, "-u", "-m", "omni.train", paths[f"train_{arm}"]]
        if args.smoke:
            cmd += ["--smoke", str(args.smoke)]
        rc |= subprocess.call(cmd, cwd=HERE)
    if not args.smoke:
        rc |= subprocess.call([sys.executable, "-u", "-m", "omni.eval", paths["eval"]], cwd=HERE)
    open(os.path.join(stream_dir, "STOP"), "w").write("1")     # let a back-pressured producer exit
    p.terminate()
    return rc


def build_parser():
    """The CLI, exposed so a notebook can build the same cfg in-process: no cfg json on disk."""
    ap = argparse.ArgumentParser()
    ap.add_argument("cmd", choices=["check", "produce", "train", "eval", "all", "cfg"])
    ap.add_argument("--budget", type=int, default=None)
    ap.add_argument("--steps", type=int, default=None)
    ap.add_argument("--smoke", type=int, default=0)
    ap.add_argument("--arm", choices=["block", "ar", "strict"], default="block")
    ap.add_argument("--inproc", action="store_true",
                    help="run the duplex loop in this process instead of spawning a trainer; use "
                         "from a notebook so the kernel's python, packages and output are kept")
    ap.add_argument("--live", action="store_true",
                    help="feed the duplex run from the Hub directly instead of shards. Measured "
                         "1.5 samples/s in one process vs 45 for a sharded producer, so this is "
                         "for a short kill test where a corpus is unwanted, not for a real run.")
    ap.add_argument("--log-every", type=int, default=None, dest="log_every",
                    help="steps between metric lines. Default 10 = one line every ~100 s on a live "
                         "stream; pass 1 to get a line every step so silence always means a stall")
    ap.add_argument("--save-every", type=int, default=None, dest="save_every",
                    help="checkpoint interval; the live-stream path has no replay, so this is "
                         "the only way to survive a Colab session limit")
    ap.add_argument("--shuffle-buf", type=int, default=0, dest="shuffle_buf",
                    help="rows buffered per worker before the first shard can be written "
                         "(default 512; lower it for smoke corpora)")
    ap.add_argument("--eval-every", type=int, default=None, dest="eval_every",
                    help="validation loss every N steps on the held-out shards (default 1000)")
    ap.add_argument("--arms", default=None,
                    help="comma list to train in one process off the live stream, e.g. strict,ar. "
                         "No shard corpus is produced or read: both arms step the same batch.")
    ap.add_argument("--foreground", action="store_true")
    ap.add_argument("--workers", type=int, default=1,
                    help="producer processes; each reads index i mod N of every stream")
    ap.add_argument("--moe", type=int, default=0,
                    help="n_experts (1 per modality). 0 = dense trunk, the old layout")
    ap.add_argument("--moe-ffn", type=int, default=0, dest="moe_ffn",
                    help="expert SwiGLU width h. 0 = mlp_ratio 4.0 (2048 at d=768). 4880 gives "
                         "359.8M per expert at d=1536/L=16, i.e. the 360M rung")
    ap.add_argument("--hetero-vocab", type=int, default=0, dest="hetero_vocab",
                    help="finest-level text vocab, e.g. 8000 or 32000; 0 = one codebook per level")
    ap.add_argument("--preflight-lr", type=float, default=None, dest="preflight_lr",
                    help="preflight learning rate. Default 2x the preset's train lr, clamped to "
                         "[3e-4, 3e-3] - a fixed 3e-3 is 15x too hot for md")
    ap.add_argument("--preflight-steps", type=int, default=None, dest="preflight_steps",
                    help="test C steps that run before training (default 300)")
    ap.add_argument("--no-preflight", action="store_true", dest="no_preflight",
                    help="skip test C - only to debug the trainer itself")
    ap.add_argument("--preset", choices=sorted(PRESETS), default=None,
                    help="sm=10.44M (debug), md=84.6M (kill test), lg=454M")
    ap.add_argument("--base-len", type=int, default=0,
                    help="text/image finest length; audio scales x1.5, video stays 32; "
                         "0 keeps 64 (504 packed positions). 1024 => ~7.1k positions")
    ap.add_argument("--holdout-mult", type=int, default=0,
                    help="multiply the held-out block sizes (4 => 128/96/64/32, SE ~0.028)")
    ap.add_argument("--batch", type=int, default=0,
                    help="micro-batch. This is the dominant throughput knob: the step is "
                         "launch-bound long before it is FLOP-bound, so 8 -> 32-64 buys most of "
                         "the missing TFLOPS. Raise lr ~sqrt(batch/8) when you do.")
    ap.add_argument("--no-compile", action="store_true", dest="no_compile",
                    help="train eager instead of torch.compile(model). Measured on an 8 GB "
                         "RTX 3070 laptop it is 1.7x slower per step (186 vs 323 ms at "
                         "batch 16) and needs more VRAM; compile costs ~8-75 s at startup")
    ap.add_argument("--max-ahead", type=int, default=0,
                    help="producer rolling-buffer cap in shards; 0 keeps the default 24, "
                         "which blocks forever when nothing deletes shards (batch runs: 100000)")
    ap.add_argument("--wait-timeout", type=int, default=3600)
    return ap


def main(argv=None):
    """argv=None keeps the CLI; a notebook passes a list and never leaves the kernel."""
    a = build_parser().parse_args(argv)
    if a.cmd == "cfg":
        write_cfgs(apply_overrides(default_cfg(), a), a.budget)
        return 0
    return {"check": cmd_check, "produce": cmd_produce, "train": cmd_train,
            "eval": cmd_eval, "all": cmd_all}[a.cmd](a) or 0


if __name__ == "__main__":
    sys.exit(main())


In [ ]:
%%writefile .ncm_stage/omni/__init__.py
"""omni - next-concept multimodal model.

Import shim, first line of defence: keep TensorFlow out of the process.

torch imports `opt_einsum`, whose `backends/__init__` eagerly imports every backend, including
`backends/tensorflow.py` (`import tensorflow as tf` inside a try/except ImportError). On Colab TF
is installed, so that import succeeds: two `absl` initialisations, ~600 MB RSS, ~3 s of startup,
and on a healthy GPU TF claims VRAM on import. Nothing here uses TF.

Poisoning the name with None makes `import tensorflow` raise ImportError, which is exactly the
branch opt_einsum already handles, so its TF backend degrades to `tf = None`.

    NCM_ALLOW_TF=1   keep TensorFlow importable (debugging a TF path)
"""

import os as _os
import sys as _sys

if _os.environ.get("NCM_ALLOW_TF", "") != "1":
    # setdefault, not assignment: if the host already imported TF, leave it alone.
    # None is the documented way to poison an import; pyright types the mapping as ModuleType.
    _sys.modules.setdefault("tensorflow", None)          # type: ignore[arg-type]


In [ ]:
%%writefile .ncm_stage/omni/schedule.py
"""Modality-aware level schedule: level k = the same semantic altitude in every modality.

The paper's text schedule makes level k have l_k positions. For omni we cannot give two
modalities the same *length*, because 30 s of audio and 64 characters of text are not
comparable. What we can hold fixed is the COMPRESSION RATIO: level k in every modality is
the input coarsened by R**(K-1-k). That is the binding that makes cross-modal attention at a
level mean something. Lengths are then derived per modality:

    l[m][k] = max(1, round(base_len[m] / R**(K-1-k)))

Packing order is level-major: [level 0: text|image|audio|video][level 1: ...]...[level K-1].
So one block = one altitude across all modalities, and the block-diagonal mask lets level k
see the whole level-k block (every modality) but nothing finer. That single line of mask code
is the entire omni thesis; `cross_level=True` turns it into the ablation.
"""
from dataclasses import dataclass, field
from typing import Dict, Sequence, Tuple

import torch

# Video is deliberately absent: a frame is an image, so video samples are encoded as
# image samples and there is no separate code distribution to model or route.
MODALITIES = ("text", "image", "audio")


@dataclass
class OmniSchedule:
    base_len: Dict[str, int]                 # finest-level length per modality
    K: int = 6
    R: int = 2
    null_code: int = 0                       # reserved in every modality's codebook
    modalities: Tuple[str, ...] = MODALITIES
    prefix_len: int = 0

    def __post_init__(self):
        assert self.R >= 2 and self.K >= 1
        for m in self.modalities:
            assert m in self.base_len, m
        # level 0 is the COARSEST (short) and level K-1 the finest (== base_len), so lengths must
        # be non-decreasing in k. Getting this direction wrong is exactly what `check` caught.
        for m in self.modalities:
            lens = self.level_lens(m)
            assert all(a <= b for a, b in zip(lens, lens[1:])), (m, lens)

    # ---------------------------------------------------------------- geometry
    def level_len(self, m: str, k: int) -> int:
        return max(1, round(self.base_len[m] / self.R ** (self.K - 1 - k)))

    def level_lens(self, m: str) -> Tuple[int, ...]:
        return tuple(self.level_len(m, k) for k in range(self.K))

    def block_width(self, k: int) -> int:
        return sum(self.level_len(m, k) for m in self.modalities)

    def block_offsets(self) -> Tuple[int, ...]:
        out, c = [], self.prefix_len
        for k in range(self.K):
            out.append(c)
            c += self.block_width(k)
        return tuple(out)

    @property
    def total_len(self) -> int:
        return self.prefix_len + sum(self.block_width(k) for k in range(self.K))

    def offsets(self, m: str) -> Tuple[int, ...]:
        """Within-modality flat offset of each level, used by the shard layout."""
        out, c = [], 0
        for l in self.level_lens(m):
            out.append(c)
            c += l
        return tuple(out)

    def flat_len(self, m: str) -> int:
        return sum(self.level_lens(m))

    def positions(self):
        """Per packed position: (level k, modality index, position within that modality-level)."""
        lev, mod, pos = [], [], []
        for k in range(self.K):
            for mi, m in enumerate(self.modalities):
                l = self.level_len(m, k)
                lev += [k] * l
                mod += [mi] * l
                pos += list(range(l))
        if self.prefix_len:
            lev = [-1] * self.prefix_len + lev
            mod = [len(self.modalities)] * self.prefix_len + mod
            pos = list(range(self.prefix_len)) + pos
        return (torch.tensor(lev), torch.tensor(mod), torch.tensor(pos))

    # ---------------------------------------------------------------- masks
    def dense_mask(self, dtype=torch.float32, device="cpu", cross_level=False) -> torch.Tensor:
        lev, _, _ = self.positions()
        q, kv = lev[:, None], lev[None, :]
        allowed = (kv == -1) | (kv == q)
        if cross_level:
            allowed = allowed | ((q >= 0) & (kv >= 0) & (kv < q))
        m = torch.zeros(self.total_len, self.total_len, dtype=dtype)
        m.masked_fill_(~allowed, float("-inf"))
        return m.view(1, 1, self.total_len, self.total_len).to(device)

    def flex_block_mask(self, B, H, device="cuda", cross_level=False):
        from torch.nn.attention.flex_attention import create_block_mask
        lev, _, _ = self.positions()
        lev = lev.to(device)

        def mask_mod(b, h, q_idx, kv_idx):
            ql, kl = lev[q_idx], lev[kv_idx]
            allowed = (kl == -1) | (kl == ql)
            if cross_level:
                allowed = allowed | ((ql >= 0) & (kl >= 0) & (kl < ql))
            return allowed

        return create_block_mask(mask_mod, B=B, H=H, Q_LEN=self.total_len,
                                 KV_LEN=self.total_len, device=device)

    def describe(self) -> str:
        rows = []
        for m in self.modalities:
            rows.append(f"{m:6s} base={self.base_len[m]:4d} levels={self.level_lens(m)}")
        tot = {m: self.flat_len(m) for m in self.modalities}
        return ("K=%d R=%d  total packed positions=%d\n" % (self.K, self.R, self.total_len)
                + "\n".join(rows) + f"\nper-sample token counts {tot}")


In [ ]:
%%writefile .ncm_stage/omni/codecs.py
"""Codecs: raw text/image/audio/video -> K nested code levels. Frozen, never trained here.

Hard contract, the one thing this file exists to guarantee:

    encode(obj) -> list of int arrays, one per level, with len(level_k) == schedule.level_len(m, k)

Every level is produced by *pooling the finest representation*, never by independently
re-encoding at a different size. That is what makes level k "compression R^(K-1-k)" instead of
"level k is a different thumbnail", and it is why the cross-modal level alignment means
something. Violations are caught in producer.encode_sample, which logs and masks the modality.

Tiers:
  TextByteCodec        UTF-8 bytes at the finest level (exact), hashed spans coarser
  ImagePyramidCodec    patch-mean pyramid, 6 bins/channel -> <256 codes, decodable (PSNR real)
  VideoPyramidCodec    frame x patch pyramid, same pooling ladder, decodable per frame
  AudioMelPyramidCodec log-mel frames quantised against a FIXED seeded codebook (1 code/frame)
  SpanHashCodec        no dependencies, no semantics: plumbing test only

The image/audio/video codebooks here are stand-ins, and the README says so. The ladder geometry
is the real thing; the codebooks are what a trained VQ (NCM text RVQ, Infinity image VQ, DAC
audio) replaces, and `build_codec` is the single place to swap them.
"""
import hashlib

import numpy as np


class Codec:
    """Interface. K / R / base_len must match the schedule the producer was configured with."""
    name = "codec"
    kind = "synthetic"
    exact_at_finest = False
    decodable = False

    def __init__(self, K=6, R=2, codebook_size=256, base_len=64):
        self.K, self.R, self.V = K, R, codebook_size
        self.base_len = base_len

    def level_len(self, k):
        return max(1, round(self.base_len / self.R ** (self.K - 1 - k)))

    def encode(self, obj):
        raise NotImplementedError

    def decode(self, codes):
        raise NotImplementedError

    def available(self):
        return True

    def info(self):
        return dict(name=self.name, kind=self.kind, K=self.K, R=self.R, V=self.V,
                    base_len=self.base_len, decodable=self.decodable,
                    exact_at_finest=self.exact_at_finest)


# ------------------------------------------------------------------ shared machinery
def pool_feats(feat: np.ndarray, l: int) -> np.ndarray:
    """[n, ...] -> [l, ...] by mean-pooling contiguous groups; resample if not divisible."""
    n = len(feat)
    if l == n:
        return feat
    if n == 0:
        return np.zeros((l,) + feat.shape[1:], dtype=np.float32)
    if l < n and n % l == 0:
        return feat.reshape(l, n // l, *feat.shape[1:]).mean(1)
    idx = np.linspace(0, n - 1, l).round().astype(int)          # upsampling or non-integer ratio
    return feat[idx]


def quant_rgb(feat: np.ndarray, V: int) -> np.ndarray:
    """[l, 3] in [0,1] -> [l] codes in 1..B^3. 1-based: index 0 is the NULL slot, never a code."""
    B = max(2, int(round(V ** (1 / 3))))
    q = np.clip((np.asarray(feat, dtype=np.float32) * B).astype(np.int32), 0, B - 1)
    return (q[..., 0] * B * B + q[..., 1] * B + q[..., 2] + 1).astype(np.int32)


def dequant_rgb(codes, V: int) -> np.ndarray:
    B = max(2, int(round(V ** (1 / 3))))
    c = np.asarray(codes, dtype=np.int32) - 1
    return np.stack([c // (B * B), (c // B) % B, c % B], -1).astype(np.float32) / B


def patch_means(img, grid: int) -> np.ndarray:
    """PIL image -> [grid*grid, 3] patch means in [0,1]."""
    g = np.asarray(img.convert("RGB").resize((grid * 8, grid * 8), resample=2),
                   dtype=np.float32) / 255.0
    return g.reshape(grid, 8, grid, 8, 3).mean(axis=(1, 3)).reshape(-1, 3)


# ------------------------------------------------------------------ text
class SpanHashCodec(Codec):
    """Any bytes -> deterministic codes per level. No semantics: validates plumbing only."""
    kind, name = "synthetic", "spanhash"

    def __init__(self, K=6, R=2, codebook_size=256, base_len=64, seed=0):
        super().__init__(K, R, codebook_size, base_len)
        self.seed = seed

    def _codes(self, buf: bytes, l: int, k: int) -> np.ndarray:
        if len(buf) == 0:
            return np.ones(l, dtype=np.int32)                 # 1 = the first real code, not NULL
        span = max(1, int(np.ceil(len(buf) / l)))
        out = np.empty(l, dtype=np.int32)
        for i in range(l):
            chunk = buf[i * span:(i + 1) * span]
            h = hashlib.blake2b(bytes([k]) + chunk, digest_size=8,
                                salt=self.seed.to_bytes(8, "little")).digest()
            out[i] = int.from_bytes(h, "little") % self.V + 1
        return out

    def encode(self, obj):
        raw = obj if isinstance(obj, (bytes, bytearray)) else str(obj).encode("utf-8", "ignore")
        return [self._codes(raw, self.level_len(k), k) for k in range(self.K)]


class TextByteCodec(Codec):
    """UTF-8 bytes at the finest level; coarser levels hash R^k spans.

    The finest level decodes exactly, which is the number worth reporting (the paper's RVQ
    tokenizer gets ~99.99% exact text reconstruction; bytes get 100% by construction, so the
    honest claim here is only "the bottom of the ladder is lossless").
    """
    kind, name, exact_at_finest, decodable = "real", "text_byte", True, True

    def __init__(self, K=6, R=2, codebook_size=256, base_len=64, seed=1):
        super().__init__(K, R, codebook_size, base_len)
        self.seed = seed

    def encode(self, text):
        raw = (text or "").encode("utf-8", "ignore")[: self.base_len]
        hasher = SpanHashCodec(self.K, self.R, self.V, self.base_len, self.seed)
        out = []
        for k in range(self.K):
            l = self.level_len(k)
            if k == self.K - 1:                        # lossless bottom, shifted to 1..256
                b = np.zeros(l, dtype=np.int32)
                b[: len(raw)] = np.frombuffer(raw, dtype=np.uint8)[:l].astype(np.int32) + 1
                out.append(b)
            else:
                out.append(hasher._codes(raw, l, k))
        return out

    def decode(self, codes):
        b = np.clip(np.asarray(codes[-1], dtype=np.int32) - 1, 0, 255).astype(np.uint8)
        return b.tobytes().rstrip(b"\x00").decode("utf-8", "ignore")


# ------------------------------------------------------------------ image
class ImagePyramidCodec(Codec):
    """PIL image -> patch-mean pyramid. Level k pools the finest grid by R^(K-1-k).

    decode() inverts the finest level, so per-level PSNR against the real image is a number you
    can report rather than a placeholder.
    """
    kind, name, decodable = "real", "image_pyramid", True

    def __init__(self, K=6, R=2, codebook_size=256, base_len=64):
        super().__init__(K, R, codebook_size, base_len)
        self.grid = max(1, int(round(np.sqrt(base_len))))

    def _feats(self, img):
        return patch_means(img, self.grid)                       # [grid*grid, 3]

    def encode(self, img):
        f = self._feats(img)
        return [quant_rgb(pool_feats(f, self.level_len(k)), self.V) for k in range(self.K)]

    def decode(self, codes, side=None):
        px = dequant_rgb(codes[-1], self.V)                      # [grid*grid, 3]
        g = side or self.grid
        return px[: g * g].reshape(g, g, 3)


# ------------------------------------------------------------------ video
class VideoPyramidCodec(Codec):
    """Frames -> (frame x patch) pyramid. Level k pools *both* the frame rate and the patch grid.

    Video is data-path only in this project, but it has to obey the same ladder: level k of a
    clip must be the clip at R^(K-1-k) the spatio-temporal detail, or its slots are not
    comparable to the image slots at the same level.
    """
    kind, name, decodable = "real", "video_pyramid", True

    def __init__(self, K=6, R=2, codebook_size=256, base_len=32, n_frames=8, grid=2):
        super().__init__(K, R, codebook_size, base_len)
        self.n_frames, self.grid = n_frames, grid
        assert n_frames * grid * grid == base_len, "frames x patches must equal base_len"

    def _feats(self, frames):
        pick = np.linspace(0, len(frames) - 1, self.n_frames).round().astype(int)
        feats = [patch_means(frames[i], self.grid) for i in pick]
        return np.concatenate(feats, 0)                          # [n_frames*grid^2, 3]

    def encode(self, frames):
        f = self._feats(frames)
        return [quant_rgb(pool_feats(f, self.level_len(k)), self.V) for k in range(self.K)]

    def decode(self, codes):
        px = dequant_rgb(codes[-1], self.V)                      # [n_frames*grid^2, 3]
        return px.reshape(self.n_frames, self.grid, self.grid, 3)


# ------------------------------------------------------------------ audio
def _mel_fb(n_mels: int, n_fft: int, sr: int) -> np.ndarray:
    """[n_fft//2+1, n_mels] triangular mel filterbank (HTK scale)."""
    def hz2mel(f):
        return 2595.0 * np.log10(1.0 + np.asarray(f) / 700.0)

    def mel2hz(m):
        return 700.0 * (10.0 ** (np.asarray(m) / 2595.0) - 1.0)

    hz = mel2hz(np.linspace(hz2mel(0.0), hz2mel(sr / 2), n_mels + 2))
    bins = np.clip(np.floor((n_fft + 1) * hz / sr).astype(int), 0, n_fft // 2)
    fb = np.zeros((n_fft // 2 + 1, n_mels), dtype=np.float32)
    for i in range(n_mels):
        l, c, r = bins[i], max(bins[i] + 1, bins[i + 1]), max(bins[i + 1] + 2, bins[i + 2])
        l, c, r = min(l, n_fft // 2), min(c, n_fft // 2), min(r, n_fft // 2 + 1)
        if c > l:
            fb[l:c, i] = np.linspace(0, 1, c - l, endpoint=False)
        if r > c:
            fb[c:r, i] = np.linspace(1, 0, r - c, endpoint=False)
    return fb


def _mel_numpy(x: np.ndarray, sr, n_mels=32, n_fft=400, hop=160) -> np.ndarray:
    """Log-mel without torchaudio, so a torch/torchaudio version skew cannot kill the pipeline."""
    if sr and sr != 16000:
        n = max(1, int(round(len(x) * 16000 / sr)))
        x = np.interp(np.linspace(0, len(x) - 1, n), np.arange(len(x)), x).astype(np.float32)
    if len(x) < n_fft:
        x = np.pad(x, (0, n_fft - len(x)))
    win = np.hanning(n_fft).astype(np.float32)
    frames = np.lib.stride_tricks.sliding_window_view(x, n_fft)[::hop] * win
    spec = np.abs(np.fft.rfft(frames, axis=1)).astype(np.float32) ** 2
    return np.log(spec @ _mel_fb(n_mels, n_fft, 16000) + 1e-6).T          # [n_mels, T]


class AudioMelPyramidCodec(Codec):
    """Waveform -> log-mel frames, one code per frame, levels = frame-rate pyramid.

    Audio's hierarchy is temporal *rate*, not spatial resolution: level k is the spectrogram at
    1/R^(K-1-k) the frame rate. One code per frame means the codebook has to carry a whole mel
    column, so this uses a fixed seeded random codebook (nearest prototype in normalised mel
    space). The ladder is real; the codebook is a stand-in for DAC/EnCodec.
    """
    kind, name, decodable = "real", "audio_mel_pyramid", True

    def __init__(self, K=6, R=2, codebook_size=256, base_len=96, n_mels=32, seed=7):
        super().__init__(K, R, codebook_size, base_len)
        self.n_mels = n_mels
        g = np.random.default_rng(seed)
        C = g.standard_normal((n_mels, codebook_size)).astype(np.float32)
        self.C = C / (np.linalg.norm(C, axis=0, keepdims=True) + 1e-8)     # [n_mels, V]

    def _mel(self, obj):
        wav, sr = obj
        x = np.asarray(wav, dtype=np.float32).flatten()
        if x.size == 0:
            return np.zeros((self.n_mels, 1), dtype=np.float32)
        try:
            import torch
            import torchaudio
            w = torch.as_tensor(x)
            if sr and sr != 16000:
                w = torchaudio.functional.resample(w, sr, 16000)
            mel = torchaudio.transforms.MelSpectrogram(
                16000, n_fft=400, hop_length=160, n_mels=self.n_mels)(w)
            return torch.log(mel + 1e-6).numpy()                            # [n_mels, T]
        except Exception:                          # torchaudio missing or built for another torch
            return _mel_numpy(x, sr, n_mels=self.n_mels)

    def encode(self, obj):
        mel = self._mel(obj).T                                             # [T, n_mels]
        out = []
        for k in range(self.K):
            x = pool_feats(mel, self.level_len(k))                         # [l, n_mels]
            x = x - x.mean(0, keepdims=True)
            x = x / (np.linalg.norm(x, axis=1, keepdims=True) + 1e-6)
            out.append((x @ self.C).argmax(1).astype(np.int32) + 1)        # 1..V, never NULL
        return out

    def decode(self, codes):
        idx = np.clip(np.asarray(codes[-1], dtype=np.int32) - 1, 0, self.V - 1)
        return self.C[:, idx].T                                            # [l, n_mels]


def build_codec(modality, K, R, V, base_len, prefer_real=True):
    """Pick the strongest codec that is importable here. Never silently pretend."""
    if modality == "text":
        return TextByteCodec(K, R, V, base_len) if prefer_real else SpanHashCodec(K, R, V, base_len)
    if modality == "image":
        return ImagePyramidCodec(K, R, V, base_len)
    if modality == "video":
        grid = 2 if base_len % 4 == 0 else 1
        frames = max(2, base_len // (grid * grid))
        return VideoPyramidCodec(K, R, V, frames * grid * grid, n_frames=frames, grid=grid)
    if modality == "audio":
        return AudioMelPyramidCodec(K, R, V, base_len)
    return SpanHashCodec(K, R, V, base_len)


In [ ]:
%%writefile .ncm_stage/omni/producer.py
"""Omni streaming producer: HF sources -> frozen codecs -> nested code shards.

Same architecture as the DAPT stream producer, with one structural change: the unit written to
disk is not a 2048-token text block but a SAMPLE carrying K code levels for each modality that
sample contains (text / image / audio / video). Shards are .npz, self-describing (the schedule
travels in every file), and the training process can start as soon as EVAL_READY appears.

Why a separate process: image pyramid + mel encoding are Python/numpy heavy and would fight the
training loop for the GIL. Why rolling shards and not one big precompute: you can start training
a minute in, and a crash costs one shard, not a CPU-day.

Run:  python -m omni.producer cfg.json [--selfcheck]
"""
import argparse
import json
import os
import random
import sys
import time
import traceback

import numpy as np

from omni.codecs import build_codec
from omni.schedule import MODALITIES, OmniSchedule

HERE = os.path.dirname(os.path.abspath(__file__))


def log(*a):
    print(time.strftime("%H:%M:%S"), *a, flush=True)


# ------------------------------------------------------------------ sources
def _s(path, split="train", **kw):
    """Streaming loader, resolved lazily so one bad dataset id cannot kill the producer.

    `path` (not `name`) so callers can pass the config through the `name=` keyword the way
    load_dataset spells it: _s("hf-internal-testing/librispeech_asr_dummy", name="clean").
    """
    def load():
        from datasets import load_dataset
        return load_dataset(path, split=split, streaming=True, **kw)
    return load


def _stream(name, config=None, take=None):
    def load():
        from datasets import load_dataset
        ds = load_dataset(name, config, split="train", streaming=True)
        return ds.take(take) if take else ds
    return load


def _raw(ids, config=None, split="train", column=None, **kw):
    """Like `_stream`, but leaves audio/video **undecoded**.

    `datasets` >= 4 needs torchcodec to decode media columns, and torchcodec only matches a
    narrow range of torch builds. We never want that dependency: the bytes go to soundfile
    (`audio`) or PyAV (`video`) ourselves. Accepts a list of ids and tries them in order, so a
    rename costs one group and not the run.
    """
    if isinstance(ids, str):
        ids = [ids]

    def load():
        from datasets import features, load_dataset
        feat = {"audio": features.Audio(decode=False),
                "video": features.Video(decode=False)}.get(column)
        err = None
        for i in ids:
            try:
                ds = load_dataset(i, config, split=split, streaming=True, **kw)
                return ds.cast_column(column, feat) if feat else ds
            except Exception as e:                             # noqa: BLE001
                err = e
                log(f"[source] {i} ({split}) unavailable: {type(e).__name__}: {str(e)[:70]}")
        raise err
    return load


def _txt(ex):
    for k in ("text", "content", "raw_content", "document"):
        v = ex.get(k)
        if isinstance(v, str) and v.strip():
            return {"text": v}
    return {}


def _two(*keys):
    """Join the first two non-empty string fields: problem+solution, question+answer, prompt+response.

    Math, physics and code corpora all split the source material across paired fields; reading only
    `text` on those datasets yields nothing at all.
    """
    def fmt(ex):
        parts = []
        for k in keys:
            v = ex.get(k)
            if isinstance(v, str) and v.strip():
                parts.append(v.strip())
            elif isinstance(v, list) and v and isinstance(v[0], str):
                parts.append(v[0].strip())
        return {"text": "\n\n".join(parts)} if parts else {}
    return fmt


def _img_text(img_keys=("image", "img"), cap_keys=("text", "caption", "captions", "alt_text",
                                                   "question", "label")):
    """Image + the best available string beside it. Field names differ across every vision set."""
    def fmt(ex):
        img = next((ex.get(k) for k in img_keys if ex.get(k) is not None), None)
        if img is None:
            return {}
        for k in cap_keys:
            v = ex.get(k)
            if isinstance(v, str) and v.strip():
                return {"image": img, "text": v.strip()}
            if isinstance(v, dict):
                inner = next((str(x) for x in v.values() if isinstance(x, str) and x.strip()), "")
                if inner:
                    return {"image": img, "text": inner}
        return {"image": img}
    return fmt


def _img_cap(ex):
    img = ex.get("image")
    if img is None:
        img = ex.get("img")                                    # cifar-style column name
    if img is None:
        return {}
    cap = ""
    for k in ("text", "caption", "captions", "alt_text", "label"):
        v = ex.get(k)
        if isinstance(v, str) and v.strip():
            cap = v
            break
    return {"image": img, "text": str(cap)} if cap else {"image": img}


def _scienceqa(ex):
    """ScienceQA: an image with a multiple-choice question whose answer indexes `choices`.

    `_img_text` would caption the image with the question alone and throw the options away, and the
    answer is an index, not a string - so the pair has to be assembled here.
    """
    img = ex.get("image")
    if img is None:
        return {}
    q = ex.get("question")
    q = q.strip() if isinstance(q, str) else ""
    ch = ex.get("choices") if isinstance(ex.get("choices"), list) else []
    a = ex.get("answer")
    text = q
    if q and ch:
        opts = "\n".join(f"{chr(65 + i)}. {c}" for i, c in enumerate(ch) if isinstance(c, str))
        ans = ch[a] if isinstance(a, int) and 0 <= a < len(ch) else ""
        text = f"{q}\n{opts}" + (f"\nAnswer: {ans}" if ans else "")
    if not text:
        for k in ("solution", "lecture", "hint"):
            v = ex.get(k)
            if isinstance(v, str) and v.strip():
                text = v.strip()
                break
    return {"image": img, "text": text} if text else {"image": img}


def _decode_audio(a):
    """(raw bytes|path|already decoded) -> (float32 mono, sr). None means drop the sample."""
    if isinstance(a, dict):
        raw, path = a.get("bytes"), a.get("path")
        try:
            import io
            import soundfile as sf
            wav, sr = sf.read(io.BytesIO(raw) if raw else path, dtype="float32", always_2d=True)
            return wav[:, 0], int(sr)
        except Exception as e:                                 # noqa: BLE001
            log(f"[audio] decode failed: {type(e).__name__}: {str(e)[:70]}")
            return None
    if isinstance(a, (list, tuple)) and len(a) == 2:
        return np.asarray(a[0], dtype=np.float32).flatten(), int(a[1])
    return None


def _audio_text(ex):
    wav = _decode_audio(ex.get("audio"))
    if wav is None:
        return {}
    t = ""
    for k in ("text", "transcription", "transcript", "sentence", "text_normalized",
              "text_original", "raw_transcription"):
        v = ex.get(k)
        if isinstance(v, str) and v.strip():
            t = v
            break
    return {"audio": wav, "text": t} if t else {"audio": wav}


def _local_path(src):
    """hf://datasets/<repo>@<rev>/<file> -> a real local file PyAV can open."""
    if not (isinstance(src, str) and src.startswith("hf://")):
        return src
    from huggingface_hub import hf_hub_download
    body = src[len("hf://datasets/"):]
    repo, rest = body.split("@", 1)
    rev, fname = rest.split("/", 1)
    return hf_hub_download(repo_id=repo, filename=fname, repo_type="dataset", revision=rev)


def _video_frames(ex, max_frames=8):
    """Clip -> list of PIL frames. Video is data-path only, so a decode failure just drops it."""
    v = ex.get("video") or ex.get("videos") or ex.get("frames")
    if v is None:
        return {}
    if isinstance(v, (list, tuple)) and v and hasattr(v[0], "convert"):
        frames = list(v)[:max_frames]                         # already decoded by `datasets`
    else:
        try:
            import io
            import av
            src = v.get("bytes") if isinstance(v, dict) else v
            if not src:
                src = v.get("path") if isinstance(v, dict) else v
            if isinstance(src, (bytes, bytearray)):
                src = io.BytesIO(src)
            src = _local_path(src)
            if src is None:
                return {}
            frames = []
            with av.open(src) as c:
                for i, fr in enumerate(c.decode(video=0)):
                    if i % 3 == 0:
                        frames.append(fr.to_image())
                    if len(frames) >= max_frames:
                        break
        except Exception as e:                                # noqa: BLE001
            log(f"[video] decode failed: {type(e).__name__}: {str(e)[:70]}")
            return {}
    if len(frames) < 2:
        return {}
    t = ex.get("text") or ex.get("caption")
    return {"video": frames, "text": t} if isinstance(t, str) and t.strip() else {"video": frames}


# name -> (loader, extractor). Anything that fails to open is skipped with a log line, so a stale
# dataset id degrades the mixture instead of killing the run.
#
# Every id below was opened against the live Hub and its first record decoded (July..Oct 2026
# Hub state). Pruned because they no longer stream at all: lambdalabs/pokemon-blip-captions
# (gated), nlphuji/flickr30k, google/docci, facebook/textvqa, poloclub/diffusiondb, allenai/peS2o
# (all script-based), bigcode/the-stack-smol (gated), huggingface/food101 (renamed ethz/food101),
# HuggingFaceTB/smollm-corpus/python-edu (ships metadata only, no text column).
SOURCES = {
    # --- general + educational text
    "text_fineweb":     (_stream("HuggingFaceFW/fineweb-edu", "sample-10BT"), _txt),
    "text_cosmopedia":  (_stream("HuggingFaceTB/smollm-corpus", "cosmopedia-v2"), _txt),
    "text_wikipedia":   (_stream("wikimedia/wikipedia", "20231101.en"), _txt),
    # --- math
    "text_math_web":    (_stream("open-web-math/open-web-math"), _txt),
    "text_math_fine":   (_stream("HuggingFaceTB/finemath", "finemath-3plus"), _txt),
    "text_math_numina": (_stream("AI-MO/NuminaMath-CoT"), _two("problem", "solution")),
    # --- physics / science papers
    "text_arxiv":       (_stream("ccdv/arxiv-summarization"), _two("article", "abstract")),
    # --- code
    "text_code":        (_stream("codeparrot/codeparrot-clean"), _two("content")),
    "text_code_inst":   (_stream("ise-uiuc/Magicoder-OSS-Instruct-75K"),
                         _two("problem", "solution")),
    # --- physics / chemistry / math instruction corpora
    # `message_1` / `message_2` are the question and the worked answer: `_txt` finds no `text` key
    # on these and would silently yield nothing, which is why `_two` exists.
    "text_physics":     (_stream("camel-ai/physics"), _two("message_1", "message_2")),
    "text_chemistry":   (_stream("camel-ai/chemistry"), _two("message_1", "message_2")),
    "text_math_inst":   (_stream("camel-ai/math"), _two("message_1", "message_2")),
    # --- vision
    "image_docvqa":     (_s("HuggingFaceM4/DocumentVQA"), _img_text()),
    "image_imagenet":   (_s("evanarlian/imagenet_1k_resized_256"), _img_text()),
    "image_coco":       (_s("detection-datasets/coco"), _img_text()),
    "image_wikiart":    (_s("huggan/wikiart"), _img_text(cap_keys=("artist", "style", "genre"))),
    "image_flowers":    (_s("nelorth/oxford-flowers"), _img_text()),
    "image_food":       (_s("ethz/food101"), _img_text()),
    # --- math / science figures. `_img_text` is wrong for both: LaTeX_OCR's `text` is a LaTeX
    # transcription with no spaces, and ScienceQA's answer is an index into `choices`.
    "image_latex":      (_s("linxy/LaTeX_OCR"), _img_text(img_keys=("image",), cap_keys=("text",))),
    "image_scienceqa":  (_s("derek-thomas/ScienceQA"), _scienceqa),
    # --- audio (always decode=False + soundfile)
    "audio_libri":      (_raw("openslr/librispeech_asr", "clean", "train.100",
                              column="audio"), _audio_text),
    "audio_libritts":   (_raw("mythicinfinity/libritts_r", "clean", "train.clean.100",
                              column="audio"), _audio_text),
    "audio_fleurs":     (_raw("google/fleurs", "en_us", column="audio"), _audio_text),
    # "clean", not "microset": the 30x audio budget needs ~31k hours, microset is a demo slice.
    "audio_peoples":    (_raw("MLCommons/peoples_speech", "clean", column="audio"),
                         _audio_text),
    # --- video (data path only; kinetics-mini streams, ucf101-subset is 2 clips)
    "video_ucf":        (_raw(["nateraw/kinetics-mini", "sayakpaul/ucf101-subset"],
                              column="video"), _video_frames),
}
GROUPS = {k: "text" for k in ("text_fineweb", "text_cosmopedia", "text_wikipedia",
                              "text_math_web", "text_math_fine", "text_math_numina",
                              "text_arxiv", "text_code", "text_code_inst",
                              "text_physics", "text_chemistry", "text_math_inst")}
GROUPS.update({k: "image" for k in ("image_docvqa", "image_imagenet", "image_coco",
                                    "image_wikiart", "image_flowers", "image_food",
                                    "image_latex", "image_scienceqa")})
GROUPS.update({k: "audio" for k in ("audio_libri", "audio_libritts", "audio_fleurs",
                                    "audio_peoples")})
GROUPS["video_ucf"] = "video"


class Source:
    """One stream with retry and a reopen that fast-forwards past what was already read."""

    def __init__(self, name, loader, fmt, shard=(0, 1)):
        self.name, self.loader, self.fmt = name, loader, fmt
        self.n, self.fails, self.dead, self.bad = 0, 0, False, 0
        self.shard, self.sharded = shard, False
        self.it = self._open()

    def _open(self):
        """Split the stream at the *source* level, not after downloading it.

        `ds.shard(N, i)` makes the backend read only its share of the parquet files. Filtering on
        `index % N` instead - which is what the first version did - still downloads every row in
        every worker, so N workers cost N times the bandwidth and run N times slower. The fallback
        filter stays for backends that refuse to shard, but it must not be applied on top of a real
        shard or each worker would see 1/N^2 of the data.
        """
        ds = self.loader()
        if self.shard[1] > 1:
            try:
                ds = ds.shard(num_shards=self.shard[1], index=self.shard[0], contiguous=False)
                self.sharded = True
            except Exception as e:                             # noqa: BLE001
                # Not a defect: `batch()` row-strides unsharded sources by `self.n % N`, so the rows
                # stay correct. The cost is bandwidth - every worker reads the whole stream - which is
                # exactly what the docstring above warns about. Striding again here would compound it
                # into 1/N^2 and silently starve the source.
                log(f"[source] {self.name}: backend shard unavailable "
                    f"({type(e).__name__}: {str(e)[:60]}) - full stream per worker, "
                    f"row stride in batch() (N x bandwidth, correct rows)")
        return iter(ds)

    def reopen(self, err):
        while self.fails < 5:
            self.fails += 1
            log(f"[source] {self.name}: {type(err).__name__}: {str(err)[:90]} -> reopen {self.fails}/5")
            time.sleep(min(30, 4 * self.fails))
            try:
                it = self._open()
                for _ in range(self.n):                        # fast-forward, no encoding
                    next(it)
                self.it = it
                return True
            except Exception as e:                             # noqa: BLE001
                err = e
        return False

    def batch(self, k):
        out = []
        while len(out) < k and not self.dead:
            try:
                ex = next(self.it)
            except StopIteration:
                self.dead = True
                break
            except Exception as e:                             # noqa: BLE001
                if not self.reopen(e):
                    self.dead = True
                    break
                continue
            self.n += 1                                       # raw index: what reopen() replays
            if not self.sharded and self.shard[1] > 1 and (self.n - 1) % self.shard[1] != self.shard[0]:
                continue
            try:
                d = self.fmt(ex)
            except Exception:                                  # noqa: BLE001
                self.bad += 1
                continue
            if d:
                out.append(d)
        return out


# ------------------------------------------------------------------ encoding
def make_codecs(sched, V, prefer_real=True):
    return {m: build_codec(m, sched.K, sched.R, V, sched.base_len[m], prefer_real)
            for m in MODALITIES}


def encode_sample(codecs, sched, parts):
    """dict modality -> raw object  =>  dict modality -> int32[flat_len], plus presence vector."""
    codes, present = {}, {}
    for m in MODALITIES:
        obj = parts.get(m)
        ok = obj is not None
        if ok:
            try:
                lv = codecs[m].encode(obj)
            except Exception as e:                             # noqa: BLE001
                log(f"[encode] {m}: {type(e).__name__}: {str(e)[:80]}")
                ok = False
        if ok:
            flat = np.concatenate([np.asarray(x, dtype=np.int32).ravel() for x in lv])
            if flat.size != sched.flat_len(m):                 # codec disagreeing with schedule
                log(f"[encode] {m}: got {flat.size}, schedule wants {sched.flat_len(m)}")
                ok = False
            else:
                codes[m] = np.clip(flat, 1, codecs[m].V)           # 0 is reserved for NULL
        if not ok:
            codes[m] = np.full(sched.flat_len(m), sched.null_code, dtype=np.int32)
        present[m] = bool(ok)
    return codes, present


def stack_samples(rows, sched, mods=MODALITIES):
    out = {}
    for m in mods:
        out[m] = np.stack([r["codes"][m] for r in rows]).astype(np.int32)
        out[f"{m}_present"] = np.array([r["present"][m] for r in rows], dtype=bool)
    return out


def schedule_meta(sched, codecs, V):
    return dict(K=sched.K, R=sched.R, null_code=sched.null_code,
                base_len=dict(sched.base_len), modalities=list(mods_default()),
                level_lens={m: list(sched.level_lens(m)) for m in MODALITIES},
                offsets={m: list(sched.offsets(m)) for m in MODALITIES},
                vocab={m: codecs[m].V for m in MODALITIES},
                codecs={m: codecs[m].info() for m in MODALITIES},
                version=1)


def mods_default():
    return MODALITIES


def atomic_npz(path, arrays, meta):
    tmp = path + ".tmp.npz"
    np.savez(tmp, meta=np.array(json.dumps(meta)), **arrays)
    os.replace(tmp, path)


# ------------------------------------------------------------------ main
def run(cfg, selfcheck=False):
    out = cfg["out_dir"]
    os.makedirs(out, exist_ok=True)
    sched = OmniSchedule(base_len=cfg["base_len"], K=cfg["K"], R=cfg["R"],
                         null_code=cfg.get("null_code", 0))
    V = cfg["codebook_size"]
    codecs = make_codecs(sched, V, cfg.get("prefer_real", True))
    meta = schedule_meta(sched, codecs, V)
    log(sched.describe())
    log("codecs:", {m: codecs[m].name for m in MODALITIES})

    if selfcheck:
        # Every modality, with synthetic objects only: no network, no dataset ids. What is being
        # proved here is the ladder contract - len(level_k) == schedule.level_len(m, k) - and
        # that no code ever collides with the NULL slot.
        import numpy as _np
        from PIL import Image as _Image

        def _img(seed):
            g = _np.random.default_rng(seed)
            a = (g.random((32, 32, 3)) * 255).astype("uint8")
            return _Image.fromarray(a)

        parts = [
            {"text": "a blood pressure of 140 over 90 is stage two hypertension",
             "image": _img(0),
             "video": [_img(i) for i in range(8)],
             "audio": (_np.sin(_np.linspace(0, 400, 16000)).astype("float32"), 16000)},
            {"text": "short", "image": _img(1)},
        ]
        rows = [dict(zip(("codes", "present"), encode_sample(codecs, sched, p))) for p in parts]
        for m in MODALITIES:
            assert rows[0]["present"][m], f"{m}: synthetic sample failed to encode"
            assert len(rows[0]["codes"][m]) == sched.flat_len(m), \
                f"{m}: flat {len(rows[0]['codes'][m])} != schedule {sched.flat_len(m)}"
            c = rows[0]["codes"][m]
            assert int(c.min()) >= 1, f"{m}: a code hit the NULL slot"
            assert int(c.max()) <= codecs[m].V, f"{m}: code {int(c.max())} > codebook {codecs[m].V}"
            off = 0
            for k in range(sched.K):
                l = sched.level_len(m, k)
                assert len(rows[0]["codes"][m][off:off + l]) == l, f"{m} level {k} length"
                off += l
        assert not rows[1]["present"]["audio"], "audio should be absent for a text-only sample"

        arr = stack_samples(rows, sched)
        atomic_npz(os.path.join(out, "selfcheck.npz"), arr, meta)
        log("selfcheck wrote", os.path.join(out, "selfcheck.npz"),
            {k: v.shape for k, v in arr.items()})
        log("selfcheck: ladder contract holds for", {m: codecs[m].name for m in MODALITIES})
        return 0

    budget = cfg["sample_budget"]
    shard_n = cfg["shard_samples"]
    max_ahead = cfg["max_ahead_shards"]
    worker, nworkers = cfg.get("worker", (0, 1))
    assert 0 <= worker < nworkers, f"worker {worker} of {nworkers}"
    # Only worker 0 writes the held-out blocks. Every worker reads a disjoint slice, so the rows it
    # holds out are invisible to the others - one copy of each eval file, no duplicates.
    holdout = {g: (n if worker == 0 else 0) for g, n in cfg["holdout"].items()}
    skip = cfg.get("skip_samples", 0)
    mix = {k: float(v) for k, v in cfg["mix"].items() if k in SOURCES}
    only = cfg.get("only_sources")
    if only:
        mix = {k: v for k, v in mix.items() if k in only}
    assert mix, "no usable sources in the mix"

    live, skipped = {}, {}
    for name in mix:
        for attempt in range(4):
            try:
                live[name] = Source(name, SOURCES[name][0], SOURCES[name][1],
                                    shard=(worker, nworkers))
                break
            except Exception as e:                             # noqa: BLE001
                err = f"{type(e).__name__}: {str(e)[:90]}"
                if attempt == 3:
                    skipped[name] = err
                    log(f"[source] {name} SKIPPED after 4 attempts: {err}")
                else:
                    time.sleep(3 * (attempt + 1))              # 429 from a burst of stream opens
    if not live:
        log("FATAL: no source could be opened")
        return 2
    if skipped:
        log(f"WARNING: {len(skipped)} source(s) missing - the mix is NOT the configured one:",
            list(skipped))
    tot = sum(mix[n] for n in live)
    frac = {n: mix[n] / tot for n in live}
    log("sources:", {n: round(frac[n], 3) for n in live})
    needed = {GROUPS[n] for n in live}        # only groups with a live source gate EVAL_READY

    # Per-worker seed. One shared seed made every worker walk the sources in the same order with
    # the same weights, so N workers piled onto the same heavy source at the same instant - the
    # thundering herd that makes a 4-worker run hungrier and no faster than a 1-worker run.
    rng = random.Random(int(cfg["seed"]) + 7919 * worker)
    used = {n: 0 for n in live}
    dropped = {n: 0 for n in live}
    held = {g: [] for g in holdout}
    shuf, cur, shard_idx, written = [], [], 0, 0
    seen = 0
    t0 = time.time()

    def next_shard_path():
        nonlocal shard_idx
        tag = f"_w{worker}" if nworkers > 1 else ""
        p = os.path.join(out, f"shard{tag}_{shard_idx:06d}.npz")
        shard_idx += 1
        return p

    def flush(final=False):
        nonlocal cur, written
        while len(cur) >= shard_n or (final and cur):
            rows, cur = cur[:shard_n], cur[shard_n:]
            if not rows:
                break
            arr = stack_samples(rows, sched)
            if seen_so_far[0] > skip:                          # never rewrite what a resume already trained on
                mine = f"shard_w{worker}_" if nworkers > 1 else "shard_"
                while (sum(1 for f in os.listdir(out)
                           if f.startswith(mine) and f.endswith(".npz")) >= max_ahead):
                    if os.path.exists(os.path.join(out, "STOP")):
                        log("STOP seen, exiting")
                        sys.exit(0)
                    time.sleep(1)
                atomic_npz(next_shard_path(), arr, meta)
                written += 1

    seen_so_far = [0]

    def emit(row, group):
        """Hold out the first N samples per group, then shuffle-buffer the rest into shards."""
        if len(held[group]) < holdout.get(group, 0):
            held[group].append(row)
            if len(held[group]) == holdout[group]:
                atomic_npz(os.path.join(out, f"eval_{group}.npz"),
                           stack_samples(held[group], sched), meta)
                log(f"[eval] {group}: {holdout[group]} held out")
                if all(len(held.get(g, [])) >= holdout.get(g, 0) for g in needed):
                    if nworkers == 1:
                        open(os.path.join(out, "EVAL_READY"), "w").write("1")
            return
        seen_so_far[0] += 1
        if len(shuf) < cfg["shuffle_buf"]:
            shuf.append(row)
            return
        j = rng.randrange(len(shuf))
        out_row, shuf[j] = shuf[j], row
        cur.append(out_row)

    active = list(live)
    st = dict(pull=0.0, enc=0.0, write=0.0)        # wall clock per stage, flushed into progress

    while active and seen_so_far[0] < budget:
        name = rng.choices(active, weights=[frac[n] for n in active])[0]
        group = GROUPS[name]
        _t = time.perf_counter()
        batch = live[name].batch(cfg["batch_docs"])
        st["pull"] += time.perf_counter() - _t
        if not batch:
            active.remove(name)
            log(f"[source] {name} exhausted/dead")
            continue
        for parts in batch:
            if seen_so_far[0] >= budget:
                break
            used[name] += 1
            _t = time.perf_counter()
            codes, present = encode_sample(codecs, sched, parts)
            st["enc"] += time.perf_counter() - _t
            if not any(present.values()):
                dropped[name] += 1
                continue
            emit({"codes": codes, "present": present}, group)
        _t = time.perf_counter()
        flush()
        st["write"] += time.perf_counter() - _t
        el = max(1e-9, time.time() - t0)
        tot = max(1e-9, sum(st.values()))
        json.dump({"samples": seen_so_far[0], "budget": budget, "shards": written,
                   "per_source": used,
                   # rows/s per source: a source that is alive but starving shows up here without
                   # anyone having to diff two progress files by hand.
                   "per_source_s": {n: round(v / el, 3) for n, v in used.items() if v},
                   "dropped": {n: v for n, v in dropped.items() if v},
                   "eval_held": {g: len(v) for g, v in held.items()},
                   "elapsed_min": round(el / 60, 2),
                   "samples_per_s": round(seen_so_far[0] / el, 3),
                   # Where the wall clock went: pull = blocked on the Hub, enc = decode + codec,
                   # write = npz. A produce run that looks stuck is almost always pull.
                   "stage_pct": {k: round(100 * v / tot, 1) for k, v in st.items()},
                   "stage_s": {k: round(v, 1) for k, v in st.items()}},
                  open(os.path.join(out, "progress.json" if nworkers == 1
                                    else f"progress_w{worker}.json"), "w"))
        if seen_so_far[0] % 25 == 0:
            raw = {n: live[n].n for n in live}
            log(f"progress {seen_so_far[0]}/{budget} samples | {written} shards | "
                f"{seen_so_far[0] / el:.2f} samples/s | "
                f"pull {100 * st['pull'] / tot:.0f}% enc {100 * st['enc'] / tot:.0f}% "
                f"write {100 * st['write'] / tot:.0f}% | dropped {sum(dropped.values())} | "
                f"pulled {sum(raw.values())} rows | sharded="
                f"{sum(1 for n in live if live[n].sharded)}/{len(live)}")

    for row in shuf:                                           # flush the shuffle buffer
        cur.append(row)
    flush(final=True)
    for g, rows in held.items():                               # under-filled groups still get a file
        if rows and len(rows) < holdout.get(g, 0):
            atomic_npz(os.path.join(out, f"eval_{g}.npz"), stack_samples(rows, sched), meta)
    stats = dict(samples=seen_so_far[0], shards=written, per_source=used, dropped=dropped,
                 budget=budget, skip_samples=skip, schedule=meta, worker=[worker, nworkers],
                 skipped_sources=skipped,
                 elapsed_min=round((time.time() - t0) / 60, 2))
    if nworkers == 1:                                          # single process: owns the sentinels
        open(os.path.join(out, "EVAL_READY"), "w").write("1")
        json.dump(stats, open(os.path.join(out, "stats.json"), "w"), indent=2)
        open(os.path.join(out, "DONE"), "w").write("1")
    else:                                                       # coordinator aggregates and decides
        json.dump(stats, open(os.path.join(out, f"stats_w{worker}.json"), "w"), indent=2)
    log("DONE", {k: stats[k] for k in ("samples", "shards", "elapsed_min")})
    return 0


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("cfg")
    ap.add_argument("--selfcheck", action="store_true")
    ap.add_argument("--worker", default=None, help="i/N - this process's slice of every stream")
    a = ap.parse_args()
    cfg = json.load(open(a.cfg))
    if a.worker:
        i, n = a.worker.split("/")
        cfg["worker"] = (int(i), int(n))
    try:
        return run(cfg, selfcheck=a.selfcheck)
    except SystemExit:
        raise
    except Exception:                                          # noqa: BLE001
        traceback.print_exc()
        return 1


if __name__ == "__main__":
    sys.exit(main())


In [ ]:
%%writefile .ncm_stage/omni/livestream.py
"""Rows straight off the Hub, with no shard corpus in between.

`omni.producer` writes shards because separate processes - one per arm - need to read identical
data. A duplex run does not: both arms consume the same batch, in the same process, in the same
order, so the stream *is* the control variable and 337 MB of npz files never have to exist.

Pulls are concurrent. A single-threaded loop over 20 Hub streams stalls completely whenever one
source is slow - an audio decode or a video download blocks `Source.batch` and nothing else gets
a turn. Text alone measures 4.9 docs/s, so that stall, not bandwidth, is what makes a naive
single-process stream look dead. The producer hid it by running 8 processes.

The trade is reproducibility. A shard corpus can be re-read after a crash; a stream is one-pass.
That is right for a kill test (short, one comparison) and wrong for a long pretraining run.
"""
import os
import queue
import random
import threading
import time

from collections import deque

from omni.producer import (GROUPS, SOURCES, Source, atomic_npz, encode_sample, log as plog,
                           make_codecs, schedule_meta, stack_samples)
from omni.schedule import OmniSchedule


class _Weighted:
    """Deficit round-robin over sources, so the yielded mix follows `frac` and not the pumps' speed.

    The queue is one FIFO of whatever finished first, and text costs far less to encode than audio
    (no decode, no mel pyramid), so a plain `q.get()` hands back a text-dominated stream: measured on
    the Hub, the first ~600 rows of a run contained no image at all and the first ~4250 no audio.
    `frac` was computed and then never used, so the achieved mix was a function of throughput - a
    458-step run (3664 samples) never reached the first audio row.

    A source may run ahead; its surplus is buffered up to `cap` rows and the oldest dropped beyond
    that (there is always more where it came from, and dropping text is cheap). The source with the
    largest deficit against its target share is served next, so the long-run mix converges on `frac`.
    """
    def __init__(self, frac, cap=64):
        self.frac = {k: float(v) for k, v in frac.items()}
        self.cap = int(cap)
        self.buf = {k: deque() for k in self.frac}
        self.emitted = dict.fromkeys(self.frac, 0)
        self.dropped = dict.fromkeys(self.frac, 0)
        self.cool = set()                    # owed a slot, but it made us wait: stop blocking on it
        self.total = 0

    def push(self, name, item):
        self.cool.discard(name)               # it delivered: it may be waited on again
        buf = self.buf.get(name)
        if buf is None:                       # a source outside the mix: accept it, weight it 0
            self.buf[name] = buf = deque()
            self.frac[name] = 0.0
            self.emitted[name] = 0
            self.dropped[name] = 0
        buf.append(item)
        if len(buf) > self.cap:
            buf.popleft()
            self.dropped[name] += 1

    def next(self):
        """The source most owed a slot right now - buffered or not, and possibly not yet delivered.

        Choosing on deficit alone is the whole point. Choosing among *buffered* sources is FIFO in
        disguise: the fast source is always the one with something in hand, so it always wins.
        """
        best, best_def = None, None
        n = max(1, self.total)
        for name in self.frac:
            if self.frac[name] <= 0.0 or name in self.cool:   # retired, unmixed, or just timed out
                continue
            d = self.frac[name] * n - self.emitted[name]
            if best_def is None or d > best_def:
                best, best_def = name, d
        return best

    def best_ready(self):
        """The most owed source that already has a row buffered - the patience fallback."""
        best, best_def = None, None
        n = max(1, self.total)
        for name, buf in self.buf.items():
            if not buf or self.frac.get(name, 0.0) <= 0.0:
                continue
            d = self.frac[name] * n - self.emitted[name]
            if best_def is None or d > best_def:
                best, best_def = name, d
        return best

    def ready(self, name):
        return bool(self.buf.get(name))

    def take(self, name):
        self.emitted[name] += 1
        self.total += 1
        return self.buf[name].popleft()

    def retire(self, name):
        """A pump is gone: stop waiting on it, but still serve whatever it already delivered."""
        if name in self.frac:
            self.frac[name] = 0.0
            self.emitted[name] = 0
        self.cool.discard(name)

    def cool_down(self, name):
        """It made us wait past `patience`; hand its slot on until it actually delivers again.

        Without this a source that is merely slow costs one row per patience window forever, which
        stalls the stream far worse than the mixing bug did.
        """
        self.cool.add(name)

    def shares(self):
        n = max(1, self.total)
        return {k: round(v / n, 3) for k, v in sorted(self.emitted.items()) if v}


def _sched_of(pcfg):
    """The schedule lives under `sched` in a train cfg and flat in a produce cfg; accept both."""
    s = pcfg.get("sched") or {}

    def pick(k, d=None):
        for src in (pcfg, s):
            v = src.get(k)
            if v is not None:
                return v
        return d

    return OmniSchedule(base_len=pick("base_len"), K=pick("K"), R=pick("R"),
                        null_code=pick("null_code", 0))


def _pump(src, out, stop, batch_docs):
    """One thread per source: pull, hand over, repeat. Daemon so a hung source cannot block exit."""
    while not stop.is_set():
        got = src.batch(batch_docs)
        if not got:
            out.put((src.name, None))
            return
        for parts in got:
            if stop.is_set():
                return
            out.put((src.name, parts))


def iter_rows(pcfg, holdout=None, eval_dir=None, verbose=True, max_queue=512):
    """Yield encoded samples, interleaved by the configured source weights.

    If `holdout` is given ({group: n}), the first n rows of each group are diverted, encoded as
    eval_{group}.npz under `eval_dir`, and never yielded - the held-out set comes off the same
    stream the arms read, so it cannot drift from training.
    """
    sched = _sched_of(pcfg)
    V = pcfg["codebook_size"]
    codecs = make_codecs(sched, V, pcfg.get("prefer_real", True))
    meta = schedule_meta(sched, codecs, V)
    rng = random.Random(pcfg.get("seed", 3407))

    mix = {k: float(v) for k, v in pcfg["mix"].items() if float(v) > 0}
    total_w = sum(mix.values())
    frac = {k: v / total_w for k, v in mix.items()}

    holdout = dict(holdout or {})
    held = {g: [] for g in set(GROUPS.values())}
    if holdout and eval_dir:
        os.makedirs(eval_dir, exist_ok=True)

    q, stop = queue.Queue(maxsize=max_queue), threading.Event()
    live, threads = {}, []
    for name in mix:
        src = Source(name, *SOURCES[name][:2])
        live[name] = src
        t = threading.Thread(target=_pump, args=(src, q, stop, int(pcfg.get("batch_docs", 1))),
                             daemon=True)
        t.start()
        threads.append(t)
        if verbose:
            plog(f"[live] {name} pumping (sharded={src.sharded})")

    # Weights the draw order, not the arrival order. `patience` bounds how long a source that is owed
    # a slot may hold everything up: past it the most-owed *available* source is served instead, so a
    # slow pump throttles the stream while a dead one cannot hang it.
    wq = _Weighted(frac)
    patience = float(pcfg.get("mix_patience", 3.0))
    pulled = 0
    try:
        while True:
            # The clock, not a counter of empty reads. Text almost never lets the queue go empty, so
            # anything that only counted timeouts would wait forever on the first source that is owed
            # a slot but has not delivered yet - which is exactly the startup stall (19 pumps up, no
            # rows out, for 150 s).
            t_draw = time.monotonic()
            while True:
                owed = wq.next()
                if owed is None:
                    return                      # every source retired: nothing left to wait for
                if wq.ready(owed):
                    name = owed
                    break
                fallback = wq.best_ready()
                if fallback is not None and time.monotonic() - t_draw >= patience:
                    wq.cool_down(owed)          # it kept us waiting: let the next draw skip it
                    name = fallback
                    break
                if not any(t.is_alive() for t in threads) and q.empty():
                    if fallback is not None:
                        name = fallback
                        break
                    return                      # every pump dead: exhausted, or it raised
                try:
                    src, parts = q.get(timeout=0.25)
                except queue.Empty:
                    continue
                if parts is None:
                    live.pop(src, None)
                    wq.retire(src)
                    plog(f"[live] {src} exhausted ({len(live)} left)")
                    continue
                wq.push(src, parts)
            parts = wq.take(name)
            pulled += 1
            codes, present = encode_sample(codecs, sched, parts)
            if not any(present.values()):
                continue
            row = {"codes": codes, "present": present}
            group = GROUPS[name]
            want = holdout.get(group, 0)
            if want and len(held[group]) < want:
                held[group].append(row)
                if len(held[group]) == want and eval_dir:
                    p = os.path.join(eval_dir, f"eval_{group}.npz")
                    atomic_npz(p, stack_samples(held[group], sched), meta)
                    plog(f"[eval] {group}: {want} held out -> {p}")
                continue
            yield row
            if verbose and pulled % 500 == 0:
                plog(f"[live] {pulled} rows | sources {len(live)} | mix {wq.shares()} "
                     f"| dropped {sum(wq.dropped.values())} "
                     f"| held { {g: len(v) for g, v in held.items()} }")
    finally:
        stop.set()


In [ ]:
%%writefile .ncm_stage/omni/data.py
"""Consumer side: shard directory -> packed multimodal batches.

Reads what producer.py wrote, in order, deleting each shard once consumed (the producer keeps
MAX_AHEAD_SHARDS ahead, so the disk footprint stays bounded). If the producer died, the stream
raises with the tail of its log attached instead of hanging forever.
"""
import glob
import json
import os
import time

import numpy as np
import torch
from torch.utils.data import Dataset, IterableDataset

from omni.schedule import MODALITIES, OmniSchedule


def read_npz(path):
    z = np.load(path, allow_pickle=False)
    raw = z["meta"]
    meta = json.loads(raw.item() if getattr(raw, "shape", ()) == () else str(raw))
    return z, meta


def schedule_from_meta(meta: dict) -> OmniSchedule:
    return OmniSchedule(base_len=meta["base_len"], K=meta["K"], R=meta["R"],
                        null_code=meta.get("null_code", 0))


def rows_from_npz(z, meta, mods=MODALITIES):
    n = len(z[f"{mods[0]}_present"]) if f"{mods[0]}_present" in z else 0
    for i in range(n):
        yield {"codes": {m: z[m][i] for m in mods if m in z},
               "present": {m: bool(z[f"{m}_present"][i]) for m in mods if f"{m}_present" in z}}


class ShardStream(IterableDataset):
    """Infinite-ish sampler over the producer's rolling shards."""

    def __init__(self, shard_dir, consume=False, start_sample=0, poll=0.25,
                 producer_log=None, timeout_s=None):
        # consume=False by default: shards are the only copy of the data, and the two arms of the
        # experiment must read the same rows. Deleting them after arm 1 leaves arm 2 with a
        # truncated stream - which silently shortens its step budget instead of failing loudly.
        self.dir, self.consume, self.start = shard_dir, consume, start_sample
        self.poll, self.producer_log, self.timeout_s = poll, producer_log, timeout_s

    def _shards(self):
        return sorted(glob.glob(os.path.join(self.dir, "shard_*.npz")))

    def _tail(self, n=20):
        if not self.producer_log or not os.path.exists(self.producer_log):
            return "(producer log unavailable)"
        with open(self.producer_log) as f:
            return "".join(f.readlines()[-n:])

    def __iter__(self):
        seen, t0, i, last_say = 0, time.time(), 0, 0.0
        while True:
            files = self._shards()
            if files:
                # consume=True: the list shrinks as we delete, so index 0 is always next.
                # consume=False: nothing is deleted, so walking index 0 forever re-reads the
                # same shard - which is a silent single-shard corpus, not a data bug you can see
                # in the loss. Walk the sorted list and wrap for the next epoch.
                path = files[0] if self.consume else files[i % len(files)]
                try:
                    z, meta = read_npz(path)
                except Exception:                        # half-written from the producer
                    time.sleep(self.poll)
                    continue
                for row in rows_from_npz(z, meta):
                    if seen < self.start:                # resume: skip without yielding
                        seen += 1
                        continue
                    seen += 1
                    yield row
                if self.consume:
                    try:
                        os.remove(path)
                    except OSError:
                        pass
                else:
                    i += 1
                continue
            if os.path.exists(os.path.join(self.dir, "DONE")):
                return                                   # producer finished and nothing is left
            if self.timeout_s and time.time() - t0 > self.timeout_s:
                raise TimeoutError(f"no shards after {self.timeout_s}s\n{self._tail()}")
            # Silence here is indistinguishable from a hang: the loop can legitimately wait the
            # whole stream_timeout (1800s) for a producer that is still filling its shuffle
            # buffer. Say so every 15s, with the producer's last log line if we have one.
            # Nobody is producing: an empty dir with no producer log to tail means this is not a
            # slow start, it is a mis-sequenced run. 20s is enough to tell the difference, and far
            # better than blocking out the full stream_timeout.
            busy = (os.path.isdir(self.dir) and bool(os.listdir(self.dir))) or (
                self.producer_log and os.path.exists(self.producer_log))
            if not files and time.time() - t0 > 20 and not busy:
                raise RuntimeError(
                    f"no shards in {self.dir} and no producer is writing there "
                    f"({self.producer_log or 'no producer_log'} missing).\n"
                    f"run_omni.py produce must finish first:  run_omni.py produce --budget 2000 "
                    f"--shuffle-buf 128 --workers 4 --foreground")
            if time.time() - last_say > 15:
                last_say = time.time()
                have = (f"{len(os.listdir(self.dir))} files" if os.path.isdir(self.dir)
                        else f"{self.dir} does not exist")
                tail = self._tail().strip().splitlines()[-1:] if self.producer_log else []
                print(f"  waiting for shards: {have}, {time.time() - t0:.0f}s of "
                      f"{self.timeout_s or '-'}s{' | ' + tail[0][:120] if tail else ''}",
                      flush=True)
            time.sleep(self.poll)


class RowsSet(Dataset):
    """Held-out rows held in memory.

    Same interface as EvalSet but with no npz behind it: the direct feed peeks N rows before
    training starts and validates on those, so the val curve is stable within a run without
    anything touching the disk.
    """

    def __init__(self, rows):
        self.rows = list(rows)

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, i):
        return self.rows[i]


class EvalSet(Dataset):
    def __init__(self, npz_path):
        z, self.meta = read_npz(npz_path)
        self.rows = list(rows_from_npz(z, self.meta))
        self.sched = schedule_from_meta(self.meta)

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, i):
        return self.rows[i]


def collate(rows, sched, mods=None):
    """List of sample dicts -> {"codes": {m: int64 [B, flat_len(m)]}, "present": {m: bool [B]}}.

    Widths always come from the schedule, never from the rows: a shard written by a codec that
    disagreed with the schedule must fail loudly here rather than produce a ragged batch.
    """
    mods = tuple(mods or sched.modalities)
    B = len(rows)
    codes, present = {}, {}
    for m in mods:
        flat = sched.flat_len(m)
        arr = np.full((B, flat), sched.null_code, dtype=np.int64)
        pres = np.zeros(B, dtype=bool)
        for i, r in enumerate(rows):
            c = r["codes"].get(m)
            if c is not None and len(c) == flat:
                arr[i] = c
                pres[i] = bool(r["present"].get(m, False))
        codes[m] = torch.from_numpy(arr)
        present[m] = torch.from_numpy(pres)
    return {"codes": codes, "present": present}


def to_device(batch, device):
    return {"codes": {m: v.to(device, non_blocking=True) for m, v in batch["codes"].items()},
            "present": {m: v.to(device, non_blocking=True) for m, v in batch["present"].items()}}


def synthetic_batch(sched: OmniSchedule, vocab, B=2, seed=0, device="cpu", present_all=True):
    """A batch that needs no producer: for selfcheck and for the overfit-one-batch smoke test."""
    g = torch.Generator().manual_seed(seed)
    codes, present = {}, {}
    for m in sched.modalities:
        v = vocab[m]
        top = max(v) if isinstance(v, (list, tuple, dict)) else int(v)   # per-level vocab
        flat = torch.randint(1, top, (B, sched.flat_len(m)), generator=g)
        codes[m] = flat.to(device)
        present[m] = torch.ones(B, dtype=torch.bool, device=device)
    if not present_all:
        present[sched.modalities[-1]] = torch.zeros(B, dtype=torch.bool, device=device)
    return {"codes": codes, "present": present}


In [ ]:
%%writefile .ncm_stage/omni/model.py
"""Omni generator: one transformer over level-major packs of every modality.

Pack layout (see schedule.py):   [L0: text|image|audio|video][L1: ...] ... [LK-1: ...]

  block-diagonal mask  -> level k sees its own level across ALL modalities (the omni claim),
                          and nothing finer. One forward predicts every code of every modality
                          at that altitude.
  mask_mode="ar"       -> same pack, block-causal + causal within the level: a matched-params
                          autoregressive baseline over the raster order, one token at a time.

Conditioning follows the paper in spirit, not in letter: NCM conditions level k on the codec's
reconstruction z_hat(k-1); our codecs are discrete-only, so we condition on the mean embedding
of the level-(k-1) codes actually fed in. Named deviation, `cond_mode="gt_embed"`.
"""
from collections import OrderedDict
from typing import Dict, List, Optional

import torch
import torch.nn as nn
import torch.nn.functional as F

import os
import sys

try:
    from core import Block, HAS_FLEX, RMSNorm, precompute_rope_cos_sin
except ModuleNotFoundError:                                # core.py lives in the sibling package
    for _cand in (os.path.dirname(os.path.dirname(os.path.abspath(__file__))),
                  os.path.expanduser("~/projects/ncm_fused"),
                  os.environ.get("NCM_CORE_DIR", "")):
        if _cand and os.path.exists(os.path.join(_cand, "core.py")):
            sys.path.insert(0, _cand)
            break
    from core import Block, HAS_FLEX, RMSNorm, precompute_rope_cos_sin
from omni.schedule import MODALITIES, OmniSchedule

try:
    from torch.nn.attention.flex_attention import create_block_mask
except Exception:                                          # pragma: no cover
    create_block_mask = None


try:                                # per-level widths differ (24 vs 56 keys), so flex recompiles
    torch._dynamo.config.cache_size_limit = 32   # once per level and hits the default limit of 8
except Exception:                   # -> silent fallback to the unfused eager kernel
    pass


def normalize_vocab(vocab, modalities, K):
    """-> {m: (V_0, ..., V_{K-1})}.

    Accepts, in order of age:
      * {m: int}                       one codebook per modality, every level (the old layout)
      * {m: [int]*K} or {m: {k: int}}  a width per level (heterogeneous head: the finest text
                                       level holds BPE ids, the coarse levels hold 256-way codes)
      * int                            the same for every modality

    Keeping every level the same width is still the default and still the only layout that loads
    the pre-existing checkpoints: a heterogeneous model cannot read a `heads.text.weight`.
    """
    out = {}
    for m in modalities:
        v = vocab.get(m, 256) if isinstance(vocab, dict) else vocab
        if isinstance(v, dict):
            seq = [int(v[k]) for k in range(K)]
        elif isinstance(v, (list, tuple)):
            seq = [int(x) for x in v]
            seq = (seq + [seq[-1]] * K)[:K]
        else:
            seq = [int(v)] * K
        assert all(x >= 1 for x in seq), (m, seq)
        out[m] = tuple(seq)
    return out


class OmniConfig:
    def __init__(self, sched: OmniSchedule, vocab: Dict[str, int], n_embd=384, n_layer=6,
                 n_head=6, n_kv_head=2, dropout=0.0, qk_norm=True,
                 cond_mode="gt_embed", mask_mode="block", max_pos=None,
                 n_experts=0, expert_map=None, moe_ffn=0, use_flex=None):
        self.sched = sched
        self.vocab_raw = vocab
        self.vocab = normalize_vocab(vocab, sched.modalities, sched.K)
        self.n_embd, self.n_layer = n_embd, n_layer
        self.n_head, self.n_kv_head = n_head, n_kv_head
        self.dropout, self.qk_norm = dropout, qk_norm
        self.cond_mode, self.mask_mode = cond_mode, mask_mode
        self.n_experts = int(n_experts or 0)
        self.expert_map = expert_map
        self.moe_ffn = int(moe_ffn or 0)          # 0 = derive the expert width from mlp_ratio
        # None = auto (flex on sm_80+), False = force the dense SDPA mask. Measured on an A100 at
        # T=1764, B=8, H=16: dense materialises [B,H,T,T] = 1.59 GiB per tensor and its backward
        # keeps several, so it OOMs while asking for 1.48 GiB with 33.6 GiB already resident - and
        # it runs at 28 TFLOPS against 99 for the block-sparse path. Flex's Triton kernel failing to
        # compile (182 KB of shared memory wanted, 166 KB available) is still the better trade: the
        # unfused fallback is fast. Leave this None unless the packed length is small.
        self.use_flex = use_flex
        self.max_pos = max_pos or max(sched.base_len[m] for m in sched.modalities)

    # -- vocab -------------------------------------------------------------
    def vocab_at(self, m, k):
        return self.vocab[m][k]

    def vocab_max(self, m):
        return max(self.vocab[m])

    def is_hetero(self, m=None):
        mods = self.sched.modalities if m is None else (m,)
        return any(len(set(self.vocab[mm])) > 1 for mm in mods)

    def to_dict(self):
        # A uniform modality serialises back to the old `int` form so old cfgs round-trip.
        v = {m: (list(seq) if len(set(seq)) > 1 else seq[0]) for m, seq in self.vocab.items()}
        return dict(K=self.sched.K, R=self.sched.R, base_len=dict(self.sched.base_len),
                    vocab=v, n_embd=self.n_embd, n_layer=self.n_layer,
                    n_head=self.n_head, n_kv_head=self.n_kv_head, dropout=self.dropout,
                    qk_norm=self.qk_norm, cond_mode=self.cond_mode, mask_mode=self.mask_mode,
                    n_experts=self.n_experts, expert_map=self.expert_map)


def level_ids_for(sched: OmniSchedule, T: int, device="cpu"):
    """Level id per packed position, truncated to a prefix of length T (generation grows T)."""
    lev, mod, pos = sched.positions()
    return lev[:T].to(device), mod[:T].to(device), pos[:T].to(device)


_CACHE_MAX = 8                   # entries for _STATIC below


def _geometry(sched: OmniSchedule):
    """Everything a per-T tensor depends on, as a hashable key.

    Not `id(sched)`: a schedule rebuilt from the same cfg between calls must hit the same entry.
    """
    return (tuple(sorted(sched.base_len.items())), sched.K, sched.R,
            sched.prefix_len, tuple(sched.modalities), sched.null_code)


_STATIC = {}                     # (geometry, T, device, head_dim, emap) -> lev, mod, pos, rope, eid


def static_tensors(sched: OmniSchedule, T: int, device, head_dim: int, emap=None):
    """Level ids, positional ids, RoPE tables and expert routing for a packed length T.

    Built once instead of once per step. encode() used to rebuild all of it on every call: three
    fresh tensors and their host->device copies, a python loop over the modality masks, and two
    sin/cos tables over T positions. They are pure functions of the schedule and T, so at 0.18 s
    per step that was pure launch overhead that never touched the maths.
    """
    key = (_geometry(sched), int(T), str(device), int(head_dim),
           None if emap is None else tuple(emap))
    hit = _STATIC.get(key)
    if hit is None:
        lev, mod, pos = level_ids_for(sched, T, device)
        eid = None
        if emap is not None:
            eid = torch.zeros_like(mod)
            for i, ex in enumerate(emap):        # prefix slots (-1) stay on expert 0
                eid[mod == i] = ex
        # per-expert position index, so the MoE path never branches on tensor values
        plan = None
        if eid is not None and emap is not None:
            plan = tuple((eid == e).nonzero(as_tuple=False).squeeze(1)
                         for e in range(max(emap) + 1))
        hit = (lev, mod, pos, precompute_rope_cos_sin(pos, head_dim), eid, plan)
        if len(_STATIC) >= _CACHE_MAX:
            _STATIC.pop(next(iter(_STATIC)))
        _STATIC[key] = hit
    return hit


def trunc_mask(sched: OmniSchedule, T: int, dtype=torch.float32, device="cpu",
               cross_level=False, mask_mode="block"):
    """Additive [1,1,T,T] mask for a packed length T.

    Deliberately not cached. It was, keyed on (geometry, T, dtype, device, cross_level, mode), and
    the cache had to refuse anything over 64 MB - i.e. it silently rebuilt every step at any
    T above ~4000, which is exactly the range we care about. So it bought nothing where it
    mattered and was a second place for the mask to be wrong. Build it; it is pure.
    """
    lev, _, _ = level_ids_for(sched, T, device)
    q, kv = lev[:, None], lev[None, :]
    if mask_mode == "ar":
        # raster order: everything strictly before, plus the prefix
        idx = torch.arange(T, device=device)
        allowed = idx[None, :] <= idx[:, None]
    elif mask_mode == "strict":
        # Generation reality: every code of level k is emitted in ONE forward pass, so no position
        # may see a sibling of its own level. Level k attends to levels < k only. Level 0 has no
        # coarser level, so it attends to its own block - which pack() always fills with NULL, at
        # train time too, so there is nothing to copy there either.
        allowed = (kv == -1) | ((q == 0) & (kv == 0)) | ((kv >= 0) & (kv < q))
    else:
        allowed = (kv == -1) | (kv == q)
        if cross_level:
            allowed = allowed | ((q >= 0) & (kv >= 0) & (kv < q))
    m = torch.zeros(T, T, dtype=dtype, device=device)
    m.masked_fill_(~allowed, float("-inf"))
    return m.view(1, 1, T, T)


def kv_ids(lev):                                            # kept for clarity in trunc_mask
    return lev


_FLEX_CACHE = OrderedDict()      # LRU: the block mask is ~200 MB at T=7119, so keep few
_FLEX_CACHE_MAX = 8


def trunc_flex(sched: OmniSchedule, B: int, H: int, T: int, device="cuda",
               cross_level=False, mask_mode="block"):
    """Block mask for the fused kernel, built once per (length, mask) and broadcast.

    Two things matter at kill-test length. The mask depends only on (level, position), never on
    b or h, so B/H go in as None: create_block_mask then derives block sparsity from one T x T
    grid instead of a B x H x T x T one, which at T=7119, B=2, H=12 is 9 GiB of floats and OOMs
    on any consumer card. And it is cached, because rebuilding a 200 MB mask every step costs
    more than the forward pass at these sizes.
    """
    assert create_block_mask is not None, "flex_attention unavailable"
    key = (T, mask_mode, bool(cross_level), str(device),
           tuple(sorted(sched.base_len.items())), sched.K)
    hit = _FLEX_CACHE.get(key)
    if hit is not None:
        _FLEX_CACHE.move_to_end(key)
        return hit
    lev, _, _ = level_ids_for(sched, T, device)

    def mask_mod(b, h, q_idx, kv_idx):
        if mask_mode == "ar":
            return kv_idx <= q_idx
        ql, kl = lev[q_idx], lev[kv_idx]
        if mask_mode == "strict":
            return (kl == -1) | ((ql == 0) & (kl == 0)) | ((kl >= 0) & (kl < ql))
        allowed = (kl == -1) | (kl == ql)
        if cross_level:
            allowed = allowed | ((ql >= 0) & (kl >= 0) & (kl < ql))
        return allowed

    bm = create_block_mask(mask_mod, B=None, H=None, Q_LEN=T, KV_LEN=T, device=device)
    _FLEX_CACHE[key] = bm
    while len(_FLEX_CACHE) > _FLEX_CACHE_MAX:
        _FLEX_CACHE.popitem(last=False)
    return bm


class OmniNCM(nn.Module):
    def __init__(self, cfg: OmniConfig):
        super().__init__()
        self.cfg, self.sched = cfg, cfg.sched
        d, K = cfg.n_embd, cfg.sched.K
        # The embedding table is sized by the widest level of that modality; coarse levels simply
        # never index past their own (smaller) range.
        self.tok_emb = nn.ModuleDict({m: nn.Embedding(cfg.vocab_max(m) + 1, d)
                                      for m in cfg.sched.modalities})
        self.mod_emb = nn.Embedding(len(cfg.sched.modalities) + 1, d)   # +1 = prefix slot
        self.level_emb = nn.Embedding(K, d)
        self.pos_emb = nn.Embedding(cfg.max_pos + 1, d)
        self.cond_proj = nn.Linear(d, d)
        # MoE: shared trunk, per-modality experts. A uniform model keeps the dense FFN, so old
        # checkpoints keep loading.
        self.moe = bool(cfg.n_experts)
        if self.moe:
            from omni.moe import MoEBlock, resolve_expert_map
            self._emap, self.n_experts = resolve_expert_map(
                cfg.sched.modalities, cfg.expert_map, cfg.n_experts)
            blocks = [MoEBlock(Block(d, cfg.n_head, cfg.n_kv_head, cfg.dropout, qk_norm=cfg.qk_norm),
                               d, self.n_experts, ffn_hidden=cfg.moe_ffn or None)
                      for _ in range(cfg.n_layer)]
        else:
            self._emap, self.n_experts = None, 0
            blocks = [Block(d, cfg.n_head, cfg.n_kv_head, cfg.dropout, qk_norm=cfg.qk_norm)
                      for _ in range(cfg.n_layer)]
        self.blocks = nn.ModuleList(blocks)
        self.ln_f = RMSNorm(d)
        # One head per modality when every level shares a width (old layout, loads old ckpts);
        # one head per (modality, level) when the finest level is wider (BPE ids).
        self.heads = nn.ModuleDict()
        for m in cfg.sched.modalities:
            if cfg.is_hetero(m):
                for k in range(K):
                    self.heads[f"{m}_L{k}"] = nn.Linear(d, cfg.vocab_at(m, k), bias=False)
            else:
                self.heads[m] = nn.Linear(d, cfg.vocab_at(m, 0), bias=False)

    def head(self, m, k):
        key = f"{m}_L{k}"
        return self.heads[key] if key in self.heads else self.heads[m]

    # ------------------------------------------------------------------ packing
    def _mod_level_emb(self, m, k, codes_mk, cond):
        l = codes_mk.size(-1)
        e = (self.tok_emb[m](codes_mk)
             + self.level_emb.weight[k]
             + self.mod_emb.weight[self.sched.modalities.index(m)]
             + self.pos_emb(torch.arange(l, device=codes_mk.device)))
        return e if cond is None else e + self.cond_proj(cond)

    def pack(self, codes: Dict[str, torch.Tensor], upto: Optional[int] = None,
             dummy_level: Optional[int] = None):
        """codes[m] = [B, flat_len(m)] -> packed [B, T, d] over levels < upto.

        The packed input at a position is NEVER that position's own target code, in either mode:

          * `strict` (level-parallel): level k is fed the codes of level k-1, upsampled to l_k.
            Level 0 is fed NULL - it has no coarser level, so it is a prior over the coarsest
            codes and nothing else.
          * `ar`: the raster-shifted sequence, i.e. position j is fed code j-1.

        Without that shift the head can read its own input token straight out of the residual
        stream (the token embedding is in `h` by construction), so the model learns an identity
        map: train loss -> 1e-8, and at generation - where the input is NULL or a guess - it
        emits garbage, scoring BELOW a constant predictor. `block` keeps the old (leaky) layout
        for comparison and should not be quoted.
        """
        S, K = self.sched, self.sched.K
        B = next(iter(codes.values())).size(0)
        stop = K if upto is None else upto
        mode = self.cfg.mask_mode
        shifted = {}
        if mode == "ar":                                  # position j is fed code j-1
            for m in S.modalities:
                c = codes[m]
                shifted[m] = torch.cat([torch.zeros(B, 1, dtype=c.dtype, device=c.device),
                                        c[:, :-1]], -1)
        parts, conds = [], {}
        for k in range(spread(stop, dummy_level)):
            block = []
            for m in S.modalities:
                l = S.level_len(m, k)
                off = S.offsets(m)[k]
                if mode == "strict":
                    if k == 0:
                        c = torch.zeros(B, l, dtype=torch.long, device=self._dev)
                    else:
                        prev = codes[m][:, S.offsets(m)[k - 1]:S.offsets(m)[k]]
                        lp = prev.size(-1)
                        idx = (torch.arange(l, device=prev.device) * lp) // l   # nearest neighbour
                        c = prev[:, idx]
                    if dummy_level is not None and k > dummy_level:
                        c = torch.zeros(B, l, dtype=torch.long, device=self._dev)
                elif mode == "ar":
                    c = shifted[m][:, off:off + l]
                elif dummy_level is not None and k == dummy_level:
                    c = torch.zeros(B, l, dtype=torch.long, device=self._dev)
                else:
                    c = codes[m][:, off:off + l]
                block.append(self._mod_level_emb(m, k, c, conds.get(m)))
            parts.append(torch.cat(block, 1))
            # condition for the next level = mean of this level's embeddings, per modality
            for i, m in enumerate(S.modalities):
                conds[m] = block[i].mean(1, keepdim=True).expand(-1, S.level_len(m, k + 1), -1) \
                    if k + 1 < K else None
        return torch.cat(parts, 1)

    @property
    def _dev(self):
        return self.pos_emb.weight.device

    # ------------------------------------------------------------------ forward
    def _expert_ids(self, mod_idx):
        """Packed-position modality index -> expert id under this model's expert map.

        preflight calls this to count the tokens each expert would receive, so a forced mapping that
        leaves an expert with nothing shows up before training instead of being inferred from a flat
        loss curve. It was referenced from omni/preflight.py long before it existed here: the MoE
        branch of preflight had never run, so nothing caught the missing method.
        """
        if self._emap is None:
            raise ValueError("dense model has no experts; _expert_ids needs an expert_map")
        eid = torch.zeros_like(mod_idx)
        for i, ex in enumerate(self._emap):        # prefix slots keep expert 0, as in static_tensors
            eid[mod_idx == i] = ex
        return eid

    def encode(self, x, mask_mode=None, cross_level=False, use_flex=None):
        T, B = x.size(1), x.size(0)
        mask_mode = mask_mode or self.cfg.mask_mode
        if use_flex is None:
            use_flex = self.cfg.use_flex      # config override; False forces the dense SDPA mask
        if use_flex is None:
            # flex_attention's fused kernel needs sm_80+. On a free T4 (sm_75) it either errors or
            # falls back to something slower than the dense SDPA path, so gate on capability.
            use_flex = (HAS_FLEX and x.is_cuda
                        and torch.cuda.get_device_capability(x.device)[0] >= 8)
        kw = {}
        if use_flex:
            kw["block_mask"] = trunc_flex(self.sched, B, self.cfg.n_head, T, x.device,
                                          cross_level, mask_mode)
        else:
            kw["dense_mask"] = trunc_mask(self.sched, T, x.dtype, x.device, cross_level, mask_mode)
        lev, mod, pos, rope, eid, plan = static_tensors(
            self.sched, T, x.device, self.cfg.n_embd // self.cfg.n_head,
            tuple(self._emap) if self._emap is not None else None)
        h = x
        for b in self.blocks:
            h = b(h, rope=rope, mod_id=eid, plan=plan, **kw) if self.moe \
                else b(h, rope=rope, **kw)
        return self.ln_f(h)


    def forward(self, batch, cross_level=False, use_flex=None):
        """batch = {"codes": {m: [B,flat]}, "present": {m: [B] bool}} -> {(m,k): logits}."""
        codes = batch["codes"]
        x = self.pack(codes)
        h = self.encode(x, cross_level=cross_level, use_flex=use_flex)
        out = {}
        for k in range(self.sched.K):
            q0 = self._qoff(k)
            for m in self.sched.modalities:                        # modality slot inside the level block
                l = self.sched.level_len(m, k)
                off_in_block = sum(self.sched.level_len(mm, k)
                                   for mm in self.sched.modalities[:self.sched.modalities.index(m)])
                out[(m, k)] = self.head(m, k)(h[:, q0 + off_in_block:q0 + off_in_block + l])
        return out

    def _qoff(self, k):
        """Packed offset of level k's *text* slot plus preceding modalities."""
        S = self.sched
        return sum(S.block_width(j) for j in range(k))

    # ------------------------------------------------------------------ generation
    @torch.no_grad()
    def generate(self, batch, level_cutoff=None, temperature=0.0, use_flex=None):
        """Coarse-to-fine: NFE == level_cutoff (one forward per level). Returns codes[m] flat."""
        S = self.sched
        K = S.K if level_cutoff is None else level_cutoff
        codes = {m: torch.zeros_like(batch["codes"][m]) for m in S.modalities}
        for k in range(K):
            x = self.pack(codes, upto=k + 1, dummy_level=k)
            h = self.encode(x, use_flex=use_flex)
            q0 = self._qoff(k)
            c = 0
            for m in S.modalities:
                l = S.level_len(m, k)
                lg = self.head(m, k)(h[:, q0 + c:q0 + c + l])
                c += l
                if temperature and temperature > 0:
                    idx = torch.multinomial(F.softmax(lg / temperature, -1).reshape(-1, lg.size(-1)), 1)
                    idx = idx.reshape(lg.size(0), l)
                else:
                    idx = lg.argmax(-1)
                off = S.offsets(m)[k]
                codes[m][:, off:off + l] = idx
        return codes

    @torch.no_grad()
    def generate_ar(self, batch, max_len=None, use_flex=None):
        """Autoregressive baseline: one forward per token. NFE == number of tokens."""
        S = self.sched
        dev = self._dev
        flat = {m: torch.zeros(S.flat_len(m), dtype=torch.long, device=dev) for m in S.modalities}
        order = [(k, m, i) for k in range(S.K) for m in S.modalities
                 for i in range(S.level_len(m, k))]
        if max_len:
            order = order[:max_len]
        buf = {m: flat[m].unsqueeze(0) for m in S.modalities}
        for (k, m, i) in order:
            x = self.pack(buf, upto=k + 1, dummy_level=k)
            h = self.encode(x, use_flex=use_flex)
            q0 = self._qoff(k) + sum(S.level_len(mm, k) for mm in S.modalities[:S.modalities.index(m)]) + i
            idx = self.head(m, k)(h[:, q0:q0 + 1]).argmax(-1)
            buf[m][0, S.offsets(m)[k] + i] = idx[0, 0]
        return {m: buf[m] for m in S.modalities}, len(order)


def spread(stop, dummy_level):
    """How many levels to build: up to `stop`, extended by one if the dummy sits at the end."""
    return stop if dummy_level is None else max(stop, dummy_level + 1)


# ---------------------------------------------------------------------- loss
def omni_loss(logits, batch, sched: OmniSchedule, label_smoothing=0.0, chunk=None):
    """CE over every present, non-null code. Absent modalities and padded slots are masked.

    Sync-free on purpose: the divisor is accumulated as a tensor and an all-masked level is
    zeroed with torch.where instead of `if not keep.any(): continue`. Each of those old branches
    was a device-wide barrier in the middle of the graph - 18 of them per step at K=6, three
    modalities - so the forward could never be queued ahead of the host. The sum is identical.
    """
    codes, pres = batch["codes"], batch["present"]
    total, n = None, None
    for k in range(sched.K):
        for m in sched.modalities:
            l, off = sched.level_len(m, k), sched.offsets(m)[k]
            tg = codes[m][:, off:off + l]
            keep = (tg != sched.null_code) & pres[m].unsqueeze(1)
            cnt = keep.sum()
            lg = logits[(m, k)]
            flat_lg = lg.reshape(-1, lg.size(-1))
            flat_keep = keep.reshape(-1)
            # one masked target vector for both paths: -100 is CE's ignore index, null_code is a
            # legal code and must never be counted as a target.
            flat_tg = tg.reshape(-1).clone()
            flat_tg[~flat_keep] = -100
            if chunk:
                loss_k = flat_lg.new_zeros(())
                for i in range(0, flat_lg.size(0), chunk):
                    loss_k = loss_k + F.cross_entropy(
                        flat_lg[i:i + chunk], flat_tg[i:i + chunk], reduction="sum",
                        ignore_index=-100, label_smoothing=label_smoothing)
            else:
                loss_k = F.cross_entropy(flat_lg, flat_tg, reduction="sum",
                                         ignore_index=-100, label_smoothing=label_smoothing)
            # every target masked -> CE is nan; contribute nothing instead of poisoning the mean
            loss_k = torch.where(cnt > 0, loss_k, loss_k.new_zeros(()))
            total = loss_k if total is None else total + loss_k
            n = cnt if n is None else n + cnt
    if total is None or n is None:
        return torch.zeros((), device=self_dev(codes))
    return total / n.clamp(min=1)


def omni_loss_stats(logits, batch, sched: OmniSchedule, label_smoothing=0.0):
    """-> (mean CE, {(m,k): {"ce": float, "n": int}}).

    Same arithmetic as `omni_loss`, but it also returns the per-level breakdown, which is what
    the preflight gates on and what TensorBoard plots. Kept separate so the optimiser path stays
    the single scalar it always was.
    """
    codes, pres = batch["codes"], batch["present"]
    total, n, out = None, 0, {}
    for k in range(sched.K):
        for m in sched.modalities:
            l, off = sched.level_len(m, k), sched.offsets(m)[k]
            tg = codes[m][:, off:off + l]
            keep = (tg != sched.null_code) & pres[m].unsqueeze(1)
            if not keep.any():
                continue
            lg = logits[(m, k)]
            flat_lg, flat_keep = lg.reshape(-1, lg.size(-1)), keep.reshape(-1)
            flat_tg = tg.reshape(-1).clone()
            flat_tg[~flat_keep] = -100
            loss_k = F.cross_entropy(flat_lg, flat_tg, reduction="sum", ignore_index=-100,
                                     label_smoothing=label_smoothing)
            cnt = int(flat_keep.sum())
            out[(m, k)] = {"ce": float(loss_k.detach()) / max(1, cnt), "n": cnt}
            total = loss_k if total is None else total + loss_k
            n += cnt
    mean = (total / max(1, n)) if total is not None else torch.zeros((), device=self_dev(codes))
    return mean, out


def self_dev(codes):
    return next(iter(codes.values())).device


# ---------------------------------------------------------------------- selfcheck
def selfcheck(verbose=True):
    """Invariants that must hold before any training. Run this on the target GPU, not in review."""
    from omni.schedule import OmniSchedule as Sch

    sched = Sch(base_len={"text": 8, "image": 8, "audio": 8, "video": 4}, K=3, R=2)
    vocab = {m: 16 for m in MODALITIES}
    cfg = OmniConfig(sched, vocab, n_embd=32, n_layer=2, n_head=4, n_kv_head=2)
    net = OmniNCM(cfg)
    B = 2
    codes = {m: torch.randint(1, 16, (B, sched.flat_len(m))) for m in MODALITIES}
    pres = {m: torch.ones(B, dtype=torch.bool) for m in MODALITIES}
    batch = {"codes": codes, "present": pres}
    out = net(batch)

    # 1. shapes
    for m in MODALITIES:
        for k in range(sched.K):
            assert out[(m, k)].shape == (B, sched.level_len(m, k), vocab[m]), (m, k, out[(m, k)].shape)

    # 1b. expert width. `moe_ffn` has to mean what it says: the mlp_ratio path rounds through
    # int(2*r*d/3//8*8), so "360M per expert" comes out a slightly different number, and the whole
    # 50x token budget is derived from this count. Asserted, not trusted.
    wcfg = OmniConfig(sched, vocab, n_embd=32, n_layer=2, n_head=4, n_kv_head=2,
                      n_experts=3, moe_ffn=128)
    wnet = OmniNCM(wcfg)
    e0 = wnet.blocks[0].experts
    per_layer = sum(w.numel() for w in (e0.w_gate[0].weight, e0.w_up[0].weight,
                                        e0.w_down[0].weight))
    assert per_layer == 3 * wcfg.n_embd * 128, {
        "h": e0.w_gate[0].weight.shape[0], "per_layer": per_layer, "want": 3 * wcfg.n_embd * 128}
    assert e0.w_gate[0].weight.shape[0] == 128, "ffn_hidden not applied to the gate"
    got = sum(p.numel() for n, p in wnet.named_parameters() if ".experts.w_" in n)
    want = wcfg.n_layer * 3 * (3 * wcfg.n_embd * 128 + 2 * 128 + wcfg.n_embd)  # + gate/up/down bias
    assert got == want, f"expert total: {got} != {want}"

    # 2. level k cannot see level k+1: perturb the finest level, the coarsest logits must not move
    net.eval()
    with torch.no_grad():
        base = net(batch)
        c2 = {m: codes[m].clone() for m in MODALITIES}
        fine_off = sched.offsets("text")[sched.K - 1]
        c2["text"][:, fine_off:] = (c2["text"][:, fine_off:] + 1) % 16
        after = net({"codes": c2, "present": pres})
        d0 = (base[("text", 0)] - after[("text", 0)]).abs().max().item()
        assert d0 == 0.0, f"level 0 moved when the finest level changed: {d0}"

    # 3. cross-level knob is wired: turning it on MUST change level-1 logits
    with torch.no_grad():
        cl = net(batch, cross_level=True)
        d1 = (base[("text", 1)] - cl[("text", 1)]).abs().max().item()
        assert d1 > 0, "cross_level=True changed nothing - the mask knob is dead"

    # 4. gradients reach every modality and every level
    net.train()
    loss = omni_loss(net(batch), batch, sched)
    loss.backward()
    for m in MODALITIES:
        g = net.tok_emb[m].weight.grad
        assert g is not None and g.abs().sum() > 0, f"no gradient on {m} embeddings"
    for k in range(sched.K):
        assert net.level_emb.weight.grad[k].abs().sum() > 0, f"level {k} unused"

    # 5. absent modality is masked, not silently trained
    p2 = {m: torch.ones(B, dtype=torch.bool) for m in MODALITIES}
    p2["audio"] = torch.zeros(B, dtype=torch.bool)
    l_absent = omni_loss(net(batch), {"codes": codes, "present": p2}, sched)
    l_full = omni_loss(net(batch), batch, sched)
    assert l_absent.item() != l_full.item(), "masking an absent modality did not change the loss"

    # 6. AR mode is causal: perturbing the last position cannot change earlier logits
    net.eval()
    cfg_ar = OmniConfig(sched, vocab, n_embd=32, n_layer=2, n_head=4, n_kv_head=2, mask_mode="ar")
    ar = OmniNCM(cfg_ar).eval()
    with torch.no_grad():
        a0 = ar(batch)[("text", 0)]
        c3 = {m: codes[m].clone() for m in MODALITIES}
        c3["text"][:, -1:] = (c3["text"][:, -1:] + 3) % 16
        a1 = ar({"codes": c3, "present": pres})[("text", 0)]
        assert (a0[:, :-1] - a1[:, :-1]).abs().max().item() == 0.0, "AR mask leaks the future"

    # 7. generation costs exactly K forwards and returns the right shapes
    gen = net.generate(batch)
    for m in MODALITIES:
        assert gen[m].shape == (B, sched.flat_len(m)), (m, gen[m].shape)

    if verbose:
        print("omni selfcheck: 7/7 invariants hold")
    return True


if __name__ == "__main__":
    selfcheck()


In [ ]:
%%writefile .ncm_stage/omni/moe.py
"""Modality specialists: one FFN group per modality, routed by modality tag, not by a router.

Why forced routing instead of a learned gate: the training mix is 70% text / 20% vision / 8%
audio / 2% video. A learned router collapses onto text within a few thousand steps at that
imbalance, and the collapse is silent - the loss looks fine because text is most of the loss.
Tagging the expert group by modality removes the failure mode entirely and turns load balance
into a data-mix decision instead of a training bug.

What this buys: the trunk (attention, embeddings, norms) is shared, so levels still talk across
modalities at the same altitude - that is the omni thesis - while each modality's feed-forward
capacity is its own. Text and audio stop fighting over one FFN, which is what the 0.92-vs-0.26
finest-level gap in the first eval was measuring.

Video has no expert of its own: a video sample is a stack of frames encoded as images, so it has
no separate code distribution to specialise on. It routes to the image expert by default.

Active parameters per token = trunk + one expert. Total parameters = trunk + n_experts.
"""
import torch
import torch.nn as nn
import torch.nn.functional as F

# video -> image: frames are images, not a modality (see the module docstring).
DEFAULT_EXPERT_MAP = {"text": 0, "image": 1, "audio": 2, "video": 1}


def resolve_expert_map(modalities, mapping=None, n_experts=None):
    """-> (per-modality expert index list in `modalities` order, n_experts)."""
    mp = dict(DEFAULT_EXPERT_MAP if mapping is None else mapping)
    idx = [int(mp.get(m, 0)) for m in modalities]
    n = int(n_experts or (max(idx) + 1))
    if n < 1:
        raise ValueError(f"n_experts must be >= 1, got {n}")
    return [min(max(e, 0), n - 1) for e in idx], n


class ModalityExperts(nn.Module):
    """n_experts independent SwiGLU FFNs; every token goes to exactly one of them."""

    def __init__(self, n_embd, n_experts, mlp_ratio=4.0, ffn_hidden=None):
        super().__init__()
        # `ffn_hidden` pins the SwiGLU width directly. mlp_ratio alone is a lossy way to ask for it:
        # int(2*r*d/3//8*8) rounds, so at a target like 360M per expert the ratio stops meaning what
        # it says, and a 4.8 ratio at d=1536 is a number nobody can read back.
        h = int(ffn_hidden) // 8 * 8 if ffn_hidden else int(2 * mlp_ratio * n_embd / 3 // 8 * 8)
        self.n_experts = n_experts
        self.w_gate = nn.ModuleList([nn.Linear(n_embd, h) for _ in range(n_experts)])
        self.w_up = nn.ModuleList([nn.Linear(n_embd, h) for _ in range(n_experts)])
        self.w_down = nn.ModuleList([nn.Linear(h, n_embd) for _ in range(n_experts)])

    def forward(self, x, mod_id, plan=None):
        """x = [B, T, d], mod_id = [T] expert index per packed position -> [B, T, d].

        `plan` is the per-expert position index, precomputed once with the routing tensor. Routing
        is a pure function of the schedule, so it never changes between steps: computing
        `(mod_id == e).nonzero()` here instead would be 3 kernels per block per step, and the
        `if not bool(sel.any()): continue` this replaces was a device sync plus a dynamo graph
        break in the middle of the forward - same bug the training loss had.
        """
        B, T, d = x.shape
        if mod_id is None:
            raise ValueError("ModalityExperts needs mod_id; forced routing has no fallback")
        if mod_id.dim() != 1 or mod_id.numel() != T:
            raise ValueError(f"mod_id must be [T] with T={T}, got {tuple(mod_id.shape)}")
        if plan is None:
            plan = tuple((mod_id == e).nonzero(as_tuple=False).squeeze(1)
                         for e in range(self.n_experts))
        out = x.new_zeros(B, T, d)
        for e, idx in enumerate(plan):
            if idx.numel() == 0:
                continue                                   # expert got no tokens: host-side shape
            xe = x.index_select(1, idx)
            ye = self.w_down[e](F.silu(self.w_gate[e](xe)) * self.w_up[e](xe))
            out.index_copy_(1, idx, ye.to(out.dtype))      # autocast emits bf16, out is fp32
        return out

    @torch.no_grad()
    def usage(self, mod_id):
        """Token count per expert - logged so a dead expert is visible, not inferred."""
        return torch.bincount(mod_id, minlength=self.n_experts).tolist()


class MoEBlock(nn.Module):
    """A core.Block with its dense FFN swapped for per-modality experts.

    Takes ownership of the block's attention and norms and drops the original FFN parameters
    instead of leaving them dead, so the parameter count means what it says. Submodule names are
    kept (ln1, ln2, attn), so state_dict keys stay blocks.<i>.attn... as before.
    """

    def __init__(self, base, n_embd, n_experts, mlp_ratio=4.0, ffn_hidden=None):
        super().__init__()
        self.ln1, self.ln2, self.attn = base.ln1, base.ln2, base.attn
        for name in ("w_gate", "w_up", "w_down"):
            if hasattr(base, name):
                delattr(base, name)                        # frees the dense FFN, not just ignores it
        self.experts = ModalityExperts(n_embd, n_experts, mlp_ratio, ffn_hidden=ffn_hidden)

    def forward(self, x, block_mask=None, dense_mask=None, rope=None, mod_id=None, plan=None):
        x = x + self.attn(self.ln1(x), block_mask=block_mask, dense_mask=dense_mask, rope=rope)
        return x + self.experts(self.ln2(x), mod_id, plan)


def active_params(model):
    """Trunk + one expert, i.e. what a forward pass actually touches per token."""
    total = sum(p.numel() for p in model.parameters())
    experts = 0
    for mod in model.modules():
        if isinstance(mod, ModalityExperts):
            per = sum(p.numel() for p in mod.parameters()) / mod.n_experts
            experts += sum(p.numel() for p in mod.parameters()) - per   # keep just one expert
    return total - experts


In [ ]:
%%writefile .ncm_stage/omni/preflight.py
"""Preflight (test C): does the new head + expert routing actually learn, before any real run?

Runs automatically before training (see train.run / train.run_duplex) and refuses to start a
paid run when the answer is no. It is deliberately NOT a data-quality test - it uses a synthetic
stream whose coarse and fine levels are a deterministic function of one latent per sample, so
the only thing being measured is whether information reaches each level through the chosen mask.

Why synthetic: on real data a level can be unlearnable because the corpus is hard, which tells
you nothing about the code. Here the relation is trivially learnable ("copy the coarser level"),
so a level that does not converge is a wiring bug, not a data problem. The copy relation is also
the exact operation the strict mask permits, so this is the strongest cheap test of the thesis.

What it gates on (k >= 1 only - level 0 has no coarser level and can only learn a marginal):
  * every fine level's CE falls by >= 0.5 nats from ln(V_k)  -> information flows
  * the widest level (BPE 8k/32k text) reaches CE < 1.0      -> the big head is trainable
  * no NaN/inf loss, and every expert group gets gradient    -> routing is alive, not collapsed

Level 0 is printed for reference and never gated: it is a prior over the coarsest codes.
"""
import copy
import math
import sys
import time
from typing import Dict

import torch
from tqdm.auto import tqdm

from omni.data import to_device
from omni.model import omni_loss_stats
from omni.schedule import OmniSchedule


def _structured_batch(sched, ocfg, B, device, seed=0):
    """ocfg is the OmniConfig (per-level vocab lives there), not the raw cfg dict."""
    """One latent per sample; every level of every modality is a deterministic function of it.

    code_k[i] = (latent * 3 + i) % (V_k - 1) + 1  ->  level k+1 can be read off level k, and the
    finest level of a heterogeneous head must emit it through a much wider softmax.
    """
    g = torch.Generator().manual_seed(seed)
    latent = torch.randint(0, 16, (B, 1), generator=g)
    codes, present = {}, {}
    for m in sched.modalities:
        flat = torch.zeros(B, sched.flat_len(m), dtype=torch.long)
        for k in range(sched.K):
            V = ocfg.vocab_at(m, k)
            l, off = sched.level_len(m, k), sched.offsets(m)[k]
            pos = torch.arange(l).unsqueeze(0)
            flat[:, off:off + l] = (latent * 3 + pos) % max(1, V - 1) + 1
        codes[m] = flat.to(device)
        present[m] = torch.ones(B, dtype=torch.bool, device=device)
    return {"codes": codes, "present": present}


def run_preflight(cfg, writer=None, steps=None, batch=None, device=None, verbose=True):
    """-> dict(passed, rows, reason). Aborts the caller unless it passed."""
    from omni.train import build as build_model          # local: train imports this module

    pf = cfg.get("preflight", {}) or {}
    steps = int(steps or pf.get("steps", 300))
    B = int(batch or pf.get("batch", 8))
    device = device or ("cuda" if torch.cuda.is_available() else "cpu")

    c = copy.deepcopy(cfg)
    for k in ("n_embd", "n_layer", "n_head", "n_kv_head"):
        if k in pf:
            c["model"][k] = pf[k]
    c["model"]["mask_mode"] = pf.get("mask_mode", "strict")

    t0 = time.time()
    model, sched = build_model(c)
    oc = model.cfg
    model.to(device).train()
    P = sum(p.numel() for p in model.parameters())
    active = P
    if getattr(model, "moe", False):
        from omni.moe import ModalityExperts
        for mod in model.modules():
            if isinstance(mod, ModalityExperts):
                per = sum(p.numel() for p in mod.parameters()) / mod.n_experts
                active -= (mod.n_experts - 1) * per

    if verbose:
        print("\n=== PREFLIGHT (test C) ===")
        if str(device) == "cpu":
            print("!! running on CPU: 300 steps at this size takes minutes, not seconds.")
            print("   Colab: Runtime -> Change runtime type -> GPU, then re-run the hardware cell.")
        print(f"model {P/1e6:.2f}M total | {active/1e6:.2f}M active/token | "
              f"experts {getattr(model, 'n_experts', 0)} | mask {c['model']['mask_mode']}")
        vs = " ".join(f"{m}=" + (str(oc.vocab[m][0]) if len(set(oc.vocab[m])) == 1
                                 else f"{oc.vocab[m][0]}..{oc.vocab[m][-1]}")
                      for m in sched.modalities)
        print(f"levels K={sched.K} | vocab {vs}")
        print(sched.describe())

    # A deliberately hotter LR than the main run: this is a wiring test, not a quality test, and
    # the copy relation is learnable in a few hundred steps at 3e-3. Gating on the real run's LR
    # would turn a slow warmup into a false "the head is broken".
    # Hotter than the real run on purpose: this is a wiring test, not a quality run, and gating on
    # the warmup LR turns a slow start into a false "the head is broken". But it must stay in the
    # same regime as the model it is testing: 3e-3 was tuned on a 3.25M (d128) model and applied
    # unchanged to d768 here, which is 15x the preset's own 2e-4. That produced a sawtooth
    # (4.4 -> 3.2 -> 2.9 -> 3.3 -> 1.98) and mid levels pinned at ~3.2. Scale with width instead.
    base_lr = float((cfg.get("train") or {}).get("lr", 2e-4))
    lr = float(pf.get("lr") or min(3e-3, max(3e-4, 2.0 * base_lr)))
    print(f"  preflight lr {lr:g} (train lr {base_lr:g}; override: preflight.lr)")
    opt = torch.optim.AdamW(model.parameters(), lr=lr,
                            betas=(0.9, 0.95), weight_decay=0.01)
    use_bf16 = device == "cuda"
    first, last = {}, {}
    usage = None
    bad = None
    # tqdm draws with \r, and the Colab cell reads the child through a pipe line-by-line, so a bar
    # on a non-tty is invisible until it finishes - which reads as a hang and gets Ctrl-C'd. Show
    # the bar only on a real terminal; otherwise print newline-terminated lines.
    interactive = verbose and sys.stderr.isatty()
    pbar = tqdm(range(steps), desc="preflight", disable=not interactive, leave=True)
    for step in pbar:
        batch_d = _structured_batch(sched, oc, B, device, seed=step)
        with torch.autocast("cuda", dtype=torch.bfloat16, enabled=use_bf16):
            logits = model(batch_d)
            loss, stats = omni_loss_stats(logits, batch_d, sched)
        if not torch.isfinite(loss):
            bad = f"non-finite loss at step {step}"
            break
        opt.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()
        if first is None:
            first = {k: v["ce"] for k, v in stats.items()}
        last = {k: v["ce"] for k, v in stats.items()}
        if getattr(model, "moe", False):
            eid = model._expert_ids(torch.tensor(
                [i for i, m in enumerate(sched.modalities) for _ in range(sched.flat_len(m))]).to(device))
            u = [c * B for c in torch.bincount(eid, minlength=model.n_experts).tolist()]
            usage = u if usage is None else [a + b for a, b in zip(usage, u)]
        if writer is not None:
            writer.add_scalar("preflight/loss", float(loss.detach()), step)
            for (m, k), v in stats.items():
                writer.add_scalar(f"preflight/ce_{m}_L{k}", v["ce"], step)
        if not interactive and verbose and (step + 1) % max(1, steps // 10) == 0:
            print(f"  preflight {step+1:>4}/{steps}  loss {float(loss.detach()):.3f}  "
                  f"{time.time()-t0:.0f}s", flush=True)
        pbar.set_postfix(loss=f"{float(loss.detach()):.3f}", **{
            f"{m[0]}{k}": f"{v['ce']:.2f}" for (m, k), v in list(stats.items())[-3:]})

    rows, passed, reasons = [], True, []
    for m in sched.modalities:
        for k in range(sched.K):
            key = (m, k)
            if key not in last:
                continue
            V = oc.vocab_at(m, k)
            base = math.log(V)
            ce0, ce1 = first.get(key, base), last[key]
            drop = base - ce1
            # Gated only where a coarser level exists to condition on (k >= 1). Two conditions:
            # most of the headroom is gone, and (for the wide BPE head) the absolute CE is far
            # below ln(V), i.e. the softmax is putting its mass on the right code.
            ok = True
            if k >= 1:
                # Scale-free on purpose: ln(V) is the no-learning floor, so half of it means the
                # level is emitting real structure whatever its width. A broken head sits at ~1.0
                # of ln(V), so this catches it without a magic absolute number that a slow warmup
                # can trip.
                ok = (drop >= 1.5) and (ce1 <= 0.5 * base)
            rows.append(dict(m=m, k=k, V=V, base=base, ce0=ce0, ce1=ce1, drop=drop, ok=ok))
            if not ok:
                passed = False
                reasons.append(f"{m} L{k} (V={V}) CE {ce0:.2f}->{ce1:.2f}, drop {drop:.2f}")

    if bad:
        passed, reasons = False, [bad]
    if usage is not None and any(u == 0 for u in usage):
        passed = False
        reasons.append(f"dead expert(s): usage {usage}")

    if verbose:
        print(f"\n  {'level':<12}{'V':>7}{'ln V':>8}{'CE start':>10}{'CE end':>9}{'drop':>8}  gate")
        for r in rows:
            print(f"  {r['m'] + ' L' + str(r['k']):<12}{r['V']:>7}{r['base']:>8.2f}"
                  f"{r['ce0']:>10.2f}{r['ce1']:>9.2f}{r['drop']:>8.2f}  "
                  f"{'PASS' if r['ok'] else 'FAIL'}")
        if usage is not None:
            print(f"  expert token usage: {usage}")
        print(f"\n  PREFLIGHT {'PASSED' if passed else 'FAILED'} in {time.time()-t0:.0f}s")
        if not passed:
            for r in reasons:
                print("   -", r)
    if writer is not None:
        writer.add_scalar("preflight/passed", int(passed), 0)
        writer.flush()
    return dict(passed=passed, rows=rows, reason="; ".join(reasons), usage=usage)


In [ ]:
%%writefile .ncm_stage/omni/train.py
"""Training loop for the omni generator. One loop, two arms (block-diagonal vs AR mask).

Deliberately plain: the run is small enough that a custom loop beats fighting a Trainer, and it
keeps train/val/eval on the exact same batch contract the producer wrote.

python -m omni.train cfg.json            # real run
python -m omni.train cfg.json --smoke 10  # 10 steps, print s/step + projected cost, save nothing
"""
import argparse
import json
import math
import os
import sys
import time

import torch
from tqdm.auto import tqdm

from omni.data import EvalSet, RowsSet, ShardStream, collate, schedule_from_meta, to_device
from omni.model import OmniConfig, OmniNCM, omni_loss, omni_loss_stats
from omni.schedule import MODALITIES, OmniSchedule


def fast_math():
    """Turn on the fast paths that are actually available on this box.

    On CUDA the matmuls already go to cuBLAS and the attention to SDPA/flash - there is no
    separate library to call. What is NOT on by default: TF32 for fp32 matmuls (A100/H100/L40S
    only) and cuDNN benchmarking. Setting them is worth 10-40% on Ampere+ and costs nothing.
    A JAX port would re-buy the same cuBLAS kernels for a week of work and lose flex_attention,
    so the lever is kernels-per-step, not framework.
    """
    if not torch.cuda.is_available():
        return {}
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    torch.backends.cudnn.benchmark = True
    try:
        torch.set_float32_matmul_precision("high")
    except Exception:                                          # noqa: BLE001
        pass
    p = torch.cuda.get_device_properties(0)
    return dict(device=p.name, sm=f"{p.major}.{p.minor}", mem_gb=round(p.total_memory / 2**30, 1),
                tf32=True)


def adamw(params, cfg):
    """Fused AdamW when the build has it: one kernel over the whole parameter list instead of
    ~4 python-driven launches per tensor. At ~90 tensors that is a real slice of the step."""
    kw = dict(lr=cfg.get("lr", 3e-4), betas=(0.9, 0.95), weight_decay=cfg.get("wd", 0.01))
    if torch.cuda.is_available():
        try:
            return torch.optim.AdamW(params, fused=True, **kw)
        except (TypeError, RuntimeError, AssertionError) as e:      # pragma: no cover
            print(f"!! fused AdamW unavailable ({e}); using the eager optimiser")
    return torch.optim.AdamW(params, **kw)


def step_flops(P, bs, T, mult=1.0):
    """6*N*D per training step (2N forward + 4N backward) - the MFU convention, no sync."""
    return 6.0 * P * bs * T * mult


def warm_batch(sched, ocfg, B, device):
    """A batch that is legal for a heterogeneous head: every level's target is < that level's
    vocab. `synthetic_batch` draws from the widest codebook, which the 256-way L0 head rejects."""
    g = torch.Generator().manual_seed(0)
    codes, present = {}, {}
    for m in sched.modalities:
        flat = torch.zeros(B, sched.flat_len(m), dtype=torch.long)
        for k in range(sched.K):
            l, off = sched.level_len(m, k), sched.offsets(m)[k]
            flat[:, off:off + l] = torch.randint(1, ocfg.vocab_at(m, k), (B, l), generator=g)
        codes[m] = flat.to(device)
        present[m] = torch.ones(B, dtype=torch.bool, device=device)
    return {"codes": codes, "present": present}


def prepare(model, cfg, sched, bs, dev):
    """-> the callable to use for the forward (compiled when available, else the raw module).

    torch.compile fuses the ~10 small kernels per block (two RMSNorms, RoPE, the SiLU gate, the
    residual adds) and is the only way flex_attention runs as a fused kernel. It is compiled HERE
    on a synthetic batch, for three reasons: a graph break or an unsupported mask costs a warning
    instead of a silent 8x slowdown at step 0; the compile time lands before the timed loop so
    pos/s is not nonsense; and checkpoints are still saved from the raw module, so `_orig_mod.`
    never leaks into a state_dict that omni/eval.py has to load.
    """
    if not cfg.get("compile", True) or dev != "cuda":
        return model
    # A broken subgraph (older inductor builds mis-codegen flex_attention for tiny shapes) should
    # cost speed on that subgraph, not the whole run. suppress_errors falls back to eager per
    # frame; the try/except below still catches a failure at compile time.
    torch._dynamo.config.suppress_errors = True
    try:
        # `suppress_errors` falls back per frame but still logs a full traceback for every frame it
        # gives up on - on this config that is four frames and ~1,500 lines per run, which buries
        # the loss records the run exists to print. Keep the fallback, drop the narration.
        torch._logging.set_logs(dynamo=40)          # 40 = logging.ERROR
    except Exception:                                # noqa: BLE001 - never let logging break a run
        pass
    t0, amp = time.time(), torch.autocast("cuda", dtype=torch.bfloat16, enabled=True)
    try:
        fwd = torch.compile(model, dynamic=False)
        batch = warm_batch(sched, model.cfg, bs, dev)
        with amp:
            loss = omni_loss(fwd(batch), batch, sched)
        loss.backward()                                     # compiles the backward graph too
        model.zero_grad(set_to_none=True)
        model.eval()
        with torch.no_grad(), amp:
            fwd(batch)                                      # eval graph, same shapes
        model.train()
        g = torch._dynamo.utils.counters["stats"]
        print(f"torch.compile: ready in {time.time() - t0:.0f}s "
              f"({g.get('unique_graphs', 0)} graphs, {g.get('frames_compiled', 0)} frames "
              f"compiled, {g.get('frames_total', 0)} seen)")
        return fwd
    except Exception as e:                                      # noqa: BLE001
        print(f"!! torch.compile failed ({type(e).__name__}: {e}); training eager")
        model.zero_grad(set_to_none=True)
        model.train()
        return model


def build(cfg):
    sch = cfg["sched"]
    sched = OmniSchedule(base_len=sch["base_len"], K=sch["K"], R=sch["R"],
                         null_code=sch.get("null_code", 0),
                         modalities=tuple(sch.get("modalities") or MODALITIES))
    # `vocab` may be {m: int} (old), {m: [V_0..V_{K-1}]} (heterogeneous head) or {m: {k: V}}.
    # OmniConfig normalises it; passing the raw structure through keeps the cfgs diffable.
    vocab = cfg["vocab"]
    m = cfg["model"]
    ocfg = OmniConfig(sched, vocab, n_embd=m["n_embd"], n_layer=m["n_layer"],
                      n_head=m["n_head"], n_kv_head=m["n_kv_head"], dropout=m.get("dropout", 0.0),
                      qk_norm=m.get("qk_norm", True), mask_mode=m.get("mask_mode", "block"),
                      n_experts=m.get("n_experts", 0), expert_map=m.get("expert_map"),
                      moe_ffn=m.get("moe_ffn", 0), use_flex=m.get("use_flex"))
    return OmniNCM(ocfg), sched


def make_writer(out_dir):
    """TensorBoard over the same run dir. Returns None (with the fix printed) if unavailable."""
    try:
        from torch.utils.tensorboard import SummaryWriter
    except Exception:                                          # noqa: BLE001
        print("!! tensorboard is not installed: losses still go to metrics*.jsonl")
        print("   pip install -q tensorboard   then:  %load_ext tensorboard")
        print(f"   %tensorboard --logdir {os.path.join(out_dir, 'tb')}")
        return None
    d = os.path.join(out_dir, "tb")
    os.makedirs(d, exist_ok=True)
    print(f"tensorboard: %load_ext tensorboard; %tensorboard --logdir {d}")
    return SummaryWriter(d)


def preflight_gate(cfg, writer=None):
    """Runs test C before any training. Aborts unless it passed (override: preflight.allow_fail)."""
    pf = cfg.get("preflight") or {}
    if not pf.get("enabled", True):
        print("preflight: disabled in cfg (preflight.enabled=false)")
        return True
    from omni.preflight import run_preflight
    res = run_preflight(cfg, writer=writer)
    if not res["passed"] and not pf.get("allow_fail", False):
        print("\nPREFLIGHT FAILED - refusing to start training.")
        print("  the head/routing does not learn, so more steps cannot fix it.")
        print("  set preflight.allow_fail=true only to debug the trainer itself.")
        raise SystemExit(3)
    return bool(res["passed"])


def n_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


def flop_params(model):
    """Parameters a token actually flows through, which is what the TFLOPS column must divide by.

    With modality-routed experts `n_params` counts every expert while a synthetic batch only
    routes to a few, so using it inflates the number by the ratio of experts to active experts.
    """
    if getattr(model, "moe", False):
        from omni.moe import active_params
        return active_params(model)
    return n_params(model)


def lr_at(step, cfg):
    warm = max(1, int(cfg.get("warmup", 100)))
    peak = cfg.get("lr", 3e-4)
    if step < warm:
        return peak * (step + 1) / warm
    t = (step - warm) / max(1, cfg["steps"] - warm)
    return 0.05 * peak + 0.95 * peak * 0.5 * (1 + math.cos(math.pi * min(1.0, t)))


@torch.no_grad()
def evaluate(model, evals, sched, device, batch_size=8):
    """Per (modality, level) loss + teacher-forced accuracy, plus a scalar val loss per group."""
    model.eval()
    agg, tot_loss, tot_n = {}, 0.0, 0
    for group, ds in evals.items():
        for i in range(0, len(ds), batch_size):
            rows = [ds[j] for j in range(i, min(i + batch_size, len(ds)))]
            batch = to_device(collate(rows, sched), device)
            logits = model(batch)
            for k in range(sched.K):
                for m in sched.modalities:
                    l, off = sched.level_len(m, k), sched.offsets(m)[k]
                    tg = batch["codes"][m][:, off:off + l]
                    keep = (tg != sched.null_code) & batch["present"][m].unsqueeze(1)
                    if not keep.any():
                        continue
                    lg = logits[(m, k)]
                    ce = torch.nn.functional.cross_entropy(
                        lg.reshape(-1, lg.size(-1))[keep.reshape(-1)],
                        tg.reshape(-1)[keep.reshape(-1)], reduction="sum")
                    acc = (lg.argmax(-1) == tg)[keep].float().mean()
                    key = (m, k)
                    cur = agg.setdefault(key, {"ce": 0.0, "acc": 0.0, "n": 0, "groups": {}})
                    cur["ce"] += float(ce)
                    cur["acc"] += float(acc) * int(keep.sum())
                    cur["n"] += int(keep.sum())
                    g = cur["groups"].setdefault(group, {"acc": 0.0, "n": 0})
                    g["acc"] += float(acc) * int(keep.sum())
                    g["n"] += int(keep.sum())
    model.train()
    out = {}
    for (m, k), v in agg.items():
        out[f"{m}_L{k}"] = dict(ce=v["ce"] / max(1, v["n"]), acc=v["acc"] / max(1, v["n"]),
                                n=v["n"],
                                by_group={g: d["acc"] / max(1, d["n"]) for g, d in v["groups"].items()})
    return out


def run(cfg, smoke=None, feed=None, eval_rows=None):
    """Single training arm.

    `feed` is an iterator of encoded rows and replaces the shard stream entirely: the notebook
    trains straight off the live feed, so there is no produce step, no run/stream, no DONE marker
    and nothing on disk to go stale. `eval_rows` is a fixed in-memory holdout taken from that same
    feed, which keeps validation loss comparable across a run.
    """
    dev = "cuda" if torch.cuda.is_available() else "cpu"
    out_dir = cfg["out_dir"]
    os.makedirs(out_dir, exist_ok=True)
    writer = make_writer(out_dir)
    preflight_gate(cfg, writer)
    model, sched = build(cfg)
    model.to(dev).train()
    P = n_params(model)
    Pf = flop_params(model)
    T = sched.total_len
    bs = cfg.get("batch", 4)
    if dev == "cuda":
        print("fast math:", fast_math())
    print(f"params {P/1e6:.2f}M | packed positions/sample {T} | modalities {list(sched.modalities)}")
    print(sched.describe())

    evals = {}
    for group, path in (cfg.get("eval_files") or {}).items():
        if os.path.exists(path):
            evals[group] = EvalSet(path)
    if eval_rows:
        evals["direct"] = RowsSet(eval_rows)
    print("eval sets:", {g: len(d) for g, d in evals.items()} or "none - no validation")
    if feed is not None:
        it = iter(feed)
        print(f"data: direct feed | no shards | held-out rows {len(eval_rows or [])}")
    else:
        it = iter(ShardStream(cfg["shard_dir"], consume=cfg.get("consume", False),
                              start_sample=cfg.get("start_sample", 0),
                              producer_log=cfg.get("producer_log"),
                              timeout_s=cfg.get("stream_timeout", 600)))
    opt = adamw(model.parameters(), cfg)
    steps = smoke or cfg["steps"]
    ckpt_dir = os.path.join(out_dir, "ckpt")
    os.makedirs(ckpt_dir, exist_ok=True)
    metrics = open(os.path.join(out_dir, "metrics.jsonl"), "a")
    use_bf16 = dev == "cuda"
    fwd = prepare(model, cfg, sched, bs, dev)

    t0, toks = time.time(), 0

    # Every timestamp below is taken with the GPU idle: without the sync, t_fwd/t_bwd would measure
    # kernel launch, and a feed-bound run would report a fake ~99% "gpu" share. Six syncs a step is
    # noise next to a 0.5-2 s step.
    _sync = torch.cuda.synchronize if dev == "cuda" else (lambda: None)
    pbar = tqdm(range(steps), desc="train", unit="step", disable=not sys.stderr.isatty())
    for step in pbar:
        _sync()
        _t0 = time.time()
        rows = []
        while len(rows) < bs:
            try:
                rows.append(next(it))
            except StopIteration:
                print("stream ended before the step budget: stopping")
                steps = step
                break
        _sync()
        _t1 = time.time()
        if len(rows) < bs:
            break
        batch = to_device(collate(rows, sched), dev)
        _sync()
        _t2 = time.time()
        for g in opt.param_groups:
            g["lr"] = lr_at(step, {**cfg, "steps": steps})
        with torch.autocast("cuda", dtype=torch.bfloat16, enabled=use_bf16):
            logits = fwd(batch)
            loss = omni_loss(logits, batch, sched, label_smoothing=cfg.get("label_smoothing", 0.0))
        _sync()
        _t3 = time.time()
        opt.zero_grad(set_to_none=True)
        loss.backward()
        gn = torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.get("clip", 1.0))
        _sync()
        _t4 = time.time()
        opt.step()
        _sync()
        _t5 = time.time()
        toks += bs * T
        el = time.time() - t0
        rec = dict(step=step, loss=float(loss.detach()), grad_norm=float(gn.detach()),
                   lr=opt.param_groups[0]["lr"],
                   samples_per_s=round(toks / T / el, 2),   # toks/T is samples, not steps
                   steps_per_s=round((step + 1) / el, 2),
                   pos_per_s=round(toks / el, 1),
                   tflops=round(step_flops(Pf, bs, T) * (step + 1) / el / 1e12, 2),
                   # Per-step stage split. With a feed near 2 samples/s and 462M active params the
                   # question is never "how fast" but "waiting on rows or waiting on the GPU", and
                   # data_pct answers it without arithmetic on the reader's side.
                   samples=toks // T,        # what the stream actually delivered, not the plan
                   t_fetch=round(_t1 - _t0, 3), t_prep=round(_t2 - _t1, 3),
                   t_fwd=round(_t3 - _t2, 3), t_bwd=round(_t4 - _t3, 3),
                   t_opt=round(_t5 - _t4, 3),
                   data_pct=round(100 * (_t2 - _t0) / max(1e-9, _t5 - _t0), 1),
                   gpu_pct=round(100 * (_t5 - _t2) / max(1e-9, _t5 - _t0), 1))
        logging = step % max(1, cfg.get("log_every", 10)) == 0 or step == steps - 1
        if logging:
            # One loss-stats pass per logged step, computed whether or not TensorBoard is up:
            # metrics.jsonl has to carry the per-(modality, level) CE, because the notebook's
            # learning check reads plain JSON. Until now that breakdown existed only as tb event
            # files, so the only way to see "is the audio expert learning" was to install diskwriter.
            with torch.no_grad():
                _, st = omni_loss_stats(logits, batch, sched)
            rec["ce"] = {f"{mm}_L{kk}": round(float(v["ce"]), 4)
                         for (mm, kk), v in st.items()}
            if writer is not None:
                writer.add_scalar("train/loss", float(loss.detach()), step)
                writer.add_scalar("train/grad_norm", float(gn), step)
                writer.add_scalar("train/lr", rec["lr"], step)
                writer.add_scalar("data/pos_per_s", rec["pos_per_s"], step)
                writer.add_scalar("perf/tflops", rec["tflops"], step)
                writer.add_scalar("trace/data_pct", rec["data_pct"], step)
                writer.add_scalar("trace/gpu_pct", rec["gpu_pct"], step)
                writer.add_scalar("trace/t_fetch", rec["t_fetch"], step)
                writer.add_scalar("trace/t_prep", rec["t_prep"], step)
                writer.add_scalar("trace/t_fwd", rec["t_fwd"], step)
                writer.add_scalar("trace/t_bwd", rec["t_bwd"], step)
                for (mm, kk), v in st.items():
                    writer.add_scalar(f"train/ce/{mm}_L{kk}", v["ce"], step)
        pbar.set_postfix(loss=f"{rec['loss']:.4f}", pos_s=rec["pos_per_s"],
                         tflops=rec["tflops"], data=f"{rec['data_pct']:.0f}%")
        if logging:
            metrics.write(json.dumps(rec) + "\n")
            metrics.flush()
            pbar.write(json.dumps(rec))
        if evals and cfg.get("eval_every") and (step + 1) % cfg["eval_every"] == 0:
            ev = evaluate(fwd, evals, sched, dev, bs)
            metrics.write(json.dumps(dict(step=step, val=ev)) + "\n")
            metrics.flush()
            fin = {k: round(v["acc"], 4) for k, v in ev.items()}
            print("val acc:", fin)
        if cfg.get("save_every") and (step + 1) % cfg["save_every"] == 0:
            torch.save({"model": model.state_dict(), "cfg": cfg, "step": step},
                       os.path.join(ckpt_dir, f"step{step+1}.pt"))
    dt = time.time() - t0
    if smoke:
        sps = dt / max(1, steps)
        print(f"SMOKE {steps} steps | {sps:.2f} s/step | "
              f"{toks/dt:.0f} pos/s | {step_flops(Pf, bs, T) / sps / 1e12:.1f} TFLOPS | "
              f"projected {sps*cfg['steps']/3600:.2f} h for {cfg['steps']} steps")
        print(f"cost at $2.13/h on-demand H100: ${sps*cfg['steps']/3600*2.13:.0f}  "
              f"(at $1.00/h spot: ${sps*cfg['steps']/3600*1.0:.0f})")
        return
    torch.save({"model": model.state_dict(), "cfg": cfg, "step": steps - 1},
               os.path.join(out_dir, "final.pt"))
    print("saved", os.path.join(out_dir, "final.pt"))


def run_duplex(cfg, smoke=None):
    """Both arms, one stream, one pass. The batch is built once and stepped by every arm, so the
    A/B cannot drift: same rows, same order, same packing. Costs one model's memory per arm and
    replaces produce-then-train-two-times."""
    import copy

    from omni.livestream import iter_rows

    dev = "cuda" if torch.cuda.is_available() else "cpu"
    out_dir = cfg["out_dir"]
    os.makedirs(out_dir, exist_ok=True)
    writer = make_writer(out_dir)
    preflight_gate(cfg, writer)
    arms = cfg.get("arms") or ["strict", "ar"]
    steps = smoke or cfg["steps"]
    bs = cfg.get("batch", 4)
    use_bf16 = dev == "cuda"
    if dev == "cuda":
        print("fast math:", fast_math())

    models, opts, fwds, sched, arm_cfg = {}, {}, {}, None, {}
    for arm in arms:
        c = copy.deepcopy(cfg)
        c["model"]["mask_mode"] = arm
        # The checkpoint must carry ITS arm's cfg: eval names an arm from mask_mode, and the
        # shared run cfg says "block" (the default), which relabels both arms as the legacy
        # leaky arm and lets the second one silently overwrite the first.
        arm_cfg[arm] = c
        m, s = build(c)
        m.to(dev).train()
        models[arm] = m
        opts[arm] = adamw(m.parameters(), cfg)
        sched = sched or s
        print(f"{arm:6s} params {n_params(m)/1e6:.2f}M")
    T = sched.total_len
    for arm in arms:                      # after sched exists: prepare() needs the packed length
        fwds[arm] = prepare(models[arm], cfg, sched, bs, dev)
    per_step = sum(step_flops(flop_params(models[a]), bs, T) for a in arms)
    print(f"packed positions/sample {T} | arms {arms} | single pass over the live stream")

    evals = {}
    for group, path in (cfg.get("eval_files") or {}).items():
        if os.path.exists(path):
            evals[group] = EvalSet(path)
    print("eval sets:", {g: len(d) for g, d in evals.items()} or "none yet")

    live_cfg = cfg.get("live_cfg")
    from omni.livestream import iter_rows
    if live_cfg:
        rows = iter_rows(live_cfg, holdout=live_cfg.get("holdout"),
                         eval_dir=live_cfg.get("out_dir"))
        print("data: live Hub stream (one pass, ~1.5 samples/s measured - a kill test only)")
    else:
        rows = iter(ShardStream(cfg["shard_dir"], consume=cfg.get("consume", False),
                                start_sample=cfg.get("start_sample", 0),
                                producer_log=cfg.get("producer_log"),
                                timeout_s=cfg.get("stream_timeout", 600)))
        print(f"data: shards in {cfg['shard_dir']} (both arms read the same shards)")
    metrics = {a: open(os.path.join(out_dir, f"metrics_{a}.jsonl"), "a") for a in arms}
    it = iter(rows)
    t0, toks = time.time(), 0
    pbar = tqdm(range(steps), desc=f"duplex {arms}", unit="step",
              disable=not sys.stderr.isatty())
    for step in pbar:
        batch_rows = []
        while len(batch_rows) < bs:
            try:
                batch_rows.append(next(it))
            except StopIteration:
                print("stream ended: stopping")
                steps = step
                break
        if len(batch_rows) < bs:
            break
        batch = to_device(collate(batch_rows, sched), dev)
        rec: dict = {"step": step}
        for arm in arms:
            for g in opts[arm].param_groups:
                g["lr"] = lr_at(step, {**cfg, "steps": steps})
            with torch.autocast("cuda", dtype=torch.bfloat16, enabled=use_bf16):
                logits = fwds[arm](batch)
                loss = omni_loss(logits, batch, sched,
                                 label_smoothing=cfg.get("label_smoothing", 0.0))
            opts[arm].zero_grad(set_to_none=True)
            loss.backward()
            gn = torch.nn.utils.clip_grad_norm_(models[arm].parameters(), cfg.get("clip", 1.0))
            opts[arm].step()
            rec[arm] = round(float(loss.detach()), 5)
            rec[f"{arm}_gn"] = round(float(gn.detach()), 3)
            if writer is not None and (step % max(1, cfg.get("log_every", 10)) == 0
                                       or step == steps - 1):
                with torch.no_grad():
                    _, st = omni_loss_stats(logits, batch, sched)
                writer.add_scalar(f"{arm}/loss", float(loss.detach()), step)
                writer.add_scalar(f"{arm}/grad_norm", float(gn), step)
                for (mm, kk), v in st.items():
                    writer.add_scalar(f"{arm}/ce/{mm}_L{kk}", v["ce"], step)
        toks += bs * T
        rec["pos_per_s"] = round(toks / (time.time() - t0), 1)
        rec["tflops"] = round(per_step * (step + 1) / (time.time() - t0) / 1e12, 2)
        if writer is not None:
            writer.add_scalar("data/pos_per_s", rec["pos_per_s"], step)
            writer.add_scalar("perf/tflops", rec["tflops"], step)
        pbar.set_postfix({a: f"{rec[a]:.4f}" for a in arms},
                         pos_s=rec["pos_per_s"], tflops=rec["tflops"])
        if step % max(1, cfg.get("log_every", 10)) == 0 or step == steps - 1:
            pbar.write(json.dumps(rec))
            for arm in arms:
                metrics[arm].write(json.dumps(rec) + "\n")
                metrics[arm].flush()
        if evals and cfg.get("eval_every") and (step + 1) % cfg["eval_every"] == 0:
            for arm in arms:
                # evaluate() returns {m_Lk: {ce, acc, n}}; val loss is the token-weighted CE.
                ev = evaluate(models[arm], evals, sched, dev, bs)
                tot_n = sum(v["n"] for v in ev.values())
                vl = sum(v["ce"] * v["n"] for v in ev.values()) / max(1, tot_n)
                pbar.write(f"  val {arm} @{step+1}: loss {vl:.4f}")
                for key, v in ev.items():
                    metrics[arm].write(json.dumps(dict(step=step, val=True, level=key,
                                                       loss=round(v["ce"], 5),
                                                       acc=round(v["acc"], 5),
                                                       n=v["n"])) + "\n")
                metrics[arm].write(json.dumps(dict(step=step, val=True, loss=round(vl, 5))) + "\n")
                metrics[arm].flush()
                if writer is not None:
                    writer.add_scalar(f"{arm}/val_loss", vl, step)
                    for key, v in ev.items():
                        writer.add_scalar(f"{arm}/val_{key}", v["ce"], step)
        if cfg.get("save_every") and (step + 1) % cfg["save_every"] == 0:
            for arm in arms:
                d = os.path.join(out_dir, f"train_{arm}")
                os.makedirs(d, exist_ok=True)
                torch.save({"model": models[arm].state_dict(), "cfg": arm_cfg[arm], "step": step},
                           os.path.join(d, f"step{step+1}.pt"))

    # The live-stream generator holds 20 open Hub readers and daemon pump threads. Never closing
    # it leaves the interpreter hanging after the last step (process alive, cell never returns),
    # with the streams still attached. Closing runs its finally and stops the pumps.
    close = getattr(rows, "close", None)
    if close:
        try:
            close()
        except Exception:                                          # noqa: BLE001
            pass

    dt = time.time() - t0
    for arm in arms:
        d = os.path.join(out_dir, f"train_{arm}")
        os.makedirs(d, exist_ok=True)
        torch.save({"model": models[arm].state_dict(), "cfg": arm_cfg[arm], "step": steps - 1},
                   os.path.join(d, "final.pt"))
        print("saved", os.path.join(d, "final.pt"))
    print(f"DUPLEX {steps} steps | {dt/max(1, steps):.2f} s/step for {len(arms)} arms "
          f"| {toks/dt:.0f} pos/s | {steps*bs:,} samples seen once")
    if live_cfg:
        have = sorted(f for f in os.listdir(live_cfg["out_dir"]) if f.startswith("eval_"))
        print("eval blocks:", have or "NONE")
    # Colab: closing the generator stops the pump threads, but the Hub readers and hf_xet keep
    # file descriptors open and the interpreter never returns, so the cell sits RUNNING after
    # "DUPLEX ... steps" with every checkpoint already on disk. This must come after the
    # save flush and before any early return (--smoke used to return here and hang the same
    # way). Everything is on disk by now; leave hard rather than hang a finished run.
    if writer is not None:
        writer.flush()
        writer.close()
    if cfg.get("hard_exit", True):
        sys.stdout.flush()
        sys.stderr.flush()
        os._exit(0)


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("cfg")
    ap.add_argument("--smoke", type=int, default=0)
    a = ap.parse_args()
    cfg = json.load(open(a.cfg))
    if cfg.get("arms"):
        run_duplex(cfg, smoke=a.smoke or None)
    else:
        run(cfg, smoke=a.smoke or None)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile .ncm_stage/omni/eval.py
"""Evaluation: the only number that can carry the thesis.

The claim is not "lower perplexity" - it is "the same quality at fewer function evaluations".
So the headline table is quality against NFE, computed identically for both arms:

  level-parallel (ours)  NFE = number of levels generated K'
  autoregressive         NFE = number of tokens generated

Teacher-free greedy sampling, scored against the frozen codec's codes for the held-out samples
(exact-match per level). Everything is reported per level, because a coarse-level win with a
flat fine level is the failure mode this whole design is meant to avoid.
"""
import argparse
import json
import os

import torch

from omni.data import EvalSet, collate, to_device
from omni.model import OmniNCM, OmniConfig
from omni.train import build


def load_model(path, dev="cuda"):
    ck = torch.load(path, map_location="cpu", weights_only=False)
    model, sched = build(ck["cfg"])
    model.load_state_dict(ck["model"])
    return model.to(dev).eval(), sched, ck


@torch.no_grad()
def score_codes(model, sched, batch, level_cutoff=None, temp=0.0):
    """Greedy level-parallel sample; returns per-(m,k) exact-match accuracy."""
    # Auto-select the attention path. At base_len=64 eval is 48 sequences of 504 positions, where
    # dense SDPA is fine; at base_len=1024 the packed sequence is ~7.1k, and a dense T x T mask
    # (50M floats) pushes SDPA onto a path that materialises B*H*T*T scores -> OOM. encode() gates
    # flex on sm_80+, so a T4 still gets dense and an A100/3070 gets the fused kernel.
    gen = model.generate(batch, level_cutoff=level_cutoff, temperature=temp,
                         use_flex=None)
    K = sched.K if level_cutoff is None else level_cutoff
    acc = {}
    for k in range(K):
        for m in sched.modalities:
            l, off = sched.level_len(m, k), sched.offsets(m)[k]
            gt = batch["codes"][m][:, off:off + l]
            keep = (gt != sched.null_code) & batch["present"][m].unsqueeze(1)
            if not keep.any():
                continue
            acc[(m, k)] = float((gen[m][:, off:off + l] == gt)[keep].float().mean())
    return acc


@torch.no_grad()
def ar_scores(model, sched, batch, token_budgets):
    """Greedy AR sample up to the largest budget; score only the positions that budget covers."""
    budgets = sorted(set(int(b) for b in token_budgets))
    gen, _ = model.generate_ar(batch, max_len=budgets[-1], use_flex=None)
    # raster order index of every (modality, level, position) is what "token budget" means
    order = [(k, m, i) for k in range(sched.K) for m in sched.modalities
             for i in range(sched.level_len(m, k))]
    pos_of = {t: idx for idx, t in enumerate(order)}
    out = {}
    for b in budgets:
        acc = {}
        for k in range(sched.K):
            for m in sched.modalities:
                l, off = sched.level_len(m, k), sched.offsets(m)[k]
                gt = batch["codes"][m][:, off:off + l]
                present = batch["present"][m].unsqueeze(1)
                inside = torch.tensor([pos_of[(k, m, i)] < b for i in range(l)],
                                      dtype=torch.bool, device=gt.device).unsqueeze(0)
                keep = (gt != sched.null_code) & present & inside
                if not keep.any():
                    continue
                acc[(m, k)] = float((gen[m][:, off:off + l] == gt)[keep].float().mean())
        out[b] = acc
    return out


@torch.no_grad()
def cond_scores(model, sched, batch):
    """Per-(m,k) accuracy with the TRUE coarser levels given - one forward, no sampling.

    This is the metric that survives level 0 being unpredictable. Free-running generation needs
    a correct level 0 to condition everything else on, and level 0 is a 1-of-256 guess from
    nothing (val text_L0 ~ 0.01), so the free-running number is dominated by that first coin
    flip. Conditional accuracy asks the question the architecture actually answers: given the
    coarser levels, how much of level k do you reconstruct in ONE pass?
    """
    logits = model(batch, use_flex=None)
    acc = {}
    for k in range(sched.K):
        for m in sched.modalities:
            l, off = sched.level_len(m, k), sched.offsets(m)[k]
            gt = batch["codes"][m][:, off:off + l]
            keep = (gt != sched.null_code) & batch["present"][m].unsqueeze(1)
            if not keep.any():
                continue
            pred = logits[(m, k)].argmax(-1)
            acc[(m, k)] = float((pred == gt)[keep].float().mean())
    return acc


@torch.no_grad()
def ar_cond_parity(model, sched, batch):
    """AR scored with the SAME information the parallel arm has: the level's first slot only.

    At slot 0 of level k a causal model has seen every coarser level and nothing of level k -
    exactly the parallel arm's information set. At slot i>0 it also reads the i same-level slots
    before it, which is context the parallel arm is architecturally denied: text L5 is 64 slots,
    so a naive conditional comparison hands AR a 64-token language-modelling prefix. Both numbers
    are reported; this one is the one that compares architectures.
    """
    logits = model(batch, use_flex=None)
    acc = {}
    for k in range(sched.K):
        for m in sched.modalities:
            l, off = sched.level_len(m, k), sched.offsets(m)[k]
            if l < 1:
                continue
            gt = batch["codes"][m][:, off:off + 1]
            keep = (gt != sched.null_code) & batch["present"][m].unsqueeze(1)
            if not keep.any():
                continue
            pred = logits[(m, k)][:, :1].argmax(-1)
            acc[(m, k)] = float((pred == gt)[keep].float().mean())
    return acc


def copy_prior(batch, sched):
    """Accuracy of a pure copy model: repeat the previous slot in the same level.

    This is the floor that matters for AR, because a causal model reads that previous slot for
    free. A level sitting at its copy share was scored by repetition, not by using the coarser
    levels - which is what a 0.95 on text L5 has to be checked against before it is quoted.
    """
    out = {}
    for k in range(sched.K):
        for m in sched.modalities:
            l, off = sched.level_len(m, k), sched.offsets(m)[k]
            if l < 2:
                continue
            gt = batch["codes"][m][:, off:off + l]
            keep = (gt != sched.null_code) & batch["present"][m].unsqueeze(1)
            nxt, kp = gt[:, 1:], keep[:, 1:]
            if not kp.any():
                continue
            out[(m, k)] = float(((nxt == gt[:, :-1]) & kp).sum()) / int(kp.sum())
    return out


def summarise(acc, sched, n_levels):
    """Mean accuracy over generated levels + the coarsest and the finest generated level."""
    if not acc:
        return dict(n_feval=0, mean=0.0, coarse=0.0, finest=0.0)
    vals = list(acc.values())
    coarse = [v for (m, k), v in acc.items() if k == 0]
    fine = [v for (m, k), v in acc.items() if k == n_levels - 1]
    return dict(n_feval=len(vals), mean=sum(vals) / len(vals),
                coarse=sum(coarse) / max(1, len(coarse)),
                finest=sum(fine) / max(1, len(fine)))


ARMS = (("level-parallel", "block_ckpt"),            # legacy: sees its own level, do not quote
        ("level-parallel(strict)", "strict_ckpt"),   # the honest parallel arm
        ("autoregressive", "ar_ckpt"))


def run(cfg):
    dev = "cuda" if torch.cuda.is_available() else "cpu"
    # Any subset of checkpoints may exist: eval on whatever has been trained, and say what is
    # missing. A missing arm is a message, not a crash - the point is never to be blocked from
    # reading the arm you do have.
    loaded, sched, ck = {}, None, None
    for name, key in ARMS:
        p = cfg.get(key)
        if not (p and os.path.exists(p)):
            continue
        mdl, sch, meta = load_model(p, dev)
        if sched is None:
            sched, ck = sch, meta
        else:
            assert sch.total_len == sched.total_len, "arms must share the schedule"
        # Name the arm from the checkpoint's OWN mask_mode, not from which file it sat in. A
        # pre-shift `block` run parked in run/train_block is the legacy arm, and must never be
        # reported as "level-parallel".
        # A checkpoint written by the duplex trainer carries the SHARED run cfg, whose
        # mask_mode is the default ("block") for both arms - so the recorded label can lie.
        # The directory it was written to is the trainer's own naming and does not lie.
        mm = (meta.get("cfg", {}).get("model", {}) or {}).get("mask_mode", "?")
        arm = key.split("_")[0]                      # block_ckpt -> block, ar_ckpt -> ar
        if mm != "?" and mm != arm:
            print(f"  !! {p} records mask_mode={mm!r} but lives in train_{arm}/ "
                  f"- trusting the directory (a pre-fix duplex trainer saved the shared cfg)")
            mm = arm
        if mm == "strict":
            name = "level-parallel(strict)"
        elif mm == "ar":
            name = "autoregressive"
        elif mm == "block":
            name = "level-parallel(LEGACY-leaky)"
        else:
            name = f"unknown({mm})"
        print(f"  {p}  mask_mode={mm}  -> {name}")
        loaded[name] = mdl
    assert loaded, ("no checkpoints found - train at least one arm "
                    "(`run_omni.py train --arm strict`)")
    print("arms found:", ", ".join(loaded))
    block = loaded.get("level-parallel")
    strict = loaded.get("level-parallel(strict)")
    ar = loaded.get("autoregressive")

    ev = cfg.get("eval_files") or []
    if isinstance(ev, dict):                       # train cfgs store {group: path}
        ev = list(ev.values())
    files = [f for f in ev if os.path.exists(f)]
    # The eval cfg stores a SNAPSHOT of the eval_*.npz files that existed when it was written.
    # A `check` that runs before the audio/video carve finishes therefore bakes in a two-modality
    # eval, and those columns read "-" forever with no error. Re-glob beside the files we did find
    # (or beside the shards) so a stale list heals itself.
    import glob
    dirs = sorted({os.path.dirname(f) for f in files}) or [ck["cfg"].get("shard_dir", "")]
    for d in dirs:
        for f in sorted(glob.glob(os.path.join(d, "eval_*.npz"))):
            if f not in files:
                files.append(f)
    assert files, "no eval shards found: let the producer write eval_*.npz first"
    rows = []
    for f in files:
        ds_f = EvalSet(f)
        rows += [ds_f[i] for i in range(len(ds_f))]
    if cfg.get("n_samples") and cfg["n_samples"] < len(rows):
        # Round-robin over modalities. A plain `rows[:n]` silently deleted the video group,
        # because the files are read audio, image, text, video and video came last - so "video"
        # columns read as "-" and the smallest modality vanished from every table.
        by = {}
        for r in rows:                             # rows are {"codes", "present"}; key on modality
            m = next((x for x in sched.modalities if r["present"].get(x)), "?")
            by.setdefault(m, []).append(r)
        order = list(by)
        keep, i = [], 0
        while len(keep) < cfg["n_samples"] and any(by[m] for m in order):
            m = order[i % len(order)]
            if by[m]:
                keep.append(by[m].pop(0))
            i += 1
        rows = keep
    batch = to_device(collate(rows, sched), dev)
    print(f"eval on {len(rows)} held-out samples from {len(files)} group file(s) | "
          f"packed positions {sched.total_len}")

    # A constant predictor (always emit the most frequent code of each slot) is the floor any
    # real score has to clear: without it a 0.12 looks like signal when it is below the trivial
    # solution. It must come from the TRAINING distribution - estimated on 48 held-out rows it
    # reads ~0.62, because with 48 rows the modal code of a slot is "most of the slot".
    from collections import Counter
    import glob
    import numpy as np
    counts = {(m, k): Counter() for m in sched.modalities for k in range(sched.K)}
    src = "training shards"
    shards = sorted(glob.glob(os.path.join(ck["cfg"].get("shard_dir", ""), "shard_*.npz")))[:40]
    if shards:
        for f in shards:
            z = np.load(f, allow_pickle=False)
            for m in sched.modalities:
                if m not in z:
                    continue
                # absent modalities are all-NULL padding; counting them makes the baseline the
                # NULL rate, not the code distribution
                keep = z[f"{m}_present"].astype(bool) if f"{m}_present" in z else np.ones(len(z[m]), bool)
                if not keep.any():
                    continue
                for k in range(sched.K):
                    off, l = sched.offsets(m)[k], sched.level_len(m, k)
                    counts[(m, k)].update(z[m][keep][:, off:off + l].reshape(-1).tolist())
    else:                                             # shards gone: fall back, and say so
        # --live writes no shards, so this is the NORMAL path for a duplex run, not an
        # emergency one. Absent modalities are all-NULL padding after collate; counting them
        # makes every level's floor the NULL rate (they all read exactly 0.6667) instead of
        # the code distribution, and the model then looks broken against a made-up bar.
        src = "held-out rows (no shards on disk - --live stream, present-masked)"
        for m in sched.modalities:
            keep_b = batch["present"][m].cpu().numpy().astype(bool)
            for k in range(sched.K):
                off, l = sched.offsets(m)[k], sched.level_len(m, k)
                slot = batch["codes"][m][keep_b][:, off:off + l]
                counts[(m, k)].update(slot.reshape(-1).tolist())
    freq = {key: c.most_common(1)[0][1] / max(sum(c.values()), 1)
            for key, c in counts.items() if c}
    baseline = sum(freq.values()) / max(len(freq), 1)
    # per level, so the conditional table is compared against the right floor at each level
    base_lv = {k: (sum(v for (m, kk), v in freq.items() if kk == k)
                   / max(1, sum(1 for (m, kk) in freq if kk == k)))
               for k in range(sched.K)}
    print(f"constant-predictor baseline from {src} = {baseline:.4f}")

    table = []
    for name, model in loaded.items():               # whatever was actually loaded, labelled
        if "autoregressive" in name:
            continue
        for kp in range(1, sched.K + 1):
            acc = score_codes(model, sched, batch, level_cutoff=kp)
            s = summarise(acc, sched, kp)
            s.update(arm=name, nfe=kp, levels=kp)
            table.append(s)

    if ar is not None:
        T = sum(sched.flat_len(m) for m in sched.modalities)
        # AR costs one forward per token, so the full-T point is deliberately skipped: it is
        # ~T forwards of a growing prefix and would dominate the eval wall clock. T/2 already
        # shows the plateau, and the ratio to K is what the table is for.
        budgets = [max(1, T // 16), max(1, T // 8), max(1, T // 4), max(1, T // 2)]
        accs = ar_scores(ar, sched, batch, budgets)
        for b, acc in accs.items():
            s = summarise(acc, sched, sched.K)
            s.update(arm="autoregressive", nfe=b, levels=None)
            table.append(s)
    if strict is None:
        print("no strict checkpoint: the honest parallel arm has not been trained yet "
              "(`run_omni.py train --arm strict`)")
    if ar is None:
        print("no AR checkpoint: the comparison table will be one-sided "
              "(train one with model.mask_mode='ar' on the same shards)")

    cond = {}
    for name, model in loaded.items():
        cond[name] = cond_scores(model, sched, batch)
    print("\n### conditional: true coarser levels given, one forward per level (no sampling)\n")
    print("| arm | level | text | image | audio | video | mean | constant |")
    print("|---|---|---|---|---|---|---|---|")
    for name, acc in cond.items():
        for k in range(sched.K):
            cells = [acc.get((m, k)) for m in sched.modalities]
            got = [c for c in cells if c is not None]
            row = " | ".join("  -  " if c is None else f"{c:.3f}" for c in cells)
            print(f"| {name} | L{k} | {row} | {sum(got)/max(1,len(got)):.4f} | "
                  f"{base_lv[k]:.3f} |")
    print()
    par = ar_cond_parity(ar, sched, batch) if ar is not None else {}
    if par:
        print("### AR at parity: level-first slot only, no same-level prefix "
              "(the parallel arm's information)\n")
        print("| arm | level | text | image | audio | video | mean |")
        print("|---|---|---|---|---|---|---|")
        for k in range(sched.K):
            cells = [par.get((m, k)) for m in sched.modalities]
            got = [c for c in cells if c is not None]
            row = " | ".join("  -  " if c is None else f"{c:.3f}" for c in cells)
            print(f"| autoregressive(parity) | L{k} | {row} | {sum(got)/max(1, len(got)):.4f} |")
        print()

    cp = copy_prior(batch, sched)
    if cp:
        print("### copy prior: accuracy of repeating the previous slot in the same level "
              "(what AR reads for free)\n")
        print("| level | text | image | audio | video | mean |")
        print("|---|---|---|---|---|---|")
        for k in range(sched.K):
            cells = [cp.get((m, k)) for m in sched.modalities]
            got = [c for c in cells if c is not None]
            if not got:
                continue
            row = " | ".join("  -  " if c is None else f"{c:.3f}" for c in cells)
            print(f"| L{k} | {row} | {sum(got)/len(got):.4f} |")
        print()

    print("### free-running: sample from level 0, no teacher (the unconditional number)\n")
    print("| arm | NFE | mean acc | coarse | finest |")
    print("|---|---|---|---|---|")
    for r in sorted(table, key=lambda r: (r["arm"], r["nfe"])):
        print(f"| {r['arm']} | {r['nfe']} | {r['mean']:.4f} | {r['coarse']:.4f} | {r['finest']:.4f} |")

    # ---------------------------------------------------------------- verdict
    # Two numbers, and they answer different questions:
    #   conditional  given the true coarser levels, how much of level k comes back in ONE pass
    #   free-running sample from level 0 - the number a paper would quote, and the one that dies
    #                here because level 0 is a 1-of-256 guess from nothing (val text_L0 ~ 0.01),
    #                so every later level is conditioned on a wrong prefix
    mine = ("level-parallel(strict)" if "level-parallel(strict)" in loaded
            else next((n for n in loaded if "level-parallel" in n), "level-parallel"))
    fine = sched.K - 1
    my_cond = sum(cond[mine].get((m, fine), 0.0) for m in sched.modalities
                  if (m, fine) in cond[mine])
    my_cond /= max(1, sum(1 for m in sched.modalities if (m, fine) in cond[mine]))
    ar_cond = None
    if "autoregressive" in cond:
        ar_cond = sum(cond["autoregressive"].get((m, fine), 0.0) for m in sched.modalities
                      if (m, fine) in cond["autoregressive"])
        ar_cond /= max(1, sum(1 for m in sched.modalities if (m, fine) in cond["autoregressive"]))

    ar_par = None
    if par:
        ar_par = sum(par.get((m, fine), 0.0) for m in sched.modalities if (m, fine) in par)
        ar_par /= max(1, sum(1 for m in sched.modalities if (m, fine) in par))
    cp_fine = [v for (m, kk), v in cp.items() if kk == fine]
    cp_mean = sum(cp_fine) / len(cp_fine) if cp_fine else None

    free_best = max([0.0] + [r["mean"] for r in table])
    lines = [f"conditional L{fine} ({'one forward' if True else ''}): {mine} {my_cond:.4f}"
             + (f" vs AR {ar_cond:.4f} ({sched.K} forwards vs {sum(sched.flat_len(m) for m in sched.modalities)})"
                if ar_cond is not None else "")
             + f" | constant {base_lv[fine]:.4f}"]
    if ar_par is not None:
        lines.append(f"same metric, AR at parity (level-first slot, no same-level prefix): {ar_par:.4f}")
    if cp_mean is not None:
        lines.append(f"copy prior at L{fine} (repeat the previous slot): {cp_mean:.4f}")
    if my_cond <= base_lv[fine]:
        lines.append(f"VOID: the conditional finest level is at or below the constant predictor "
                     f"({base_lv[fine]:.4f}) - the model has not learned to use the coarser levels.")
    elif ar_par is not None:
        # The naive comparison hands AR the same-level slots before the one it is predicting.
        # At parity that context is removed, so this is the number that ranks architectures.
        d = my_cond - ar_par
        if d >= 0:
            lines.append(f"PARITY: at L{fine} the parallel arm is {d:+.4f} over AR once AR loses "
                         f"its same-level prefix - the naive AR lead is context, not architecture.")
        else:
            lines.append(f"PARITY: AR is {-d:.4f} ahead at L{fine} even without same-level context.")
    margin = free_best - baseline
    if free_best <= baseline:
        tail = (" - VOID, and expected: level 0 is unpredictable, so this number measures "
                "the propagation of the first wrong guess, not the architecture.")
    elif margin < 0.05:
        tail = (f" - margin {margin:+.4f} on a 48-row set is inside the sampling noise "
                f"(SE ~0.05 per slot): a tie, not a win. Do not quote it.")
    else:
        tail = f" - margin {margin:+.4f}, outside noise."
    lines.append(f"free-running best = {free_best:.4f} (constant {baseline:.4f})" + tail)
    msg = "\n".join(lines)
    print("\n" + msg)
    json.dump({"table": table, "conditional": {k: {f"{m}_L{kk}": v
                                                   for (m, kk), v in acc.items()}
                                               for k, acc in cond.items()},
               "conditional_ar_parity": {f"{m}_L{kk}": v for (m, kk), v in par.items()},
               "copy_prior": {f"{m}_L{kk}": v for (m, kk), v in cp.items()},
               "baseline": baseline, "baseline_by_level": base_lv, "summary": msg},
              open(cfg["out"], "w"), indent=2)
    print("wrote", cfg["out"])


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("cfg")
    a = ap.parse_args()
    run(json.load(open(a.cfg)))


if __name__ == "__main__":
    main()


## 3. Verify, then import

`git` mode compares the clone against the snapshot and *warns* on a difference - a pinned repo that
is behind the notebook is a fact worth seeing, not an error. `snapshot` mode *fails* on a
difference, because there the payload cell is the only copy of the code.

In [ ]:
import hashlib
import os
import shutil
import sys


def _sha12(path):
    with open(path, "rb") as f:
        return hashlib.sha256(f.read()).hexdigest()[:12]


if SRC == "git":
    print("source: git", PIN, "->", PKG, "| the clone is authoritative")
else:
    shutil.copytree(STAGE, PKG, dirs_exist_ok=True)
    print("source: snapshot ->", PKG)
shutil.rmtree(STAGE, ignore_errors=True)          # only after the copy landed

got = {}
for rel, want in EXPECTED.items():
    path = os.path.join(PKG, rel)
    got[rel] = _sha12(path) if os.path.exists(path) else None

missing = [r for r, h in got.items() if h is None]
differ = [r for r, h in got.items() if h is not None and h != EXPECTED[r]]
assert not missing, f"missing after materialising: {missing}"

if differ:
    print(f"\n!! {len(differ)} file(s) differ from this notebook's snapshot (sha256[:12]):")
    for rel in differ:
        print(f"   {rel:20s} notebook {EXPECTED[rel]}  on disk {got[rel]}")
    if SRC == "snapshot":
        raise AssertionError("payload != sha table: regenerate with python tools/build_notebook.py")
    print("   (git mode: the clone wins - the notebook is behind the pushed code)")
else:
    print(f"sha256 OK | {len(EXPECTED)} files | nothing truncated, nothing stale")

if PKG not in sys.path:
    sys.path.insert(0, PKG)
os.chdir(PKG)              # run/, checkpoints and tensorboard all land under PKG

import omni                                    # noqa: E402  (TF poisoning shim, then the package)
import core                                    # noqa: E402  (omni/model.py imports this by name)
from omni.schedule import OmniSchedule         # noqa: E402

assert os.path.dirname(os.path.abspath(omni.__file__)) == os.path.join(PKG, "omni"), \
    "omni resolved somewhere else - a stale install is shadowing PKG"
print("omni from", os.path.dirname(omni.__file__))
print("core from", core.__file__)

## 4. Hardware

Fail here, not 40 minutes into a feed. `moex` is 1.178B parameters with 462M active per token, and at
`LEN=1024` one sample packs 7,056 positions - so the batch that fits is set by VRAM, not by taste.

In [ ]:
import torch
from omni.train import fast_math

assert torch.cuda.is_available(), "Runtime -> Change runtime type -> GPU"
p = torch.cuda.get_device_properties(0)
vram = p.total_memory / 2 ** 30
print(f"{p.name} | sm {p.major}.{p.minor} | {vram:.0f} GiB")
print("fast math:", fast_math())

if not BATCH:
    BATCH = 8 if vram < 20 else (16 if vram < 40 else 32)
print(f"batch {BATCH} | 1.178B params + AdamW is ~14 GiB before a single activation")
if vram < 20:
    print("!! 1.178B with AdamW will not sit comfortably in this card: lower BATCH to 8, or free it")

## 5. Config, built in memory

`build_parser` + `apply_overrides` + `train_cfg` are the same three functions the CLI uses, so the
notebook cannot drift from `run_omni.py train`. `ARM = "strict"` is the only arm: every code of a
level in one forward, 6 forwards for 6 levels instead of 504.

In [ ]:
import os

import torch

import run_omni as R
from omni.producer import GROUPS
from omni.schedule import OmniSchedule
from omni.train import build as build_net
from omni.train import flop_params

args = R.build_parser().parse_args(
    ["train", "--preset", PRESET, "--base-len", str(LEN), "--batch", str(BATCH),
     "--moe", str(MOE), "--moe-ffn", str(MOE_FFN), "--log-every", str(LOG_EVERY)])
rt = R.apply_overrides(R.default_cfg(), args)

# ---- the token budget, resolved before anything is spent ----
# STEPS is a consequence of TOKENS, not the other way round. "Is it learning" is a question about
# tokens seen; a step count means nothing without the width and packed length that produced it.
sched_pre = OmniSchedule(base_len=rt["sched"]["base_len"], K=rt["sched"]["K"], R=rt["sched"]["R"],
                         null_code=rt["sched"].get("null_code", 0),
                         modalities=tuple(rt["sched"]["modalities"]))
mix = rt["produce"]["mix"]
w = sum(mix.values())
LIVE = sum(mix[k] / w * sched_pre.flat_len(GROUPS[k]) for k in mix)   # live codes per sample
T = sched_pre.total_len
STEPS = max(1, round(TOKENS * 1e9 / (BATCH * LIVE)))

cfg = R.train_cfg(rt, ARM, out_dir=os.path.join(PKG, "run", "direct"), steps=STEPS)
cfg["eval_every"] = EVAL_EVERY
cfg["log_every"] = LOG_EVERY
cfg["eval_files"] = {}          # no npz: validation is the holdout
cfg["hard_exit"] = False        # a notebook cell has to return; run() never needs os._exit
cfg["preflight"] = dict(steps=PRE_STEPS, batch=PRE_BATCH, mask_mode=ARM)

with torch.device("meta"):      # count the parameters without allocating 1.178B of them
    net_pre, _ = build_net(cfg)
ACTIVE = flop_params(net_pre)

# Effective bf16, not peak. This box's 3070 measures 20-30 TFLOPS end to end; an A100 lands near
# 125 at ~40% MFU, a T4 nearer 15. Guessing high here is how a "5 minute" run becomes an hour.
if TFLOPS:
    tflops_eff = float(TFLOPS) * 1e12
elif not torch.cuda.is_available():
    tflops_eff = 5e10                                  # CPU: smoke only, this is not a benchmark
else:
    _nm = torch.cuda.get_device_name(0).upper()
    _gb = torch.cuda.get_device_properties(0).total_memory / 2 ** 30
    if "A100" in _nm or "H100" in _nm:
        tflops_eff = 125e12
    elif _gb < 20:
        tflops_eff = 25e12                             # consumer card: 20-30 measured
    else:
        tflops_eff = 60e12
est = 6 * ACTIVE * STEPS * BATCH * T / tflops_eff / 60
feed = STEPS * BATCH / 2 / 60                           # Hub feed at ~2 samples/s
print(f"model  {PRESET}: {MOE} experts x {MOE_FFN} wide | {ACTIVE/1e6:.0f}M active/token")
print(f"length {T} positions/sample | {LIVE:.0f} live codes/sample ({100*LIVE/T:.0f}% of slots)")
print(f"budget {TOKENS:.3f}B live codes = {STEPS:,} steps x {BATCH} = {STEPS*BATCH:,} samples")
print(f"eta    ~{est:.0f} min at {tflops_eff/1e12:.0f} effective TFLOPS | "
      f"~{feed:.0f} min if the Hub feed sets the pace")
print(f"mask   {cfg['model']['mask_mode']} | data {'shards' if USE_SHARDS else 'direct feed'}")
print("out", cfg["out_dir"])

## 6. TensorBoard

Started before the first step so step 0 is visible. The path is `PKG/run/direct/tb` - edit it if you
moved `PKG`.

In [ ]:
%load_ext tensorboard
%tensorboard --logdir /content/ncm_omni/run/direct/tb

## 7. The direct feed

`iter_rows` pulls from the Hub, decodes with the frozen codecs and yields encoded rows in-process.
The first `HOLD` rows become the in-memory validation set and are consumed from the same iterator,
so nothing is written to disk and nothing can go stale. Stable within a run; it is not an external
test set.

In [ ]:
if USE_SHARDS:
    hold, feed = None, None                  # rows come off npz; nothing is pulled here
    print("shard mode: the feed cell is skipped")
else:
    from omni.livestream import iter_rows

    live = dict(rt["produce"])
    live.update(sched=rt["sched"], vocab=rt["vocab"], batch_docs=8)

    raw = iter_rows(live, verbose=True)      # generator: nothing is fetched until it is asked
    import time as _t

    # A silent first pull is indistinguishable from a hang, so report progress. And make the holdout
    # STRATIFIED: the first HOLD rows off the live feed are not a sample of the mix - at startup the
    # cheapest streams (text) dominate - so a first-N holdout can be one modality only, and eval then
    # reports 0.0 accuracy for the others, which reads as a broken model instead of an untested one.
    QUOTA = {"text": 24, "image": 24, "audio": 16}
    _t0, hold, got, pulled = _t.time(), [], {k: 0 for k in QUOTA}, 0
    print(f"opening {len(live['mix'])} streams, pulling a stratified holdout {QUOTA}...", flush=True)
    # Cap on TIME, not row count. Rows are cheap once a pump is up (15/s measured) but the 19 streams
    # take ~30 s to all start, and the image/audio pumps come up last: a row-count cap expires inside
    # that warm-up and the holdout ends up text-only, which then reads as 0.0 accuracy downstream.
    while _t.time() - _t0 < 180:
        r = next(raw)
        pulled += 1
        # Credit the row to the scarcest modality it satisfies, so a text+image row helps image.
        cands = [m for m, ok in r["present"].items() if ok and m in QUOTA and got[m] < QUOTA[m]]
        if cands:
            g = max(cands, key=lambda m: QUOTA[m] - got[m])
            hold.append(r)
            got[g] += 1
        if pulled % 32 == 0:
            print(f"  {pulled} rows, {_t.time() - _t0:.0f}s -> holdout {got}", flush=True)
        if all(got[k] >= QUOTA[k] for k in QUOTA):
            break
    feed = raw
    print(f"holdout {len(hold)} rows {got} in {_t.time() - _t0:.0f}s ({pulled} pulled)")
    missing = [k for k in QUOTA if got[k] < QUOTA[k]]
    if missing:
        print(f"!! never saw enough of {missing} in 180s: eval will skip those modalities. "
              f"Their streams are slow to start or throttled (set HF_TOKEN).")

## 7b. Produce, then train off the shards

`USE_SHARDS = True` builds the corpus once, then the trainer reads npz. At `LEN=256` a sample carries
~546 live codes, so the smoke budget is a few thousand samples, while the direct feed yields ~2
samples/s - an order of magnitude slower than this GPU can consume. Producing also exercises the
encoder, the codec ladder and the shard writer, which a direct-feed run never touches.

The producer writes its own output to `run/producer*.log`, so this cell polls the shard directory and
prints a progress line every 15 s instead of blocking silently.

In [ ]:
import glob
import json
import subprocess
import sys
import time

if USE_SHARDS:
    n_samples = max(STEPS * BATCH, 512)
    workers = 4 if n_samples >= 512 else 1       # one Hub stream per worker, so 4 is ~4x the rows
    cmd = [sys.executable, os.path.join(PKG, "run_omni.py"), "produce", "--foreground",
           "--budget", str(n_samples), "--base-len", str(LEN), "--shuffle-buf", "512",
           "--workers", str(workers)]
    print(f"$ python run_omni.py produce --budget {n_samples:,} --base-len {LEN} --workers {workers}")

    # The producer redirects its own stdout to run/producer*.log, so subprocess.run() sits completely
    # silent for the whole corpus - and half an hour of nothing looks exactly like a hang. Poll the
    # progress json the producer already writes: it carries the per-stage split, which is the only
    # thing that distinguishes a run waiting on the Hub from one waiting on the codec.
    def _tail(n=1):
        logs = sorted(glob.glob(os.path.join(PKG, "run", "producer*.log")),
                      key=os.path.getmtime) if os.path.isdir(PKG) else []
        if not logs:
            return ""
        lines = [l.strip() for l in open(logs[-1], errors="replace").read().splitlines() if l.strip()]
        return " | ".join(lines[-n:])[:110] if lines else ""

    def _prog():
        ps = glob.glob(os.path.join(cfg["shard_dir"], "progress*.json"))
        if not ps:
            return ""
        samples = budget = 0
        pct = {}
        for f in ps:
            try:
                d = json.load(open(f))
            except (OSError, ValueError):
                continue
            samples += int(d.get("samples", 0))
            budget += int(d.get("budget", 0))
            for k, v in (d.get("stage_pct") or {}).items():
                pct[k] = pct.get(k, 0.0) + float(v) / max(1, len(ps))
        s = f"{samples:,}/{budget:,} ({100 * samples / max(1, budget):.0f}%)"
        if pct:
            s += (f" | pull {pct.get('pull', 0):.0f}% enc {pct.get('enc', 0):.0f}%"
                  f" write {pct.get('write', 0):.0f}%")
        return s

    t0 = time.time()
    p = subprocess.Popen(cmd, cwd=PKG)
    last_n, last_t, warned = -1, time.time(), False
    while p.poll() is None:
        time.sleep(15)
        d = cfg["shard_dir"]
        got = len(glob.glob(os.path.join(d, "shard_*.npz"))) if os.path.isdir(d) else 0
        line = _prog()
        print(f"  {time.time() - t0:5.0f}s | {got:3d} shards | {line} | {_tail()}", flush=True)
        # A Hub retry loop looks identical to a hang from the outside: the sample count freezes and
        # the only clue is one repeated "Retrying in Ns" line. Say so once, with the cause.
        n = int(line.split("/")[0].replace(",", "")) if line else 0
        if n > last_n:
            last_n, last_t, warned = n, time.time(), False
        elif time.time() - last_t > 90 and not warned:
            warned = True
            print(f"  !! no new samples for {time.time() - last_t:.0f}s. If the tail says 'Retrying in "
                  f"Ns', the Hub is throttling this run - set HF_TOKEN in cell 6, or drop --workers.",
                  flush=True)
    if p.returncode != 0:
        # `run_omni.py produce` returns 1 if ANY worker returns non-zero, and the reason lives in
        # that worker's own log - not in the aggregate one. Print all of them, so the traceback is
        # in the cell instead of one grep away.
        print(f"producer exited {p.returncode} after {time.time() - t0:.0f}s - worker log tails:")
        for f in sorted(glob.glob(os.path.join(PKG, "run", "producer*.log"))):
            lines = [l.rstrip() for l in open(f, errors="replace").read().splitlines() if l.strip()]
            print(f"  == {os.path.basename(f)} ({len(lines)} lines)")
            for l in lines[-14:]:
                print("    ", l[:150])
        raise RuntimeError(f"produce failed after {time.time() - t0:.0f}s; see the tails above")

    shards = sorted(glob.glob(os.path.join(cfg["shard_dir"], "shard_*.npz")))
    evals = {os.path.basename(f)[5:-4]: f
             for f in glob.glob(os.path.join(cfg["shard_dir"], "eval_*.npz"))}
    assert len(shards) >= 1, f"no shard_*.npz in {cfg['shard_dir']} - {_tail(15) or 'no log'}"
    for cand in ("producer.log", "producer_w0.log"):
        if os.path.exists(os.path.join(PKG, "run", cand)):
            cfg["producer_log"] = os.path.join(PKG, "run", cand)
            break
    cfg["eval_files"] = evals      # the producer's held-out rows; eval_every reads these, not HOLD
    print(f"shards {len(shards)} | eval sets {sorted(evals)} in {cfg['shard_dir']} "
          f"| {time.time() - t0:.0f}s")
else:
    print("direct-feed mode: no producer run")

## 8. Preflight - the gate, in-process

The synthetic copy-relation stream: every level `k >= 1` must drop by at least 1.5 nats and land
under `0.5 * ln(V)` within `PRE_STEPS`. Catches a broken head or dead routing in a minute instead of
300 wasted GPU-hours. Asserts, because training on a model that cannot learn is the expensive
outcome, not the failed assert.

In [ ]:
from omni.preflight import run_preflight

res = run_preflight(cfg)
assert res["passed"], f"preflight failed on {res.get('failing')} - head/routing does not learn"
print("PREFLIGHT PASSED")

## 9. Train - one arm, off the feed

`run(cfg, feed=..., eval_rows=...)` bypasses `ShardStream` entirely, so the kernel survives it (no
`os._exit` on this path). Step 0 logs immediately: that line is the proof the real data path works.

In [ ]:
cfg["preflight"]["enabled"] = False    # already gated above; do not pay twice
from omni.train import run

if USE_SHARDS:
    run(cfg)                              # ShardStream. hard_exit is off, so the cell returns.
else:
    run(cfg, smoke=SMOKE or None, feed=feed, eval_rows=hold)

## 10. What came out

In [ ]:
import json
import math
import os
import statistics

path = os.path.join(cfg["out_dir"], "metrics.jsonl")
rows = [json.loads(l) for l in open(path) if l.strip()]
# metrics.jsonl is opened in append mode, so it also holds every earlier run in this out_dir - and
# an old record's loss has nothing to do with this one. A new run restarts at step 0, so the last
# descending step among the training records marks where this run begins.
start, prev = 0, -1
for i, r in enumerate(rows):
    if r.get("val"):
        continue
    s = r.get("step", 0)
    if s < prev:
        start = i
    prev = s
if start:
    print(f"ignoring {start} records from earlier runs in this file")
rows = rows[start:]
train = [r for r in rows if not r.get("val")]
val = [r for r in rows if r.get("val")]
assert train, f"no training rows in {path} - the run never reached a logged step"

first, last = train[0], train[-1]
print(f"steps {len(train)} | loss {first['loss']:.4f} -> {last['loss']:.4f}"
      f" | {last.get('pos_per_s', 0):,.0f} pos/s | {last.get('tflops', 0)} TFLOPS")

# ---------------------------------------------------------------- is it learning?
# Six assertions on metrics.jsonl alone. Per-(modality, level) CE is in the record, so this needs no
# TensorBoard reader: one expert per modality is only worth it if the three learn separately.
fails = []


def need(ok, why):
    if not ok:
        fails.append(why)


need(all(math.isfinite(r["loss"]) for r in train), "non-finite loss in the record")
n = max(1, len(train) // 5)
early = statistics.median(r["loss"] for r in train[:n])
late = statistics.median(r["loss"] for r in train[-n:])
drop = 100 * (1 - late / early) if early else 0.0
print(f"loss  median {early:.3f} -> {late:.3f}  ({drop:+.1f}%, ln(257) at init = {math.log(257):.3f})")
need(late <= 0.83 * early, f"loss moved only {drop:.1f}%, want >= 17%")

per = {}
for r in train:
    for key, v in (r.get("ce") or {}).items():
        per.setdefault(key, []).append(v)
moved, flat = [], []
for mod in ("text", "image", "audio"):
    ks = sorted(k for k in per if k.startswith(mod + "_"))
    if not ks:
        continue
    m = min(len(per[k]) for k in ks)
    series = [sum(per[k][i] for k in ks) / len(ks) for i in range(m)]
    d = 100 * (1 - series[-1] / series[0]) if series[0] else 0.0
    print(f"  {mod:5s} CE {series[0]:6.3f} -> {series[-1]:6.3f}  ({d:+5.1f}%)")
    (moved if d >= 10 else flat).append(mod)
need(len(moved) >= 2, f"{len(moved)}/3 modalities dropped >=10% ({flat} flat)")

best = {}
for r in val:
    for key, v in r["val"].items():
        if v.get("acc", 0.0) > best.get(key, 0.0):
            best[key] = v["acc"]
mini = max(0.05, 12 * 1 / 257)              # chance is 1/257 = 0.39%
won = {m for k, a in best.items() if a >= mini for m in [k.split("_")[0]]}
if best:
    print("val   best acc per modality:",
          {m: round(max([a for k, a in best.items() if k.startswith(m + '_')] or [0.0]), 3)
           for m in ("text", "image", "audio")})
    need(len(won & set(moved)) >= 2, f"val accuracy near chance for 2+ modalities ({sorted(won)})")
else:
    print("val   no eval sets - accuracy check skipped (shard mode reads eval_*.npz from produce)")

need(all(math.isfinite(r.get("grad_norm", 1.0)) for r in train), "non-finite grad norm")
need(last["pos_per_s"] > 0, "throughput is zero")
need(len(train) >= 3, "fewer than 3 logged steps: nothing to conclude")

print()
if fails:
    print("VERDICT:", "NOT LEARNING" if late > 0.95 * early else "PARTIAL")
    for f in fails:
        print("  -", f)
    print("  Do not scale this up. Fix the gate first, then turn on one modality at a time.")
else:
    # Report what actually ran, not what was planned: the shard stream ends when the corpus does,
    # so `STEPS x BATCH` can overstate the run by an order of magnitude (47 of 229 steps here).
    seen = int(last.get("samples") or 0) or STEPS * BATCH
    print(f"VERDICT: PASS | loss {drop:.1f}% down | {len(moved)}/3 modalities moving | "
          f"{seen:,} samples = {seen * LIVE / 1e6:.2f}M live codes")
    if seen < STEPS * BATCH:
        print(f"  note: ran {len(train)} logged steps of a planned {STEPS} - the corpus ran out. "
              f"{seen:,} of {STEPS * BATCH:,} samples.")
    if not best:
        print("  note: no held-out validation in this run (produce never wrote eval_*.npz), "
              "so this is training loss only.")

print("tb:", os.path.join(cfg["out_dir"], "tb"))

## 10b. Where the time went

`metrics.jsonl` carries a per-step stage split, taken with the GPU synchronised, so it measures work
and not kernel launch: `t_fetch` is the feed, `t_prep` is collate + host-to-device, `t_fwd`/`t_bwd`/
`t_opt` are the model. On a feed-bound run `data` sits near 90% and no amount of GPU tuning helps.

In [ ]:
import json
import os
import statistics

rows = [json.loads(l) for l in open(os.path.join(cfg["out_dir"], "metrics.jsonl")) if l.strip()]
train = [r for r in rows if not r.get("val") and "t_fetch" in r]
assert train, "no stage timings in metrics.jsonl - this run predates the trace change"

keys = ("t_fetch", "t_prep", "t_fwd", "t_bwd", "t_opt")
med = {k: statistics.median(r.get(k, 0.0) for r in train) for k in keys}
tot = max(1e-9, sum(med.values()))
print(f"{len(train)} logged steps | median step {tot:.3f}s")
print("  " + " ".join(f"{k.replace('t_', ''):5s} {med[k]:6.3f}s {100 * med[k] / tot:5.1f}%"
                      for k in keys))
data = 100 * (med["t_fetch"] + med["t_prep"]) / tot
print(f"\ndata {data:.0f}% | gpu {100 - data:.0f}%")
if data > 60:
    print("  feed-bound: the GPU idles most of the step. Produce ahead of training, or add workers.")
else:
    print("  compute-bound: the feed is keeping up.")

## Appendix - re-running after a fix

| you changed | do this |
|---|---|
| a `.py` in the repo | `python tools/build_notebook.py`, upload the notebook, **Run all** |
| pushed a commit, want it now | set `SRC = "git"` and `PIN = "<sha>"`, re-run cells 2-3 only |
| nothing, but the kernel is confused | re-run the purge cell, then the verify cell |

Why the purge cell is not optional in git mode: `sys.modules` holds the module *objects*. Re-pulling
files under a live kernel leaves `omni.model.OmniNCM` pointing at the old code, so the run continues
with the version you were trying to replace. Cell 2 deletes `omni*`, `core` and `run_omni` from
`sys.modules` and calls `importlib.invalidate_caches()` before anything is copied; the verify cell
then imports and asserts that `omni` resolved to `PKG/omni` and not to a site-packages copy.

`tools/build_notebook.py --check` compares every payload cell against the file on disk and exits
non-zero on drift, which is the sane thing to put in CI.